<a href="https://colab.research.google.com/github/Aritra2004679/HyBert-X/blob/main/Feature%20extraction%20and%20model%20training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

PROJECT_PATH = "/content/drive/MyDrive/Cyberbullying_Project"

df = pd.read_csv(os.path.join(PROJECT_PATH, "cyberbullying.csv"))

print("Dataset Loaded Successfully!")
print(df.shape)

Mounted at /content/drive
Dataset Loaded Successfully!
(684383, 14)


In [ ]:
import sys
import importlib

PROJECT_PATH = "/content/drive/MyDrive/Cyberbullying_Project"

if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)

import feature_engineering
importlib.reload(feature_engineering)

from feature_engineering import build_features, FeatureConfig

print("✅ feature_engineering.py reloaded successfully!")

✅ feature_engineering.py reloaded successfully!


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def initial_inspection(df):
    print(f"Initial shape: {df.shape}")
    print("Columns:", df.columns.tolist())
    print("Missing values:", df.isnull().sum())
    print("Dtypes:", df.dtypes)
    print("Sample:", df.sample(5, random_state=42))

def clean_text_column(df, text_col='text'):
    # Remove empty/null
    df = df[df[text_col].notnull() & (df[text_col].str.strip() != '')]
    # Remove non-alphabetic
    df = df[df[text_col].astype(str).str.contains(r'[a-zA-Z]', regex=True)]
    # Remove only @mention
    df = df[~df[text_col].astype(str).str.match(r'^@\w+$')]
    # Remove only emoji
    emoji_pattern = re.compile(r'^[😀-🙏🌀-🗿🚀-🛿🇠-🇿✀-➿🤀-🧿☀-⛿]+$', flags=re.UNICODE)
    df = df[~df[text_col].astype(str).apply(lambda x: bool(emoji_pattern.match(x)))]
    # Remove only textual emoji
    textual_emoji_pattern = re.compile(r'^(?::|;|=)(?:-)?(?:\)|\(|D|P)$')
    df = df[~df[text_col].fillna('').str.strip().str.match(textual_emoji_pattern)]
    # Remove only numbers
    df = df[~df[text_col].fillna('').str.strip().str.match(r'^\d+$')]
    # Remove only link
    url_pattern = re.compile(r'^(https?://\S+|www\.\S+)$')
    df = df[~df[text_col].fillna('').str.strip().str.match(url_pattern)]
    # Remove only symbol
    df = df[~df[text_col].fillna('').str.strip().str.match(r'^[^\w\s]+$')]
    # Remove only repeated char/word
    df = df[~df[text_col].fillna('').str.strip().str.match(r'^(.){2,}$|^(\w+)(\s+){2,}$')]
    # Remove only hashtags
    df = df[~df[text_col].fillna('').str.strip().str.match(r'^(#\w+\s*)+$')]
    return df

def advanced_cleaning(df, text_col='text'):
    # Remove 'Dear ...' (less than 4 words)
    pattern_dear = re.compile(r'^Dear(\s+\w+){1,2}[,.!?]*$', re.IGNORECASE)
    df = df[~(df[text_col].fillna('').str.strip().str.match(pattern_dear) & (df[text_col].fillna('').str.strip().str.split().apply(len) < 4))]
    # Remove general greetings/abbreviations/phrases (less than 3 words)
    general_phrases = [
        'hi', 'hello', 'how are you', 'who are you', 'why', 'when', 'where', 'i dont know',
        'hope you are good', 'jai shree ram', 'ganpati bappa morya', 'salamwaleikum', 'namaste',
        'bhabhai', 'bhabii', 'fuck you', 'bye', 'goodbye', 'good night', 'good morning', 'good evening',
        'pradhan ji', 'bhai ji', 'bhabhi ji', 'bhaiya ji', 'sir ji', 'madam ji', 'bhai', 'bhabhi', 'salam',
    ]
    def contains_phrase_short(text):
        text = str(text).lower().strip()
        if len(text.split()) < 3:
            for phrase in general_phrases:
                if phrase in text:
                    return True
        return False
    df = df[~df[text_col].apply(contains_phrase_short)]
    return df

In [ ]:
from sklearn.model_selection import train_test_split

# Clean the dataframe
df = clean_text_column(df, text_col='text')
df = advanced_cleaning(df, text_col='text')

# Feature column
X = df[['text']]

# All label columns
label_cols = [
    'religious_hate',
    'ethnic_hate',
    'age_discrimination',
    'gender_hate',
    'sexual_harassment',
    'threats',
    'body_shaming',
    'political_hate',
    'trolling',
    'mental_hate',
    'discrimination',
    'other_cyberbullying_types',
    'not_cyberbullying'
]

y = df[label_cols]

# Train-test split
df_train, df_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(df_train.shape)
print(df_test.shape)
print(y_train.shape)
print(y_test.shape)

(547148, 1)
(136787, 1)
(547148, 13)
(136787, 13)


In [ ]:
# ============================================================
# SARCASM-BASED FEATURE EXTRACTION
# Final heuristic score: s
# ============================================================

import re
import numpy as np
import pandas as pd

# ============================================================
# HEURISTIC WEIGHTS (Sensitivity Analysis)
# ============================================================

# Original weights (Baseline)
W_MC   = 2.0   # Sarcasm marker count
W_EX   = 1.5   # Repeated exclamation pattern count
W_CC   = 2.0   # Contextual contradiction count
W_PN   = 1.5   # Positive-negative sentiment incongruity
W_CAPS = 3.0   # ALL-CAPS ratio
W_ID   = 2.0   # Intensifier density
W_EC   = 1.5   # Emoji-text sentiment contradiction

# Normalization constant
NORMALIZATION_FACTOR = 10.0


class SarcasmFeatures:

    # --------------------------------------------------------
    # 1. Positive sentiment words
    # --------------------------------------------------------

    _POSITIVE_WORDS = {
        'great', 'wonderful', 'amazing', 'fantastic', 'brilliant',
        'excellent', 'perfect', 'awesome', 'lovely', 'beautiful',
        'nice', 'good', 'fine', 'super', 'best', 'love', 'like',
        'happy', 'glad', 'thrilled', 'exciting', 'interesting',
        'helpful', 'useful', 'smart', 'clever', 'genius',
        'impressive', 'outstanding', 'incredible', 'fabulous'
    }

    # --------------------------------------------------------
    # 2. Negative sentiment words
    # --------------------------------------------------------

    _NEGATIVE_WORDS = {
        'hate', 'terrible', 'awful', 'horrible', 'disgusting',
        'pathetic', 'stupid', 'idiot', 'moron', 'useless',
        'worthless', 'trash', 'garbage', 'fail', 'failure',
        'worst', 'bad', 'ugly', 'dumb', 'loser', 'joke',
        'seriously', 'really', 'actually'
    }

    # --------------------------------------------------------
    # 3. Sarcasm marker phrases
    # --------------------------------------------------------

    _SARCASM_MARKERS = [
        r'\boh great\b',
        r'\byeah right\b',
        r'\bsure sure\b',
        r'\bthanks a lot\b',
        r'\bthanks so much\b',
        r'\bso helpful\b',
        r'\breally helpful\b',
        r'\bso smart\b',
        r'\bwow thanks\b',
        r'\boh wow\b',
        r'\boh really\b',
        r'\bno kidding\b',
        r'\bno way\b',
        r'\bright sure\b',
        r'\bas if\b',
        r'\bcongrats\b',
        r'\bcongratulations\b',
        r'\bwhat a surprise\b',
        r'\bshocking\b',
        r'\bunbelievable\b',
        r'\bi\'m sure\b',
        r'\bim sure\b',
        r'\bi bet\b',
        r'\bof course\b',
        r'\bobviously\b',
        r'\bclearly\b',
        r'\bjust great\b',
        r'\bjust perfect\b',
        r'\bso funny\b',
        r'\bhilarious\b',
    ]

    # --------------------------------------------------------
    # 4. Contradiction patterns
    # --------------------------------------------------------

    _CONTRADICTION_PAIRS = [
        (
            r'\b(?:great|wonderful|amazing|fantastic)\b',
            r'\b(?:not|never|no|hate|terrible)\b'
        ),
        (
            r'\b(?:love|like|enjoy)\b',
            r'\b(?:not|never|no|hate|dislike)\b'
        ),
        (
            r'\b(?:good|nice|fine)\b',
            r'\b(?:not|terrible|awful|horrible)\b'
        ),
    ]

    # --------------------------------------------------------
    # 5. Initialize regular expressions
    # --------------------------------------------------------

    def __init__(self):

        self._marker_pats = [
            re.compile(pattern, re.IGNORECASE)
            for pattern in self._SARCASM_MARKERS
        ]

        self._contra_pats = [
            (
                re.compile(positive_pattern, re.IGNORECASE),
                re.compile(negative_pattern, re.IGNORECASE)
            )
            for positive_pattern, negative_pattern
            in self._CONTRADICTION_PAIRS
        ]

        self._intensifier_re = re.compile(
            r'\b(?:so|very|extremely|incredibly|absolutely|totally|'
            r'completely|literally|seriously|really|such|beyond|super)\b',
            re.IGNORECASE
        )

        self._pos_emoji_re = re.compile(
            r'[\U0001F600-\U0001F606'
            r'\U0001F609\U0001F60A'
            r'\U0001F60D\U0001F618'
            r'\U0001F61C\U0001F61D'
            r'\U0001F923]'
        )

    # --------------------------------------------------------
    # 6. Calculate final sarcasm score s
    # --------------------------------------------------------

    def transform(self, texts):

        sarcasm_scores = []

        for text in texts:

            t = str(text)
            lower = t.lower()

            # Word count (w)
            wc = max(len(lower.split()), 1)

            # Lowercase words
            lower_words = [
                word.strip('.,!?;:\'"')
                for word in lower.split()
            ]

            # ------------------------------------------------
            # Calculate variables used in the final equation
            # ------------------------------------------------

            # m_c : sarcasm marker count
            mc = sum(
                1
                for pattern in self._marker_pats
                if pattern.search(t)
            )

            # e_x : repeated exclamation pattern count
            ex = len(
                re.findall(r'!{2,}', t)
            )

            # c_c : contradiction count
            cc = sum(
                1
                for positive_pattern, negative_pattern
                in self._contra_pats
                if positive_pattern.search(t)
                and negative_pattern.search(t)
            )

            # p_c : positive word count
            pc = sum(
                1
                for word in lower_words
                if word in self._POSITIVE_WORDS
            )

            # n_c : negative word count
            nc = sum(
                1
                for word in lower_words
                if word in self._NEGATIVE_WORDS
            )

            # Indicator function:
            # I[p_c > 0 AND n_c > 0]
            pos_neg_flag = int(
                pc > 0 and nc > 0
            )

            # r_caps : ratio of ALL-CAPS words
            caps_words = len(
                re.findall(r'\b[A-Z]{2,}\b', t)
            )

            rcaps = caps_words / wc

            # i_d : intensifier word density
            intensifier_count = len(
                self._intensifier_re.findall(t)
            )

            intensity_density = intensifier_count / wc

            # e_c : emoji-text sentiment contradiction
            ec = int(
                len(self._pos_emoji_re.findall(t)) > 0
                and nc > 0
            )

            # ------------------------------------------------
            # Final normalized heuristic sarcasm score

            # ------------------------------------------------
            s = min(
    (
        W_MC   * mc
        + W_EX   * ex
        + W_CC   * cc
        + W_PN   * pos_neg_flag
        + W_CAPS * rcaps
        + W_ID   * intensity_density
        + W_EC   * ec
    ) / NORMALIZATION_FACTOR,
    1.0
)

            # Store only final normalized score s
            sarcasm_scores.append(s)

        # ----------------------------------------------------
        # Return ONLY final normalized sarcasm score s
        # ----------------------------------------------------

        return np.array(
            sarcasm_scores,
            dtype=np.float32
        ).reshape(-1, 1)


# ============================================================
# EXECUTE SARCASM FEATURE EXTRACTION
# ============================================================

# Assumes that 'df' already exists and contains a 'text' column.

sarcasm_extractor = SarcasmFeatures()

X_sarcasm = sarcasm_extractor.transform(
    df['text']
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 60)
print("SARCASM FEATURE EXTRACTION COMPLETED")
print("=" * 60)

print("Number of text samples:", len(df))

print("Sarcasm feature shape:", X_sarcasm.shape)

print("Feature used by the model:")
print("sarc_heuristic_score")

print("\nFirst 10 normalized sarcasm scores:")

for i, score in enumerate(
    X_sarcasm[:10].flatten(),
    start=1
):
    print(f"Sample {i}: {score:.6f}")

print("\nMinimum sarcasm score:",
      X_sarcasm.min())

print("Maximum sarcasm score:",
      X_sarcasm.max())

print("Mean sarcasm score:",
      X_sarcasm.mean())

print("=" * 60)

SARCASM FEATURE EXTRACTION COMPLETED
Number of text samples: 683935
Sarcasm feature shape: (683935, 1)
Feature used by the model:
sarc_heuristic_score

First 10 normalized sarcasm scores:
Sample 1: 0.000000
Sample 2: 0.200000
Sample 3: 0.000000
Sample 4: 0.000000
Sample 5: 0.153704
Sample 6: 0.000000
Sample 7: 0.000000
Sample 8: 0.004348
Sample 9: 0.000000
Sample 10: 0.000000

Minimum sarcasm score: 0.0
Maximum sarcasm score: 1.0
Mean sarcasm score: 0.031956535


In [ ]:
# ================================================================
# ORIGINAL-WEIGHT SARCASM SVM EXPERIMENT
# ================================================================



import re
import time
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    f1_score,
    precision_score,
    recall_score
)


# ================================================================
# CONFIGURATION
# ================================================================

DATA_PATH = '/content/drive/MyDrive/Cyberbullying_Project/cyberbullying.csv'

TEST_SIZE = 0.2
RANDOM_STATE = 42

# Decision threshold
# 0.0 is the default LinearSVC decision boundary.
# You can tune this later if required.
THRESHOLD = 0.0


LABEL_COLS = [
    'religious_hate',
    'ethnic_hate',
    'age_discrimination',
    'gender_hate',
    'sexual_harassment',
    'threats',
    'body_shaming',
    'political_hate',
    'trolling',
    'mental_hate',
    'discrimination',
    'other_cyberbullying_types',
    'not_cyberbullying'
]


# ================================================================
# FEATURE DEFINITIONS
# ================================================================

HATE_KEYWORDS = {

    'religious_hate': [
        'infidel','kafir','blasphemy','blasphemer','heretic',
        'cult','crusade','jihad','pagan','apostate','heathen',
        'godless','anti-christian','anti-islam','anti-hindu',
        'anti-semitic','zionist','islamophobe','christophobe'
    ],

    'ethnic_hate': [
        'nigger','nigga','chink','spic','wetback','gook','kike',
        'raghead','cracker','redneck','monkey','ape','beaner',
        'immigrant','illegal alien','foreigner','outsider',
        'invader','terrorist'
    ],

    'gender_hate': [
        'bitch','whore','slut','cunt','feminazi','tranny','dyke',
        'faggot','sissy','mangina','thot','hoe','skank','femoid',
        'incel','simp','soyboy','misogynist','misandrist'
    ],

    'sexual_harassment': [
        'rape','rapist','molest','grope','pervert','predator',
        'creep','harass','assault'
    ],

    'threats': [
        'kill','murder','shoot','stab','bomb','attack','destroy',
        'hurt','harm','threat','die','execute','eliminate',
        'exterminate','torture','burn','acid','explode','lynch',
        'hang','beat','bash','smash'
    ],

    'body_shaming': [
        'fat','ugly','obese','disgusting','gross','pig','whale',
        'skinny','anorexic','hideous','repulsive','deformed',
        'freak','monster','troll','goblin'
    ],

    'mental_hate': [
        'retard','retarded','psycho','crazy','insane','lunatic',
        'mental','schizo','bipolar','autistic','mentally ill',
        'nutcase','bonkers','deranged','demented'
    ],

    'general_hate': [
        'hate','stupid','idiot','moron','dumb','loser','trash',
        'garbage','scum','filth','worthless','pathetic','useless',
        'despise','abhor','vile','toxic','cancer','parasite',
        'vermin','subhuman'
    ]
}


POS_PATTERNS = {

    'NOUN':
        r'\b(?:man|woman|people|person|thing|way|day|time|year|'
        r'world|life|child|children|hand|part|place|case|week|'
        r'company|system|question|government|number|night|point|'
        r'home|water|room|mother|area|money|story|fact|month|lot|'
        r'right|study|book|eye|job|word|business|issue|side|kind|'
        r'head|house|service|friend|father|power|hour|game|line|'
        r'end|member|city|community|name|president|team|minute|'
        r'idea|body|information|back|parent|face|level|office|'
        r'door|health|art|war|history|party|result|change|morning|'
        r'reason|research|girl|guy|moment|air|teacher|force|education)\b',

    'VERB':
        r'\b(?:is|are|was|were|be|been|being|have|has|had|do|'
        r'does|did|will|would|could|should|may|might|shall|must|'
        r'can|go|get|make|know|think|take|see|come|want|look|use|'
        r'find|give|tell|work|call|try|ask|need|feel|become|leave|'
        r'put|mean|keep|let|begin|show|hear|play|run|move|live|'
        r'believe|hold|bring|happen|write|provide|sit|stand|lose|'
        r'pay|meet|include|continue|set|learn|change|lead|understand|'
        r'watch|follow|stop|create|speak|read|spend|grow|open|walk|'
        r'win|offer|remember|love|consider|appear|buy|wait|serve|'
        r'die|send|expect|build|stay|fall|cut|reach|kill|remain|'
        r'suggest|raise|pass|sell|require|report|decide|pull)\b',

    'ADJ':
        r'\b(?:good|new|first|last|long|great|little|own|other|'
        r'old|right|big|high|different|small|large|next|early|young|'
        r'important|public|private|real|best|free|few|same|able|'
        r'political|social|economic|national|possible|local|white|'
        r'black|strong|true|hot|happy|sad|bad|ugly|fat|stupid|evil|'
        r'dangerous|terrible|horrible|awful|disgusting|pathetic|'
        r'worthless|useless|dumb|crazy|insane|violent|racist|sexist|'
        r'offensive|hateful|toxic|radical|extreme)\b',

    'ADV':
        r'\b(?:up|so|out|just|now|how|then|more|also|here|well|'
        r'only|very|even|back|there|down|still|in|as|too|really|'
        r'most|never|much|often|always|actually|again|further|yet|'
        r'already|soon|especially|finally|simply|probably|certainly|'
        r'clearly|literally|absolutely|totally|definitely|obviously|'
        r'seriously|exactly|nearly|directly|quickly|easily)\b',

    'PRON':
        r'\b(?:i|me|my|myself|you|your|yourself|he|him|his|himself|'
        r'she|her|hers|herself|it|its|itself|we|us|our|ourselves|'
        r'they|them|their|theirs|themselves|this|that|these|those|'
        r'who|whom|which|what)\b'
}


PROFANITY = {
    'fuck','fucking','fucked','shit','ass','asshole','bitch',
    'cunt','bastard','damn','piss','cock','dick','pussy',
    'motherfucker','whore','slut','idiot','moron','loser',
    'stupid','dumb','retard'
}


# ================================================================
# SARCASM DEFINITIONS
# ORIGINAL-WEIGHT VERSION
# ================================================================

SARC_MARKERS = [

    r'\boh great\b',
    r'\byeah right\b',
    r'\bsure sure\b',
    r'\bthanks a lot\b',
    r'\bthanks so much\b',
    r'\bso helpful\b',
    r'\breally helpful\b',
    r'\bso smart\b',
    r'\bwow thanks\b',
    r'\boh wow\b',
    r'\boh really\b',
    r'\bno kidding\b',
    r'\bno way\b',
    r'\bright sure\b',
    r'\bas if\b',
    r'\bcongrats\b',
    r'\bcongratulations\b',
    r'\bwhat a surprise\b',
    r'\bshocking\b',
    r'\bunbelievable\b',
    r'\bi\'m sure\b',
    r'\bim sure\b',
    r'\bi bet\b',
    r'\bof course\b',
    r'\bobviously\b',
    r'\bclearly\b',
    r'\bjust great\b',
    r'\bjust perfect\b',
    r'\bso funny\b',
    r'\bhilarious\b'
]


SARC_POS_WORDS = {
    'great','wonderful','amazing','fantastic','brilliant',
    'excellent','perfect','awesome','lovely','nice','good',
    'fine','super','best','love','happy','glad','thrilled',
    'helpful','smart','clever','impressive','incredible'
}


SARC_NEG_WORDS = {
    'hate','terrible','awful','horrible','disgusting','pathetic',
    'stupid','idiot','moron','useless','worthless','trash',
    'garbage','fail','worst','bad','ugly','dumb','loser'
}


SARC_CONTRA = [

    (
        r'\b(?:great|wonderful|amazing|fantastic)\b',
        r'\b(?:not|never|no|hate|terrible)\b'
    ),

    (
        r'\b(?:love|like|enjoy)\b',
        r'\b(?:not|never|no|hate|dislike)\b'
    ),

    (
        r'\b(?:good|nice|fine)\b',
        r'\b(?:not|terrible|awful|horrible)\b'
    )
]


_INTENSIFIER_RE = re.compile(
    r'\b(?:so|very|extremely|incredibly|absolutely|totally|'
    r'completely|literally|seriously|really|such|beyond|super)\b',
    re.IGNORECASE
)


_POS_EMOJI_RE = re.compile(
    r'[\U0001F600-\U0001F606'
    r'\U0001F609\U0001F60A'
    r'\U0001F60D\U0001F618'
    r'\U0001F61C\U0001F61D'
    r'\U0001F923]'
)


# ================================================================
# STEP 1: LOAD AND CLEAN DATA
# ================================================================

def load_and_clean():
    print("📂 Loading data...")
    df = pd.read_csv(DATA_PATH)
    before = len(df)
    df = df[
        df['text'].notnull()
        & (df['text'].astype(str).str.strip() != '')
    ]
    df = df[
        df['text'].astype(str).str.contains(
            r'[a-zA-Z]',
            regex=True
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^@\w+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^\d+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^[^\w\s]+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^(.)\1+$|^(\w+)(\s+\2)+\s*$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^(#\w+\s*)+$'
        )
    ]

    df = df.reset_index(drop=True)

    print(
        f"   {before:,} → {len(df):,} rows after cleaning"
    )

    return df
# ============================================================
# HEURISTIC WEIGHTS
# ============================================================

W_MC   = 2.0
W_EX   = 1.5
W_CC   = 2.0
W_PN   = 1.5
W_CAPS = 3.0
W_ID   = 2.0
W_EC   = 1.5

NORMALIZATION_FACTOR = 10.0

# ================================================================
# STEP 2: FEATURE EXTRACTION
# ================================================================
def extract_features(
    texts,
    tfidf=None,
    fit=False
):

    print("   Extracting features...")

    # ------------------------------------------------------------
    # Compile patterns once
    # ------------------------------------------------------------

    hate_pats = {

        cat: re.compile(
            r'\b(?:'
            + '|'.join(
                re.escape(k)
                for k in kws
            )
            + r')\b',
            re.IGNORECASE
        )

        for cat, kws in HATE_KEYWORDS.items()
    }


    pos_pats = {

        pos: re.compile(
            pat,
            re.IGNORECASE
        )

        for pos, pat in POS_PATTERNS.items()
    }


    marker_pats = [

        re.compile(
            p,
            re.IGNORECASE
        )

        for p in SARC_MARKERS
    ]


    contra_pats = [

        (
            re.compile(
                a,
                re.IGNORECASE
            ),

            re.compile(
                b,
                re.IGNORECASE
            )
        )

        for a, b in SARC_CONTRA
    ]


    url_re = re.compile(
        r'https?://\S+|www\.\S+'
    )

    mention_re = re.compile(
        r'@\w+'
    )

    hashtag_re = re.compile(
        r'#\w+'
    )

    emoji_re = re.compile(
        r'[\U0001F300-\U0001F9FF'
        r'\U00002600-\U000027FF]',
        flags=re.UNICODE
    )

    repeat_re = re.compile(
        r'(.)\1{2,}'
    )

    caps_re = re.compile(
        r'\b[A-Z]{2,}\b'
    )


    all_rows = []


    # ============================================================
    # PROCESS EACH TEXT
    # ============================================================

    for text in texts:

        t = str(text)

        words = t.split()

        wc = max(
            len(words),
            1
        )

        cc = max(
            len(t),
            1
        )


        lw = [

            w.lower().strip(
                '.,!?;:\'"'
            )

            for w in words
        ]


        sents = max(

            len(
                re.split(
                    r'[.!?]+',
                    t
                )
            ),

            1
        )


        # --------------------------------------------------------
        # Word frequency
        # --------------------------------------------------------

        wfreq = {}

        for w in lw:

            wfreq[w] = (
                wfreq.get(
                    w,
                    0
                )
                + 1
            )


        caps = caps_re.findall(t)


        prof = sum(

            1

            for w in lw

            if w in PROFANITY
        )


        row = {}


        # ========================================================
        # HATE KEYWORD FEATURES
        # ========================================================

        total_hate = 0


        for cat, pat in hate_pats.items():

            c = len(
                pat.findall(t)
            )


            row[
                f'hate_count_{cat}'
            ] = c


            row[
                f'hate_flag_{cat}'
            ] = int(
                c > 0
            )


            total_hate += c


        row[
            'hate_total_count'
        ] = total_hate


        row[
            'hate_density'
        ] = total_hate / wc


        row[
            'hate_category_count'
        ] = sum(

            1

            for cat in HATE_KEYWORDS

            if row[
                f'hate_flag_{cat}'
            ] > 0
        )


        # ========================================================
        # POS FEATURES
        # ========================================================

        pcounts = {}


        for pos, pat in pos_pats.items():

            c = len(
                pat.findall(t)
            )


            pcounts[pos] = c


            row[
                f'pos_count_{pos.lower()}'
            ] = c


            row[
                f'pos_ratio_{pos.lower()}'
            ] = c / wc


        row[
            'pos_adj_noun_ratio'
        ] = (

            pcounts.get(
                'ADJ',
                0
            )

            /

            max(
                pcounts.get(
                    'NOUN',
                    0
                ),
                1
            )
        )


        row[
            'pos_verb_density'
        ] = (

            pcounts.get(
                'VERB',
                0
            )

            / wc
        )


        row[
            'pos_pron_density'
        ] = (

            pcounts.get(
                'PRON',
                0
            )

            / wc
        )


        row[
            'pos_modifier_density'
        ] = (

            pcounts.get(
                'ADJ',
                0
            )

            +

            pcounts.get(
                'ADV',
                0
            )

        ) / wc


        # ========================================================
        # WORD FREQUENCY FEATURES
        # ========================================================

        row.update({

            'wf_char_count':
                cc,

            'wf_word_count':
                wc,

            'wf_sentence_count':
                sents,

            'wf_avg_word_len':
                np.mean(
                    [
                        len(w)
                        for w in words
                    ]
                )
                if words
                else 0,

            'wf_avg_sent_len':
                wc / sents,

            'wf_unique_word_ratio':
                len(
                    set(lw)
                ) / wc,

            'wf_caps_word_count':
                len(caps),

            'wf_caps_ratio':
                len(caps) / wc,

            'wf_upper_char_ratio':
                sum(
                    1
                    for c in t
                    if c.isupper()
                ) / cc,

            'wf_exclaim_count':
                t.count('!'),

            'wf_question_count':
                t.count('?'),

            'wf_ellipsis_count':
                t.count('...'),

            'wf_punct_density':
                sum(
                    1
                    for c in t
                    if c in '!?.,;:\'"'
                ) / cc,

            'wf_url_count':
                len(
                    url_re.findall(t)
                ),

            'wf_mention_count':
                len(
                    mention_re.findall(t)
                ),

            'wf_hashtag_count':
                len(
                    hashtag_re.findall(t)
                ),

            'wf_emoji_count':
                len(
                    emoji_re.findall(t)
                ),

            'wf_repeat_char_count':
                len(
                    repeat_re.findall(t)
                ),

            'wf_repeat_word_count':
                sum(

                    1

                    for v in wfreq.values()

                    if v > 1
                ),

            'wf_profanity_count':
                prof,

            'wf_profanity_density':
                prof / wc
        })


        # ========================================================
        # ORIGINAL-WEIGHT SARCASM SCORE
        # ========================================================

        multi_exc = len(
            re.findall(
                r'!{2,}',
                t
            )
        )


        mc = sum(

            1

            for p in marker_pats

            if p.search(t)
        )


        pc = sum(

            1

            for w in lw

            if w in SARC_POS_WORDS
        )


        nc = sum(

            1

            for w in lw

            if w in SARC_NEG_WORDS
        )


        cc2 = sum(

            1

            for a_p, b_p in contra_pats

            if (
                a_p.search(t)
                and
                b_p.search(t)
            )
        )


        id_ = (

            len(
                _INTENSIFIER_RE.findall(t)
            )

            / wc
        )


        ec = int(

            len(
                _POS_EMOJI_RE.findall(t)
            ) > 0

            and

            nc > 0
        )

        # --------------------------------------------------------
        # UNIFORM WEIGHTS
        # --------------------------------------------------------

        # --------------------------------------------------------
        # WEIGHTED SARCASM SCORE (Sensitivity Analysis)
        # --------------------------------------------------------

        sarcasm_score = min(
        (
          W_MC * mc
          + W_EX * multi_exc
          + W_CC * cc2
          + W_PN * int(
            pc > 0
            and
            nc > 0
          )
          + W_CAPS * (len(caps) / wc)
          + W_ID * id_
          + W_EC * ec
        ) / NORMALIZATION_FACTOR,
        1.0
        )





        # --------------------------------------------------------
        # ONLY FINAL SARCASM SCORE IS ADDED
        # --------------------------------------------------------

        row[
            'sarc_heuristic_score'
        ] = sarcasm_score


        all_rows.append(row)


    # ============================================================
    # CONVERT DENSE FEATURES
    # ============================================================

    dense = pd.DataFrame(
        all_rows
    ).values.astype(
        np.float32
    )


    dense = np.clip(
        dense,
        0,
        None
    )


    # ============================================================
    # TF-IDF
    # ============================================================

    if fit:

        print(
            "   Fitting TF-IDF..."
        )


        tfidf = TfidfVectorizer(

            max_features=20000,

            ngram_range=(1, 2),

            min_df=3,

            sublinear_tf=True
        )


        tfidf_mat = tfidf.fit_transform(

            texts.astype(str)
        )


    else:

        tfidf_mat = tfidf.transform(

            texts.astype(str)
        )


    # ============================================================
    # COMBINE FEATURES
    # ============================================================

    X = hstack(

        [

            csr_matrix(
                dense
            ),

            tfidf_mat

        ],

        format='csr'
    )


    return X, tfidf


# ================================================================
# EVALUATION
# ================================================================

def evaluate(
    y_true,
    y_pred,
    elapsed
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        "ORIGINAL-WEIGHT SARCASM SVM RESULTS"
    )

    print(
        "=" * 70
    )


    # ------------------------------------------------------------
    # Accuracy
    # ------------------------------------------------------------

    subset_accuracy = accuracy_score(

        y_true,
        y_pred
    )


    hamming_accuracy = (

        1

        -

        hamming_loss(

            y_true,
            y_pred
        )
    )


    # ------------------------------------------------------------
    # F1
    # ------------------------------------------------------------

    micro_f1 = f1_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    macro_f1 = f1_score(

        y_true,
        y_pred,

        average='macro',

        zero_division=0
    )


    weighted_f1 = f1_score(

        y_true,
        y_pred,

        average='weighted',

        zero_division=0
    )


    # ------------------------------------------------------------
    # Precision / Recall
    # ------------------------------------------------------------

    micro_precision = precision_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    micro_recall = recall_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    print(
        f"\nTraining Time      : "
        f"{elapsed / 60:.2f} minutes"
    )

    print(
        f"Subset Accuracy    : "
        f"{subset_accuracy:.4f}"
    )

    print(
        f"Hamming Accuracy   : "
        f"{hamming_accuracy:.4f}"
    )

    print(
        f"Micro Precision    : "
        f"{micro_precision:.4f}"
    )

    print(
        f"Micro Recall       : "
        f"{micro_recall:.4f}"
    )

    print(
        f"Micro F1           : "
        f"{micro_f1:.4f}"
    )

    print(
        f"Macro F1           : "
        f"{macro_f1:.4f}"
    )

    print(
        f"Weighted F1        : "
        f"{weighted_f1:.4f}"
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        "FINAL MICRO F1 FOR ORIGINAL WEIGHT SARCASM"
    )

    print(
        f"{micro_f1:.6f}"
    )

    print(
        "=" * 70
    )


    return {

        'subset_accuracy':
            subset_accuracy,

        'hamming_accuracy':
            hamming_accuracy,

        'micro_precision':
            micro_precision,

        'micro_recall':
            micro_recall,

        'micro_f1':
            micro_f1,

        'macro_f1':
            macro_f1,

        'weighted_f1':
            weighted_f1
    }


# ================================================================
# MAIN
# ================================================================

def main():

    print(
        "\n"
        + "=" * 70
    )

    print("SENSITIVITY ANALYSIS: ORIGINAL HEURISTIC WEIGHTS")


    print(
        "=" * 70
    )


    # ============================================================
    # LOAD DATA
    # ============================================================

    df = load_and_clean()


    # ============================================================
    # TRAIN-TEST SPLIT
    # ============================================================

    df_train, df_test = train_test_split(

        df,

        test_size=TEST_SIZE,

        random_state=RANDOM_STATE
    )


    df_train = df_train.reset_index(
        drop=True
    )


    df_test = df_test.reset_index(
        drop=True
    )


    print(
        f"\nTrain samples : "
        f"{len(df_train):,}"
    )


    print(
        f"Test samples  : "
        f"{len(df_test):,}"
    )


    # ============================================================
    # FEATURE EXTRACTION
    # ============================================================

    print(
        "\n🔧 Extracting features..."
    )


    print("Sarcasm mode: Original heuristic weights")


    X_train, tfidf = extract_features(

        df_train['text'],

        fit=True
    )


    X_test, _ = extract_features(

        df_test['text'],

        tfidf=tfidf,

        fit=False
    )


    print(
        f"\nFeature matrix: "
        f"{X_train.shape[1]:,} features"
    )


    # ============================================================
    # LABEL MATRICES
    # ============================================================

    y_train = df_train[
        LABEL_COLS
    ].values.astype(
        np.int8
    )


    y_test = df_test[
        LABEL_COLS
    ].values.astype(
        np.int8
    )


    print(
        f"Label matrix: "
        f"{y_train.shape}"
    )


    # ============================================================
    # TRAIN LINEAR SVM
    # ============================================================

    print(
        "\n🚀 Training Linear SVM..."
    )


    print(
        "   NOTE: LinearSVC uses CPU, "
        "not the T4 GPU."
    )


    t0 = time.time()


    base_svm = LinearSVC(

        C=0.5,

        max_iter=2000,

        class_weight='balanced',

        dual=False
    )


    model = OneVsRestClassifier(

        base_svm,

        n_jobs=-1
    )


    model.fit(

        X_train,

        y_train
    )


    elapsed = (

        time.time()

        -

        t0
    )


    print(
        f"\n✅ SVM training completed "
        f"in {elapsed / 60:.2f} minutes"
    )


    # ============================================================
    # PREDICTION
    # ============================================================

    print(
        "\n🔮 Generating predictions..."
    )


    decision_scores = model.decision_function(

        X_test
    )


    y_pred = (

        decision_scores
        >= THRESHOLD
    ).astype(
        np.int8
    )


    # ============================================================
    # EVALUATION
    # ============================================================

    results = evaluate(

        y_test,

        y_pred,

        elapsed
    )


    return (

        model,

        tfidf,

        y_test,

        y_pred,

        results
    )


# ================================================================
# RUN
# ================================================================

if __name__ == '__main__':

    model, tfidf, y_test, y_pred, results = main()


SENSITIVITY ANALYSIS: ORIGINAL HEURISTIC WEIGHTS
📂 Loading data...
   684,383 → 684,172 rows after cleaning

Train samples : 547,337
Test samples  : 136,835

🔧 Extracting features...
Sarcasm mode: Original heuristic weights
   Extracting features...
   Fitting TF-IDF...
   Extracting features...

Feature matrix: 20,055 features
Label matrix: (547337, 13)

🚀 Training Linear SVM...
   NOTE: LinearSVC uses CPU, not the T4 GPU.

✅ SVM training completed in 76.16 minutes

🔮 Generating predictions...

ORIGINAL-WEIGHT SARCASM SVM RESULTS

Training Time      : 76.16 minutes
Subset Accuracy    : 0.3163
Hamming Accuracy   : 0.8506
Micro Precision    : 0.3615
Micro Recall       : 0.8034
Micro F1           : 0.4986
Macro F1           : 0.5209
Weighted F1        : 0.5334

FINAL MICRO F1 FOR ORIGINAL WEIGHT SARCASM
0.498588


In [ ]:
# ============================================================
# UNIFORM-WEIGHT SARCASM SCORE
# All seven heuristic coefficients = 1.0
# ============================================================

import re
import numpy as np
import pandas as pd


class UniformSarcasmFeatures:

    # --------------------------------------------------------
    # 1. Positive sentiment words
    # --------------------------------------------------------

    _POSITIVE_WORDS = {
        'great', 'wonderful', 'amazing', 'fantastic', 'brilliant',
        'excellent', 'perfect', 'awesome', 'lovely', 'beautiful',
        'nice', 'good', 'fine', 'super', 'best', 'love', 'like',
        'happy', 'glad', 'thrilled', 'exciting', 'interesting',
        'helpful', 'useful', 'smart', 'clever', 'genius',
        'impressive', 'outstanding', 'incredible', 'fabulous'
    }

    # --------------------------------------------------------
    # 2. Negative sentiment words
    # --------------------------------------------------------

    _NEGATIVE_WORDS = {
        'hate', 'terrible', 'awful', 'horrible', 'disgusting',
        'pathetic', 'stupid', 'idiot', 'moron', 'useless',
        'worthless', 'trash', 'garbage', 'fail', 'failure',
        'worst', 'bad', 'ugly', 'dumb', 'loser', 'joke',
        'seriously', 'really', 'actually'
    }

    # --------------------------------------------------------
    # 3. Sarcasm marker phrases
    # --------------------------------------------------------

    _SARCASM_MARKERS = [
        r'\boh great\b',
        r'\byeah right\b',
        r'\bsure sure\b',
        r'\bthanks a lot\b',
        r'\bthanks so much\b',
        r'\bso helpful\b',
        r'\breally helpful\b',
        r'\bso smart\b',
        r'\bwow thanks\b',
        r'\boh wow\b',
        r'\boh really\b',
        r'\bno kidding\b',
        r'\bno way\b',
        r'\bright sure\b',
        r'\bas if\b',
        r'\bcongrats\b',
        r'\bcongratulations\b',
        r'\bwhat a surprise\b',
        r'\bshocking\b',
        r'\bunbelievable\b',
        r'\bi\'m sure\b',
        r'\bim sure\b',
        r'\bi bet\b',
        r'\bof course\b',
        r'\bobviously\b',
        r'\bclearly\b',
        r'\bjust great\b',
        r'\bjust perfect\b',
        r'\bso funny\b',
        r'\bhilarious\b',
    ]

    # --------------------------------------------------------
    # 4. Contradiction patterns
    # --------------------------------------------------------

    _CONTRADICTION_PAIRS = [
        (
            r'\b(?:great|wonderful|amazing|fantastic)\b',
            r'\b(?:not|never|no|hate|terrible)\b'
        ),
        (
            r'\b(?:love|like|enjoy)\b',
            r'\b(?:not|never|no|hate|dislike)\b'
        ),
        (
            r'\b(?:good|nice|fine)\b',
            r'\b(?:not|terrible|awful|horrible)\b'
        ),
    ]

    # --------------------------------------------------------
    # 5. Initialize regular expressions
    # --------------------------------------------------------

    def __init__(self):

        self._marker_pats = [
            re.compile(pattern, re.IGNORECASE)
            for pattern in self._SARCASM_MARKERS
        ]

        self._contra_pats = [
            (
                re.compile(positive_pattern, re.IGNORECASE),
                re.compile(negative_pattern, re.IGNORECASE)
            )
            for positive_pattern, negative_pattern
            in self._CONTRADICTION_PAIRS
        ]

        self._intensifier_re = re.compile(
            r'\b(?:so|very|extremely|incredibly|absolutely|totally|'
            r'completely|literally|seriously|really|such|beyond|super)\b',
            re.IGNORECASE
        )

        self._pos_emoji_re = re.compile(
            r'[\U0001F600-\U0001F606'
            r'\U0001F609\U0001F60A'
            r'\U0001F60D\U0001F618'
            r'\U0001F61C\U0001F61D'
            r'\U0001F923]'
        )

    # --------------------------------------------------------
    # 6. Calculate uniform-weight sarcasm score
    # --------------------------------------------------------

    def transform(self, texts):

        sarcasm_scores = []

        for text in texts:

            t = str(text)
            lower = t.lower()

            # Word count
            wc = max(len(lower.split()), 1)

            # Lowercase words
            lower_words = [
                word.strip('.,!?;:\'"')
                for word in lower.split()
            ]

            # ------------------------------------------------
            # Calculate the SAME seven indicators
            # used in the weighted version
            # ------------------------------------------------

            # m_c : sarcasm marker count
            mc = sum(
                1
                for pattern in self._marker_pats
                if pattern.search(t)
            )

            # e_x : repeated exclamation pattern count
            ex = len(
                re.findall(r'!{2,}', t)
            )

            # c_c : contradiction count
            cc = sum(
                1
                for positive_pattern, negative_pattern
                in self._contra_pats
                if positive_pattern.search(t)
                and negative_pattern.search(t)
            )

            # p_c : positive word count
            pc = sum(
                1
                for word in lower_words
                if word in self._POSITIVE_WORDS
            )

            # n_c : negative word count
            nc = sum(
                1
                for word in lower_words
                if word in self._NEGATIVE_WORDS
            )

            # Indicator function:
            # I[p_c > 0 AND n_c > 0]
            pos_neg_flag = int(
                pc > 0 and nc > 0
            )

            # r_caps : ratio of ALL-CAPS words
            caps_words = len(
                re.findall(r'\b[A-Z]{2,}\b', t)
            )

            rcaps = caps_words / wc

            # i_d : intensifier word density
            intensifier_count = len(
                self._intensifier_re.findall(t)
            )

            id = intensifier_count / wc

            # e_c : emoji-text sentiment contradiction
            ec = int(
                len(self._pos_emoji_re.findall(t)) > 0
                and nc > 0
            )

            # ------------------------------------------------
            # Uniform-weight heuristic score
            #
            # ONLY THE SEVEN COEFFICIENTS ARE CHANGED
            #
            # Weighted:
            # 2.0, 1.5, 2.0, 1.5, 3.0, 2.0, 1.5
            #
            # Uniform:
            # 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0
            #
            # Denominator remains 10.0
            # ------------------------------------------------

            s = min(
                (
                    1.0 * mc
                    + 1.0 * ex
                    + 1.0 * cc
                    + 1.0 * pos_neg_flag
                    + 1.0 * rcaps
                    + 1.0 * id
                    + 1.0 * ec
                ) / 10.0,
                1.0
            )

            # Store ONLY final normalized score s
            sarcasm_scores.append(s)

        # ----------------------------------------------------
        # Return ONLY final normalized sarcasm score
        # ----------------------------------------------------

        return np.array(
            sarcasm_scores,
            dtype=np.float32
        ).reshape(-1, 1)


# ============================================================
# EXECUTE UNIFORM-WEIGHT SARCASM FEATURE EXTRACTION
# ============================================================

# Assumes 'df' already exists and contains the 'text' column.

uniform_sarcasm_extractor = UniformSarcasmFeatures()

X_sarcasm_uniform = uniform_sarcasm_extractor.transform(
    df['text']
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 60)
print("UNIFORM-WEIGHT SARCASM FEATURE EXTRACTION COMPLETED")
print("=" * 60)

print("Number of text samples:", len(df))

print("Uniform sarcasm feature shape:",
      X_sarcasm_uniform.shape)

print("Feature used by the model:")
print("sarc_heuristic_score")

print("\nFirst 10 uniform-weight sarcasm scores:")

for i, score in enumerate(
    X_sarcasm_uniform[:10].flatten(),
    start=1
):
    print(f"Sample {i}: {score:.6f}")

print("\nMinimum uniform sarcasm score:",
      X_sarcasm_uniform.min())

print("Maximum uniform sarcasm score:",
      X_sarcasm_uniform.max())

print("Mean uniform sarcasm score:",
      X_sarcasm_uniform.mean())

print("=" * 60)

UNIFORM-WEIGHT SARCASM FEATURE EXTRACTION COMPLETED
Number of text samples: 683935
Uniform sarcasm feature shape: (683935, 1)
Feature used by the model:
sarc_heuristic_score

First 10 uniform-weight sarcasm scores:
Sample 1: 0.000000
Sample 2: 0.100000
Sample 3: 0.000000
Sample 4: 0.000000
Sample 5: 0.101852
Sample 6: 0.000000
Sample 7: 0.000000
Sample 8: 0.002174
Sample 9: 0.000000
Sample 10: 0.000000

Minimum uniform sarcasm score: 0.0
Maximum uniform sarcasm score: 1.0
Mean uniform sarcasm score: 0.017221335


In [ ]:
from feature_engineering import HateBERTFeatures, CyberBERTFeatures

PROJECT_PATH = "/content/drive/MyDrive/Cyberbullying_Project"

hatebert = HateBERTFeatures(batch_size=32)
hatebert.extract(
    df_test["text"],
    cache_path=f"{PROJECT_PATH}/hb_test.npy"
)

cyberbert = CyberBERTFeatures(batch_size=32)
cyberbert.extract(
    df_test["text"],
    cache_path=f"{PROJECT_PATH}/cb_test.npy"
)

print("✅ Test embeddings saved!")

📂 Loading HateBERT cache
Loaded (136787, 768)
📂 Loading CyberBERT cache
Loaded (136787, 768)
✅ Test embeddings saved!


In [ ]:
!pip install -q transformers torch

from feature_engineering import build_features, FeatureConfig

PROJECT_PATH = "/content/drive/MyDrive/Cyberbullying_Project"

config = FeatureConfig(
    use_hatebert=True,
    use_cyberbert=True
)

train_folder, test_folder, names, pipeline = build_features(
    df_train,
    df_test,
    config=config,
    hatebert_train_cache=f"{PROJECT_PATH}/hb_train.npy",
    hatebert_test_cache=f"{PROJECT_PATH}/hb_test.npy",
    cyberbert_train_cache=f"{PROJECT_PATH}/cb_train.npy",
    cyberbert_test_cache=f"{PROJECT_PATH}/cb_test.npy",
)

print("✅ Feature extraction completed!")

Fitting TF-IDF vocabulary
Vocabulary size : 20,000
FEATURE ENGINEERING PIPELINE

TRAIN : 547,148 samples

Processing TRAIN chunk 0:5000
[0:5000] HateKeyword Features
[0:5000] POS Features
[0:5000] Word Frequency
[0:5000] Sarcasm
[0:5000] TF-IDF
[0:5000] HateBERT
📂 Loading HateBERT cache
Loaded (5000, 768)
[0:5000] CyberBERT
📂 Loading CyberBERT cache
Loaded (5000, 768)
[0:5000] Combining matrices
Chunk Shape : (5000, 21608)
Saved train_chunk_000.npz

Processing TRAIN chunk 5000:10000
[5000:10000] HateKeyword Features
[5000:10000] POS Features
[5000:10000] Word Frequency
[5000:10000] Sarcasm
[5000:10000] TF-IDF
[5000:10000] HateBERT
📂 Loading HateBERT cache
Loaded (5000, 768)
[5000:10000] CyberBERT
📂 Loading CyberBERT cache
Loaded (5000, 768)
[5000:10000] Combining matrices
Chunk Shape : (5000, 21608)
Saved train_chunk_001.npz

Processing TRAIN chunk 10000:15000
[10000:15000] HateKeyword Features
[10000:15000] POS Features
[10000:15000] Word Frequency
[10000:15000] Sarcasm
[10000:15000] 

In [ ]:
import os
from scipy.sparse import load_npz

train_folder = "/content/drive/MyDrive/Cyberbullying_Project/train_chunks"

files = sorted(f for f in os.listdir(train_folder) if f.endswith(".npz"))

print("Number of train chunks:", len(files))

first = load_npz(os.path.join(train_folder, files[0]))
last = load_npz(os.path.join(train_folder, files[-1]))

print("First chunk shape:", first.shape)
print("Last chunk shape:", last.shape)



Number of train chunks: 110
First chunk shape: (5000, 21608)
Last chunk shape: (2148, 21608)


KeyboardInterrupt: 

In [ ]:
import os
from scipy.sparse import load_npz

train_folder = "/content/drive/MyDrive/Cyberbullying_Project/train_chunks"

total_nnz = 0

for f in sorted(os.listdir(train_folder)):
    if f.endswith(".npz"):
        X = load_npz(os.path.join(train_folder, f))
        total_nnz += X.nnz

print("Total non-zero values:", total_nnz)

Total non-zero values: 869040601


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

model = OneVsRestClassifier(
    LinearSVC(C=0.5)
)

model.fit(X_train, y_train)

print("✅ SVM training completed!")

NameError: name 'X_train' is not defined

In [ ]:
"""
mc_naive_bayes.py  —  Multiclass | Self-Contained
Cyberbullying multiclass classification — ONE label per text.
  Features: HateKeyword+POS+WordFreq+Sarcasm+TF-IDF (20,072 features)
  Labels  : 13 categories (one per text, by priority)
  NO external files needed.

Run: python mc_naive_bayes.py
"""
import re, time
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report
)

# ── Config ────────────────────────────────────────────────────
DATA_PATH   = 'cyberbullying.csv'
SAMPLE_SIZE = None
TEST_SIZE   = 0.2
TFIDF_MAX   = 20000

LABEL_COLS = [
    'religious_hate','ethnic_hate','age_discrimination','gender_hate',
    'sexual_harassment','threats','body_shaming','political_hate',
    'trolling','mental_hate','discrimination','other_cyberbullying_types',
    'not_cyberbullying',
]
LABEL_PRIORITY = [
    'threats','sexual_harassment','ethnic_hate','religious_hate',
    'gender_hate','mental_hate','body_shaming','discrimination',
    'age_discrimination','political_hate','trolling',
    'other_cyberbullying_types','not_cyberbullying',
]
HATE_KEYWORDS = {
    'religious_hate':    ['infidel','kafir','blasphemy','blasphemer','heretic','cult','crusade','jihad','pagan','apostate','heathen','godless','anti-christian','anti-islam','anti-hindu','anti-semitic','zionist','islamophobe','christophobe'],
    'ethnic_hate':       ['nigger','nigga','chink','spic','wetback','gook','kike','raghead','cracker','redneck','monkey','ape','beaner','immigrant','illegal alien','foreigner','outsider','invader','terrorist'],
    'gender_hate':       ['bitch','whore','slut','cunt','feminazi','tranny','dyke','faggot','sissy','mangina','thot','hoe','skank','femoid','incel','simp','soyboy','misogynist','misandrist'],
    'sexual_harassment': ['rape','rapist','molest','grope','pervert','predator','creep','harass','assault'],
    'threats':           ['kill','murder','shoot','stab','bomb','attack','destroy','hurt','harm','threat','die','execute','eliminate','exterminate','torture','burn','acid','explode','lynch','hang','beat','bash','smash'],
    'body_shaming':      ['fat','ugly','obese','disgusting','gross','pig','whale','skinny','anorexic','hideous','repulsive','deformed','freak','monster','troll','goblin'],
    'mental_hate':       ['retard','retarded','psycho','crazy','insane','lunatic','mental','schizo','bipolar','autistic','mentally ill','nutcase','bonkers','deranged','demented'],
    'general_hate':      ['hate','stupid','idiot','moron','dumb','loser','trash','garbage','scum','filth','worthless','pathetic','useless','despise','abhor','vile','toxic','cancer','parasite','vermin','subhuman'],
}
POS_PATTERNS = {
    'NOUN': r'\b(?:man|woman|people|person|thing|way|day|time|year|world|life|child|children|hand|part|place|case|week|company|system|question|government|number|night|point|home|water|room|mother|area|money|story|fact|month|lot|right|study|book|eye|job|word|business|issue|side|kind|head|house|service|friend|father|power|hour|game|line|end|member|city|community|name|president|team|minute|idea|body|information|back|parent|face|level|office|door|health|art|war|history|party|result|change|morning|reason|research|girl|guy|moment|air|teacher|force|education)\b',
    'VERB': r'\b(?:is|are|was|were|be|been|being|have|has|had|do|does|did|will|would|could|should|may|might|shall|must|can|go|get|make|know|think|take|see|come|want|look|use|find|give|tell|work|call|try|ask|need|feel|become|leave|put|mean|keep|let|begin|show|hear|play|run|move|live|believe|hold|bring|happen|write|provide|sit|stand|lose|pay|meet|include|continue|set|learn|change|lead|understand|watch|follow|stop|create|speak|read|spend|grow|open|walk|win|offer|remember|love|consider|appear|buy|wait|serve|die|send|expect|build|stay|fall|cut|reach|kill|remain|suggest|raise|pass|sell|require|report|decide|pull)\b',
    'ADJ':  r'\b(?:good|new|first|last|long|great|little|own|other|old|right|big|high|different|small|large|next|early|young|important|public|private|real|best|free|few|same|able|political|social|economic|national|possible|local|white|black|strong|true|hot|happy|sad|bad|ugly|fat|stupid|evil|dangerous|terrible|horrible|awful|disgusting|pathetic|worthless|useless|dumb|crazy|insane|violent|racist|sexist|offensive|hateful|toxic|radical|extreme)\b',
    'ADV':  r'\b(?:up|so|out|just|now|how|then|more|also|here|well|only|very|even|back|there|down|still|in|as|too|really|most|never|much|often|always|actually|again|further|yet|already|soon|especially|finally|simply|probably|certainly|clearly|literally|absolutely|totally|completely|definitely|obviously|seriously|exactly|nearly|directly|quickly|easily)\b',
    'PRON': r'\b(?:i|me|my|myself|you|your|yourself|he|him|his|himself|she|her|hers|herself|it|its|itself|we|us|our|ourselves|they|them|their|theirs|themselves|this|that|these|those|who|whom|which|what)\b',
}
PROFANITY      = {'fuck','fucking','fucked','shit','ass','asshole','bitch','cunt','bastard','damn','piss','cock','dick','pussy','motherfucker','whore','slut','idiot','moron','loser','stupid','dumb','retard'}
SARC_MARKERS   = [r'\boh great\b',r'\byeah right\b',r'\bsure sure\b',r'\bthanks a lot\b',r'\bthanks so much\b',r'\bso helpful\b',r'\breally helpful\b',r'\bso smart\b',r'\bwow thanks\b',r'\boh wow\b',r'\boh really\b',r'\bno kidding\b',r'\bno way\b',r'\bright sure\b',r'\bas if\b',r'\bcongrats\b',r'\bcongratulations\b',r'\bwhat a surprise\b',r'\bshocking\b',r'\bunbelievable\b',r"\bi\'m sure\b",r'\bim sure\b',r'\bi bet\b',r'\bof course\b',r'\bobviously\b',r'\bclearly\b',r'\bjust great\b',r'\bjust perfect\b',r'\bso funny\b',r'\bhilarious\b']
SARC_POS_WORDS = {'great','wonderful','amazing','fantastic','brilliant','excellent','perfect','awesome','lovely','nice','good','fine','super','best','love','happy','glad','thrilled','helpful','smart','clever','impressive','incredible'}
SARC_NEG_WORDS = {'hate','terrible','awful','horrible','disgusting','pathetic','stupid','idiot','moron','useless','worthless','trash','garbage','fail','worst','bad','ugly','dumb','loser'}
SARC_CONTRA    = [(r'\b(?:great|wonderful|amazing|fantastic)\b',r'\b(?:not|never|no|hate|terrible)\b'),(r'\b(?:love|like|enjoy)\b',r'\b(?:not|never|no|hate|dislike)\b'),(r'\b(?:good|nice|fine)\b',r'\b(?:not|terrible|awful|horrible)\b')]
_INTENSIFIER_RE= re.compile(r'\b(?:so|very|extremely|incredibly|absolutely|totally|completely|literally|seriously|really|such|beyond|super)\b', re.IGNORECASE)
_POS_EMOJI_RE  = re.compile(r'[\U0001F600-\U0001F606\U0001F609\U0001F60A\U0001F60D\U0001F618\U0001F61C\U0001F61D\U0001F923]')


def to_multiclass(df):
    def pick_label(row):
        for label in LABEL_PRIORITY:
            if row[label] == 1:
                return label
        return 'not_cyberbullying'
    df['label'] = df[LABEL_COLS].apply(pick_label, axis=1)
    return df


def load_and_clean(data_path, sample_size):
    print("📂 Loading data...")
    df = pd.read_csv(data_path)
    before = len(df)
    df = df[df['text'].notnull() & (df['text'].str.strip() != '')]
    df = df[df['text'].astype(str).str.contains(r'[a-zA-Z]', regex=True)]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^@\w+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^\d+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^[^\w\s]+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^(.)\1+$|^(\w+)(\s+\2)+\s*$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^(#\w+\s*)+$')]
    df = df.reset_index(drop=True)
    df = to_multiclass(df)
    print(f"   {before:,} → {len(df):,} rows after cleaning")
    print("   Label distribution:")
    for lbl, cnt in df['label'].value_counts().items():
        print(f"     {lbl:<35} {cnt:>8,}")
    if sample_size:
        df = df.sample(min(sample_size, len(df)), random_state=42).reset_index(drop=True)
        print(f"   Sampled: {len(df):,} rows")
    return df


def extract_features(texts, tfidf=None, fit=False, tfidf_max=20000):
    hate_pats   = {cat: re.compile(r'\b(?:' + '|'.join(re.escape(k) for k in kws) + r')\b', re.IGNORECASE) for cat, kws in HATE_KEYWORDS.items()}
    pos_pats    = {pos: re.compile(pat, re.IGNORECASE) for pos, pat in POS_PATTERNS.items()}
    marker_pats = [re.compile(p, re.IGNORECASE) for p in SARC_MARKERS]
    contra_pats = [(re.compile(a, re.IGNORECASE), re.compile(b, re.IGNORECASE)) for a, b in SARC_CONTRA]
    url_re=re.compile(r'https?://\S+|www\.\S+'); mention_re=re.compile(r'@\w+')
    hashtag_re=re.compile(r'#\w+'); emoji_re=re.compile(r'[\U0001F300-\U0001F9FF\U00002600-\U000027FF]',flags=re.UNICODE)
    repeat_re=re.compile(r'(.)\1{2,}'); caps_re=re.compile(r'\b[A-Z]{2,}\b')
    all_rows=[]
    for text in texts:
        t=str(text); words=t.split(); wc=max(len(words),1); cc=max(len(t),1)
        lw=[w.lower().strip('.,!?;:\'"') for w in words]
        sents=max(len(re.split(r'[.!?]+',t)),1); wfreq={}
        for w in lw: wfreq[w]=wfreq.get(w,0)+1
        caps=caps_re.findall(t); prof=sum(1 for w in lw if w in PROFANITY); row={}
        total_hate=0
        for cat,pat in hate_pats.items():
            c=len(pat.findall(t)); row[f'hate_count_{cat}']=c; row[f'hate_flag_{cat}']=int(c>0); total_hate+=c
        row['hate_total_count']=total_hate; row['hate_density']=total_hate/wc
        row['hate_category_count']=sum(1 for cat in HATE_KEYWORDS if row[f'hate_flag_{cat}']>0)
        pcounts={}
        for pos,pat in pos_pats.items():
            c=len(pat.findall(t)); pcounts[pos]=c; row[f'pos_count_{pos.lower()}']=c; row[f'pos_ratio_{pos.lower()}']=c/wc
        row['pos_adj_noun_ratio']=pcounts.get('ADJ',0)/max(pcounts.get('NOUN',0),1)
        row['pos_verb_density']=pcounts.get('VERB',0)/wc; row['pos_pron_density']=pcounts.get('PRON',0)/wc
        row['pos_modifier_density']=(pcounts.get('ADJ',0)+pcounts.get('ADV',0))/wc
        row.update({'wf_char_count':cc,'wf_word_count':wc,'wf_sentence_count':sents,
            'wf_avg_word_len':np.mean([len(w) for w in words]) if words else 0,'wf_avg_sent_len':wc/sents,
            'wf_unique_word_ratio':len(set(lw))/wc,'wf_caps_word_count':len(caps),'wf_caps_ratio':len(caps)/wc,
            'wf_upper_char_ratio':sum(1 for c in t if c.isupper())/cc,'wf_exclaim_count':t.count('!'),
            'wf_question_count':t.count('?'),'wf_ellipsis_count':t.count('...'),
            'wf_punct_density':sum(1 for c in t if c in '!?.,;:\'"')/cc,
            'wf_url_count':len(url_re.findall(t)),'wf_mention_count':len(mention_re.findall(t)),
            'wf_hashtag_count':len(hashtag_re.findall(t)),'wf_emoji_count':len(emoji_re.findall(t)),
            'wf_repeat_char_count':len(repeat_re.findall(t)),'wf_repeat_word_count':sum(1 for v in wfreq.values() if v>1),
            'wf_profanity_count':prof,'wf_profanity_density':prof/wc})
        multi_exc=len(re.findall(r'!{2,}',t)); mc=sum(1 for p in marker_pats if p.search(t))
        pc=sum(1 for w in lw if w in SARC_POS_WORDS); nc=sum(1 for w in lw if w in SARC_NEG_WORDS)
        cc2=sum(1 for a_p,b_p in contra_pats if a_p.search(t) and b_p.search(t))
        id_=len(_INTENSIFIER_RE.findall(t))/wc; ec=int(len(_POS_EMOJI_RE.findall(t))>0 and nc>0)
        ss=min((mc*2.0+multi_exc*1.5+cc2*2.0+int(pc>0 and nc>0)*1.5+(len(caps)/wc)*3.0+id_*2.0+ec*1.5)/10.0,1.0)
        row.update({'sarc_exclaim_count':t.count('!'),'sarc_multi_exclaim':multi_exc,
            'sarc_question_exclaim':len(re.findall(r'[!?]{2,}',t)),'sarc_ellipsis_count':t.count('...'),
            'sarc_caps_words':len(caps),'sarc_caps_ratio':len(caps)/wc,'sarc_marker_count':mc,'sarc_marker_density':mc/wc,
            'sarc_pos_word_count':pc,'sarc_neg_word_count':nc,'sarc_pos_neg_mix':int(pc>0 and nc>0),
            'sarc_pos_neg_ratio':pc/max(nc,1),'sarc_contradiction_count':cc2,
            'sarc_quote_count':t.count('"')+t.count("'"),'sarc_has_quotes':int(t.count('"')+t.count("'")>=2),
            'sarc_emoji_contradiction':ec,'sarc_intensifier_density':id_,'sarc_heuristic_score':ss})
        all_rows.append(row)
    dense=pd.DataFrame(all_rows).values.astype(np.float32); dense=np.clip(dense,0,None)
    if fit:
        tfidf=TfidfVectorizer(max_features=tfidf_max,ngram_range=(1,2),min_df=3,sublinear_tf=True)
        tfidf_mat=tfidf.fit_transform(texts.astype(str))
    else:
        tfidf_mat=tfidf.transform(texts.astype(str))
    return hstack([csr_matrix(dense),tfidf_mat],format='csr'),tfidf


def evaluate(model_name, y_true, y_pred, elapsed=None):
    print("\n"+"="*58)
    print(f"  📊 {model_name} — Accuracy Report (Multiclass)")
    if elapsed: print(f"  ⏱  Training time: {elapsed:.1f}s")
    print("="*58)
    print(f"  Accuracy         : {accuracy_score(y_true, y_pred):.4f}  ✅ main")
    print(f"  F1 Macro         : {f1_score(y_true, y_pred, average='macro',    zero_division=0):.4f}")
    print(f"  F1 Weighted      : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"  F1 Micro         : {f1_score(y_true, y_pred, average='micro',    zero_division=0):.4f}")
    print(f"  Precision(macro) : {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"  Recall(macro)    : {recall_score(y_true,    y_pred, average='macro', zero_division=0):.4f}")
    print("\n  Per-Class Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("="*58)


def detect(text, model, tfidf, svd=None):
    X,_ = extract_features(pd.Series([text]), tfidf=tfidf, fit=False)
    if svd is not None:
        X_in = svd.transform(X)
    else:
        X_in = X.copy(); X_in.data = np.clip(X_in.data, 0, None)
    pred  = model.predict(X_in)[0]
    proba = model.predict_proba(X_in)[0]
    names = model.classes_
    print("\n"+"="*52)
    print(f"  📝 Text : {text[:65]}{'...' if len(text)>65 else ''}")
    print("="*52)
    if pred == 'not_cyberbullying':
        print(f"  ✅  NOT CYBERBULLYING  (confidence: {proba.max():.0%})")
    else:
        print(f"  🚨  CYBERBULLYING DETECTED")
        print(f"  Category  : {pred}")
        print(f"  Confidence: {proba.max():.0%}")
        print("\n  Top probabilities:")
        for i in proba.argsort()[::-1][:5]:
            bar  = '█'*int(proba[i]*10)+'░'*(10-int(proba[i]*10))
            flag = '🚨' if names[i] != 'not_cyberbullying' else '✅'
            print(f"    {flag} {names[i]:<35} {proba[i]:.0%}  {bar}")
    print("="*52)
    return pred


def run_detector(model_name, model, tfidf, svd=None):
    print("\n"+"="*52)
    print(f"  🤖 Cyberbullying Detector ({model_name}) — Multiclass")
    print("  Type 'exit' to stop")
    print("="*52+"\n")
    while True:
        text = input("Enter text: ").strip()
        if text.lower() == 'exit': print("👋 Stopping..."); break
        if not text: print("⚠️  Please enter some text\n"); continue
        detect(text, model, tfidf, svd)


# ── Main ──────────────────────────────────────────────────────
def main():
    print("="*58)
    print("  MODEL: Naive Bayes — Multiclass")
    print("="*58)

    df = load_and_clean(DATA_PATH, SAMPLE_SIZE)
    df_train, df_test = train_test_split(df, test_size=TEST_SIZE,
                                          random_state=42, stratify=df['label'])
    df_train = df_train.reset_index(drop=True)
    df_test  = df_test.reset_index(drop=True)
    print(f"   Train: {len(df_train):,}  |  Test: {len(df_test):,}")

    print("\n🔧 Extracting features...")
    X_train, tfidf = extract_features(df_train['text'], fit=True, tfidf_max=TFIDF_MAX)
    X_test,  _     = extract_features(df_test['text'],  tfidf=tfidf, tfidf_max=TFIDF_MAX)
    print(f"   ✅ Features: {X_train.shape[1]:,}")

    y_train = df_train['label'].values
    y_test  = df_test['label'].values

    print("\n🚀 Training Naive Bayes...")
    t0 = time.time()
    svd = None

    from sklearn.naive_bayes import ComplementNB
    X_tr = X_train.copy(); X_tr.data = np.clip(X_tr.data, 0, None)
    model = ComplementNB(alpha=0.1)
    model.fit(X_tr, y_train)
    X_te = X_test.copy(); X_te.data = np.clip(X_te.data, 0, None)
    y_pred = model.predict(X_te)

    elapsed = time.time()-t0
    print(f"   Done in {elapsed:.1f}s")

    evaluate('Naive Bayes', y_test, y_pred, elapsed)
    run_detector('Naive Bayes', model, tfidf, None)
    return model, tfidf

if __name__ == '__main__':
    main()

  MODEL: Naive Bayes — Multiclass
📂 Loading data...
   684,383 → 684,172 rows after cleaning
   Label distribution:
     not_cyberbullying                    107,336
     sexual_harassment                     62,503
     threats                               62,231
     gender_hate                           56,874
     ethnic_hate                           54,934
     mental_hate                           50,459
     body_shaming                          48,271
     discrimination                        46,484
     age_discrimination                    44,824
     religious_hate                        41,174
     trolling                              40,388
     political_hate                        38,263
     other_cyberbullying_types             30,431
   Train: 547,337  |  Test: 136,835

🔧 Extracting features...
   ✅ Features: 20,072

🚀 Training Naive Bayes...
   Done in 2.3s

  📊 Naive Bayes — Accuracy Report (Multiclass)
  ⏱  Training time: 2.3s
  Accuracy         : 0.5602  ✅ mai

In [ ]:
"""
realtime_binary_test.py
=======================
Real-time cyberbullying detection (Bullying vs Not Bullying)
"""

import re
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import train_test_split

# ── Config ─────────────────────────
DATA_PATH = "cyberbullying.csv"

LABEL_COLS = [
'religious_hate','ethnic_hate','age_discrimination','gender_hate',
'sexual_harassment','threats','body_shaming','political_hate',
'trolling','mental_hate','discrimination','other_cyberbullying_types'
]

# ── Feature Extraction ─────────────
def extract_features(texts, tfidf=None, fit=False):

    dense_features = []

    for text in texts:
        t = str(text)

        features = [
            len(t.split()),
            len(t),
            t.count("!"),
            t.count("?"),
            len(re.findall(r'@\w+', t)),
            len(re.findall(r'#\w+', t)),
            len(re.findall(r'[A-Z]{2,}', t))
        ]

        dense_features.append(features)

    dense = np.array(dense_features).astype(np.float32)

    if fit:
        tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2))
        tfidf_mat = tfidf.fit_transform(texts.astype(str))
    else:
        tfidf_mat = tfidf.transform(texts.astype(str))

    return hstack([csr_matrix(dense), tfidf_mat]), tfidf


# ── Train Model ────────────────────
def train_model():

    print("Loading dataset...")

    df = pd.read_csv(DATA_PATH)

    df = df[df["text"].notnull()].reset_index(drop=True)

    # Convert multilabel → binary
    df["bullying"] = (df[LABEL_COLS].sum(axis=1) > 0).astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        df["text"], df["bullying"], test_size=0.2, random_state=42
    )

    print("Extracting features...")

    X_train_vec, tfidf = extract_features(X_train, fit=True)

    print("Training model...")

    model = ComplementNB(alpha=0.1)
    model.fit(X_train_vec, y_train)

    print("Model Ready!")

    return model, tfidf


# ── Real-Time Testing ─────────────
def realtime_test(model, tfidf):

    print("\n==============================")
    print("Cyberbullying Detection")
    print("Type 'exit' to stop")
    print("==============================")

    while True:

        text = input("\nEnter text: ")

        if text.lower() == "exit":
            print("Stopping...")
            break

        df = pd.DataFrame({"text":[text]})

        X_input, _ = extract_features(df["text"], tfidf=tfidf)

        pred = model.predict(X_input)[0]

        if pred == 1:
            print("⚠️  Bullying Detected")
        else:
            print("✅ Not Bullying")


# ── Main ──────────────────────────
if __name__ == "__main__":

    model, tfidf = train_model()

    realtime_test(model, tfidf)

Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'cyberbullying.csv'

In [ ]:
"""
model_logistic_regression.py  —  Self-Contained
=================================================
Logistic Regression for multi-label cyberbullying classification.
No external model_utils.py or feature_engineering.py needed.

Run:  python model_logistic_regression.py
"""

import re
import time
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, hamming_loss, precision_score, recall_score

# ── Config ────────────────────────────────────────────────────
DATA_PATH   = 'compressed_data_csv'   # ← change path if needed
SAMPLE_SIZE = None                     # e.g. 50000 for faster run; None = full
TEST_SIZE   = 0.2
THRESHOLD   = 0.3                      # probability threshold for positive label
LABEL_COLS  = [
    'religious_hate','ethnic_hate','age_discrimination','gender_hate',
    'sexual_harassment','threats','body_shaming','political_hate',
    'trolling','mental_hate','discrimination','other_cyberbullying_types',
    'not_cyberbullying',
]

# ── Hate Keywords ─────────────────────────────────────────────
HATE_KEYWORDS = {
    'religious_hate':    ['infidel','kafir','blasphemy','heretic','cult','jihad','pagan','apostate','heathen','godless'],
    'ethnic_hate':       ['nigger','nigga','chink','spic','wetback','gook','kike','raghead','cracker','monkey','ape'],
    'gender_hate':       ['bitch','whore','slut','cunt','feminazi','tranny','dyke','faggot','sissy','thot','hoe','skank'],
    'sexual_harassment': ['rape','rapist','molest','grope','pervert','predator','harass','assault'],
    'threats':           ['kill','murder','shoot','stab','bomb','attack','destroy','hurt','harm','die','execute','lynch'],
    'body_shaming':      ['fat','ugly','obese','disgusting','gross','pig','whale','hideous','repulsive'],
    'mental_hate':       ['retard','retarded','psycho','crazy','insane','lunatic','schizo','deranged'],
    'general_hate':      ['hate','stupid','idiot','moron','dumb','loser','trash','garbage','scum','worthless','pathetic'],
}
POS_PATTERNS = {
    'VERB': r'\b(?:is|are|was|were|be|been|have|has|had|do|does|did|will|would|could|should|kill|die|hurt|hate|destroy|attack)\b',
    'ADJ':  r'\b(?:good|bad|ugly|fat|stupid|evil|dangerous|terrible|horrible|awful|disgusting|pathetic|worthless|hateful|toxic|racist|violent)\b',
    'PRON': r'\b(?:i|me|my|you|your|he|him|his|she|her|they|them|their|we|us|our|it|its)\b',
}
PROFANITY = {'fuck','fucking','shit','ass','asshole','bitch','cunt','bastard','damn','piss',
             'cock','dick','pussy','motherfucker','whore','slut','idiot','moron','retard'}

# ── Step 1: Load & Clean ──────────────────────────────────────
def load_and_clean():
    print("📂 Loading data...")
    df = pd.read_csv(DATA_PATH)
    before = len(df)
    df = df[df['text'].notnull() & (df['text'].str.strip() != '')]
    df = df[df['text'].astype(str).str.contains(r'[a-zA-Z]', regex=True)]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^@\w+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^\d+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^[^\w\s]+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^(.)\1+$|^(\w+)(\s+\2)+\s*$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^(#\w+\s*)+$')]
    df = df.reset_index(drop=True)
    print(f"   {before:,} → {len(df):,} rows after cleaning")
    if SAMPLE_SIZE:
        df = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
        print(f"   Sampled: {len(df):,} rows")
    return df

# ── Step 2: Feature Engineering ──────────────────────────────
def extract_features(texts, tfidf=None, fit=False):
    hate_patterns = {
        cat: re.compile(r'\b(?:' + '|'.join(re.escape(k) for k in kws) + r')\b', re.IGNORECASE)
        for cat, kws in HATE_KEYWORDS.items()
    }
    pos_compiled = {pos: re.compile(pat, re.IGNORECASE) for pos, pat in POS_PATTERNS.items()}

    hate_rows, pos_rows, wf_rows = [], [], []
    for text in texts:
        t     = str(text)
        words = t.split()
        wc    = max(len(words), 1)
        lower_words = [w.lower().strip('.,!?;:\'"') for w in words]

        hr = {}
        total_hate = 0
        for cat, pat in hate_patterns.items():
            c = len(pat.findall(t))
            hr[f'hate_count_{cat}'] = c
            hr[f'hate_flag_{cat}']  = int(c > 0)
            total_hate += c
        hr['hate_total']   = total_hate
        hr['hate_density'] = total_hate / wc
        hate_rows.append(hr)

        pr = {}
        for pos, pat in pos_compiled.items():
            c = len(pat.findall(t))
            pr[f'pos_count_{pos.lower()}'] = c
            pr[f'pos_ratio_{pos.lower()}'] = c / wc
        pos_rows.append(pr)

        prof = sum(1 for w in lower_words if w in PROFANITY)
        wf_rows.append({
            'wf_word_count':        wc,
            'wf_char_count':        len(t),
            'wf_avg_word_len':      np.mean([len(w) for w in words]) if words else 0,
            'wf_caps_ratio':        len(re.findall(r'\b[A-Z]{2,}\b', t)) / wc,
            'wf_exclaim_count':     t.count('!'),
            'wf_question_count':    t.count('?'),
            'wf_mention_count':     len(re.findall(r'@\w+', t)),
            'wf_hashtag_count':     len(re.findall(r'#\w+', t)),
            'wf_profanity_count':   prof,
            'wf_profanity_density': prof / wc,
            'wf_unique_ratio':      len(set(lower_words)) / wc,
            'wf_repeat_chars':      len(re.findall(r'(.)\1{2,}', t)),
        })

    dense = np.hstack([
        pd.DataFrame(hate_rows).values,
        pd.DataFrame(pos_rows).values,
        pd.DataFrame(wf_rows).values,
    ]).astype(np.float32)
    dense = np.clip(dense, 0, None)

    if fit:
        tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2), min_df=3, sublinear_tf=True)
        tfidf_mat = tfidf.fit_transform(texts.astype(str))
    else:
        tfidf_mat = tfidf.transform(texts.astype(str))

    return hstack([csr_matrix(dense), tfidf_mat], format='csr'), tfidf

# ── Step 3: Evaluate ─────────────────────────────────────────
def evaluate(y_true, y_pred):
    print("\n" + "="*55)
    print("  📊 Logistic Regression — Evaluation Report")
    print("="*55)
    print(f"  Hamming Loss    : {hamming_loss(y_true, y_pred):.4f}  (lower=better)")
    print(f"  F1 Micro        : {f1_score(y_true, y_pred, average='micro',   zero_division=0):.4f}")
    print(f"  F1 Macro        : {f1_score(y_true, y_pred, average='macro',   zero_division=0):.4f}")
    print(f"  F1 Samples      : {f1_score(y_true, y_pred, average='samples', zero_division=0):.4f}")
    print(f"  Precision(micro): {precision_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
    print(f"  Recall(micro)   : {recall_score(y_true,    y_pred, average='micro', zero_division=0):.4f}")
    print(f"\n  {'Label':<30} {'F1':>8} {'Support':>10}")
    print(f"  {'-'*30} {'-'*8} {'-'*10}")
    for i, label in enumerate(LABEL_COLS):
        f = f1_score(y_true[:,i], y_pred[:,i], zero_division=0)
        s = int(y_true[:,i].sum())
        print(f"  {label:<30} {f:>8.4f} {s:>10,}")
    print("="*55)

# ── Main ──────────────────────────────────────────────────────
def main():
    print("="*55)
    print("  MODEL: Logistic Regression")
    print("="*55)

    df = load_and_clean()
    df_train, df_test = train_test_split(df, test_size=TEST_SIZE, random_state=42)
    df_train = df_train.reset_index(drop=True)
    df_test  = df_test.reset_index(drop=True)
    print(f"   Train: {len(df_train):,}  |  Test: {len(df_test):,}")

    print("\n🔧 Extracting features...")
    X_train, tfidf = extract_features(df_train['text'], fit=True)
    X_test,  _     = extract_features(df_test['text'],  tfidf=tfidf, fit=False)
    print(f"   Feature matrix: {X_train.shape}")

    y_train = df_train[LABEL_COLS].values
    y_test  = df_test[LABEL_COLS].values

    print("\n🚀 Training Logistic Regression...")
    t0    = time.time()
    model = OneVsRestClassifier(
        LogisticRegression(C=1.0, solver='saga', max_iter=2000, class_weight='balanced', random_state=42),
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    print(f"   Done in {time.time()-t0:.1f}s")

    # Threshold-based prediction using probabilities
    proba  = model.predict_proba(X_test)
    y_pred = (proba >= THRESHOLD).astype(int)

    evaluate(y_test, y_pred)
    return model

if __name__ == '__main__':
    main()

In [ ]:
"""
model_random_forest.py  —  Fully Self-Contained
=================================================
Random Forest with ALL feature groups — NO external files needed.
  HateKeyword(19) + POS(14) + WordFreq(21) + Sarcasm(18) + TF-IDF(20k)
  + TruncatedSVD for memory efficiency

Just upload this ONE file + your dataset to Colab and run!
"""

import re
import time
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, hamming_loss, f1_score, precision_score, recall_score

# ── Config ────────────────────────────────────────────────────
DATA_PATH      = 'cyberbullying.csv'
SAMPLE_SIZE    = 50000
TEST_SIZE      = 0.2
THRESHOLD      = 0.55
SVD_COMPONENTS = 100
LABEL_COLS     = [
    'religious_hate','ethnic_hate','age_discrimination','gender_hate',
    'sexual_harassment','threats','body_shaming','political_hate',
    'trolling','mental_hate','discrimination','other_cyberbullying_types',
    'not_cyberbullying',
]

HATE_KEYWORDS = {
    'religious_hate':    ['infidel','kafir','blasphemy','blasphemer','heretic','cult','crusade','jihad','pagan','apostate','heathen','godless','anti-christian','anti-islam','anti-hindu','anti-semitic','zionist','islamophobe','christophobe'],
    'ethnic_hate':       ['nigger','nigga','chink','spic','wetback','gook','kike','raghead','cracker','redneck','monkey','ape','beaner','immigrant','illegal alien','foreigner','outsider','invader','terrorist'],
    'gender_hate':       ['bitch','whore','slut','cunt','feminazi','tranny','dyke','faggot','sissy','mangina','thot','hoe','skank','femoid','incel','simp','soyboy','misogynist','misandrist'],
    'sexual_harassment': ['rape','rapist','molest','grope','pervert','predator','creep','harass','assault'],
    'threats':           ['kill','murder','shoot','stab','bomb','attack','destroy','hurt','harm','threat','die','execute','eliminate','exterminate','torture','burn','acid','explode','lynch','hang','beat','bash','smash'],
    'body_shaming':      ['fat','ugly','obese','disgusting','gross','pig','whale','skinny','anorexic','hideous','repulsive','deformed','freak','monster','troll','goblin'],
    'mental_hate':       ['retard','retarded','psycho','crazy','insane','lunatic','mental','schizo','bipolar','autistic','mentally ill','nutcase','bonkers','deranged','demented'],
    'general_hate':      ['hate','stupid','idiot','moron','dumb','loser','trash','garbage','scum','filth','worthless','pathetic','useless','despise','abhor','vile','toxic','cancer','parasite','vermin','subhuman'],
}
POS_PATTERNS = {
    'NOUN': r'\b(?:man|woman|people|person|thing|way|day|time|year|world|life|child|children|hand|part|place|case|week|company|system|question|government|number|night|point|home|water|room|mother|area|money|story|fact|month|lot|right|study|book|eye|job|word|business|issue|side|kind|head|house|service|friend|father|power|hour|game|line|end|member|city|community|name|president|team|minute|idea|body|information|back|parent|face|level|office|door|health|art|war|history|party|result|change|morning|reason|research|girl|guy|moment|air|teacher|force|education)\b',
    'VERB': r'\b(?:is|are|was|were|be|been|being|have|has|had|do|does|did|will|would|could|should|may|might|shall|must|can|go|get|make|know|think|take|see|come|want|look|use|find|give|tell|work|call|try|ask|need|feel|become|leave|put|mean|keep|let|begin|show|hear|play|run|move|live|believe|hold|bring|happen|write|provide|sit|stand|lose|pay|meet|include|continue|set|learn|change|lead|understand|watch|follow|stop|create|speak|read|spend|grow|open|walk|win|offer|remember|love|consider|appear|buy|wait|serve|die|send|expect|build|stay|fall|cut|reach|kill|remain|suggest|raise|pass|sell|require|report|decide|pull)\b',
    'ADJ':  r'\b(?:good|new|first|last|long|great|little|own|other|old|right|big|high|different|small|large|next|early|young|important|public|private|real|best|free|few|same|able|political|social|economic|national|possible|local|white|black|strong|true|hot|happy|sad|bad|ugly|fat|stupid|evil|dangerous|terrible|horrible|awful|disgusting|pathetic|worthless|useless|dumb|crazy|insane|violent|racist|sexist|offensive|hateful|toxic|radical|extreme)\b',
    'ADV':  r'\b(?:up|so|out|just|now|how|then|more|also|here|well|only|very|even|back|there|down|still|in|as|too|really|most|never|much|often|always|actually|again|further|yet|already|soon|especially|finally|simply|probably|certainly|clearly|literally|absolutely|totally|completely|definitely|obviously|seriously|exactly|nearly|directly|quickly|easily)\b',
    'PRON': r'\b(?:i|me|my|myself|you|your|yourself|he|him|his|himself|she|her|hers|herself|it|its|itself|we|us|our|ourselves|they|them|their|theirs|themselves|this|that|these|those|who|whom|which|what)\b',
}
PROFANITY      = {'fuck','fucking','fucked','shit','ass','asshole','bitch','cunt','bastard','damn','piss','cock','dick','pussy','motherfucker','whore','slut','idiot','moron','loser','stupid','dumb','retard'}
SARC_MARKERS   = [r'\boh great\b',r'\byeah right\b',r'\bsure sure\b',r'\bthanks a lot\b',r'\bthanks so much\b',r'\bso helpful\b',r'\breally helpful\b',r'\bso smart\b',r'\bwow thanks\b',r'\boh wow\b',r'\boh really\b',r'\bno kidding\b',r'\bno way\b',r'\bright sure\b',r'\bas if\b',r'\bcongrats\b',r'\bcongratulations\b',r'\bwhat a surprise\b',r'\bshocking\b',r'\bunbelievable\b',r'\bi\'m sure\b',r'\bim sure\b',r'\bi bet\b',r'\bof course\b',r'\bobviously\b',r'\bclearly\b',r'\bjust great\b',r'\bjust perfect\b',r'\bso funny\b',r'\bhilarious\b']
SARC_POS_WORDS = {'great','wonderful','amazing','fantastic','brilliant','excellent','perfect','awesome','lovely','nice','good','fine','super','best','love','happy','glad','thrilled','helpful','smart','clever','impressive','incredible'}
SARC_NEG_WORDS = {'hate','terrible','awful','horrible','disgusting','pathetic','stupid','idiot','moron','useless','worthless','trash','garbage','fail','worst','bad','ugly','dumb','loser'}
SARC_CONTRA    = [(r'\b(?:great|wonderful|amazing|fantastic)\b',r'\b(?:not|never|no|hate|terrible)\b'),(r'\b(?:love|like|enjoy)\b',r'\b(?:not|never|no|hate|dislike)\b'),(r'\b(?:good|nice|fine)\b',r'\b(?:not|terrible|awful|horrible)\b')]
_INTENSIFIER_RE= re.compile(r'\b(?:so|very|extremely|incredibly|absolutely|totally|completely|literally|seriously|really|such|beyond|super)\b', re.IGNORECASE)
_POS_EMOJI_RE  = re.compile(r'[\U0001F600-\U0001F606\U0001F609\U0001F60A\U0001F60D\U0001F618\U0001F61C\U0001F61D\U0001F923]')


def load_and_clean():
    print("📂 Loading data...")
    df=pd.read_csv(DATA_PATH); before=len(df)
    df=df[df['text'].notnull()&(df['text'].str.strip()!='')]
    df=df[df['text'].astype(str).str.contains(r'[a-zA-Z]',regex=True)]
    df=df[~df['text'].fillna('').str.strip().str.match(r'^@\w+$')]
    df=df[~df['text'].fillna('').str.strip().str.match(r'^\d+$')]
    df=df[~df['text'].fillna('').str.strip().str.match(r'^[^\w\s]+$')]
    df=df[~df['text'].fillna('').str.strip().str.match(r'^(.)\1+$|^(\w+)(\s+\2)+\s*$')]
    df=df[~df['text'].fillna('').str.strip().str.match(r'^(#\w+\s*)+$')]
    df=df.reset_index(drop=True); print(f"   {before:,} → {len(df):,} rows after cleaning")
    if SAMPLE_SIZE:
        df=df.sample(min(SAMPLE_SIZE,len(df)),random_state=42).reset_index(drop=True)
        print(f"   Sampled: {len(df):,} rows")
    return df


def extract_features(texts, tfidf=None, fit=False):
    hate_pats={cat:re.compile(r'\b(?:'+' |'.join(re.escape(k) for k in kws)+r')\b',re.IGNORECASE) for cat,kws in HATE_KEYWORDS.items()}
    hate_pats={cat:re.compile(r'\b(?:'+('|'.join(re.escape(k) for k in kws))+r')\b',re.IGNORECASE) for cat,kws in HATE_KEYWORDS.items()}
    pos_pats={pos:re.compile(pat,re.IGNORECASE) for pos,pat in POS_PATTERNS.items()}
    marker_pats=[re.compile(p,re.IGNORECASE) for p in SARC_MARKERS]
    contra_pats=[(re.compile(a,re.IGNORECASE),re.compile(b,re.IGNORECASE)) for a,b in SARC_CONTRA]
    url_re=re.compile(r'https?://\S+|www\.\S+'); mention_re=re.compile(r'@\w+')
    hashtag_re=re.compile(r'#\w+'); emoji_re=re.compile(r'[\U0001F300-\U0001F9FF\U00002600-\U000027FF]',flags=re.UNICODE)
    repeat_re=re.compile(r'(.)\1{2,}'); caps_re=re.compile(r'\b[A-Z]{2,}\b')
    all_rows=[]
    for text in texts:
        t=str(text); words=t.split(); wc=max(len(words),1); cc=max(len(t),1)
        lw=[w.lower().strip('.,!?;:\'"') for w in words]; sents=max(len(re.split(r'[.!?]+',t)),1)
        wfreq={}
        for w in lw: wfreq[w]=wfreq.get(w,0)+1
        caps=caps_re.findall(t); prof=sum(1 for w in lw if w in PROFANITY); row={}
        total_hate=0
        for cat,pat in hate_pats.items():
            c=len(pat.findall(t)); row[f'hate_count_{cat}']=c; row[f'hate_flag_{cat}']=int(c>0); total_hate+=c
        row['hate_total_count']=total_hate; row['hate_density']=total_hate/wc
        row['hate_category_count']=sum(1 for cat in HATE_KEYWORDS if row[f'hate_flag_{cat}']>0)
        pcounts={}
        for pos,pat in pos_pats.items():
            c=len(pat.findall(t)); pcounts[pos]=c; row[f'pos_count_{pos.lower()}']=c; row[f'pos_ratio_{pos.lower()}']=c/wc
        row['pos_adj_noun_ratio']=pcounts.get('ADJ',0)/max(pcounts.get('NOUN',0),1)
        row['pos_verb_density']=pcounts.get('VERB',0)/wc; row['pos_pron_density']=pcounts.get('PRON',0)/wc
        row['pos_modifier_density']=(pcounts.get('ADJ',0)+pcounts.get('ADV',0))/wc
        row.update({'wf_char_count':cc,'wf_word_count':wc,'wf_sentence_count':sents,
            'wf_avg_word_len':np.mean([len(w) for w in words]) if words else 0,'wf_avg_sent_len':wc/sents,
            'wf_unique_word_ratio':len(set(lw))/wc,'wf_caps_word_count':len(caps),'wf_caps_ratio':len(caps)/wc,
            'wf_upper_char_ratio':sum(1 for c in t if c.isupper())/cc,'wf_exclaim_count':t.count('!'),
            'wf_question_count':t.count('?'),'wf_ellipsis_count':t.count('...'),
            'wf_punct_density':sum(1 for c in t if c in '!?.,;:\'"')/cc,
            'wf_url_count':len(url_re.findall(t)),'wf_mention_count':len(mention_re.findall(t)),
            'wf_hashtag_count':len(hashtag_re.findall(t)),'wf_emoji_count':len(emoji_re.findall(t)),
            'wf_repeat_char_count':len(repeat_re.findall(t)),'wf_repeat_word_count':sum(1 for v in wfreq.values() if v>1),
            'wf_profanity_count':prof,'wf_profanity_density':prof/wc})
        multi_exc=len(re.findall(r'!{2,}',t)); mc=sum(1 for p in marker_pats if p.search(t))
        pc=sum(1 for w in lw if w in SARC_POS_WORDS); nc=sum(1 for w in lw if w in SARC_NEG_WORDS)
        cc2=sum(1 for a_p,b_p in contra_pats if a_p.search(t) and b_p.search(t))
        id_=len(_INTENSIFIER_RE.findall(t))/wc; ec=int(len(_POS_EMOJI_RE.findall(t))>0 and nc>0)
        ss=min((mc*2.0+multi_exc*1.5+cc2*2.0+int(pc>0 and nc>0)*1.5+(len(caps)/wc)*3.0+id_*2.0+ec*1.5)/10.0,1.0)
        row.update({'sarc_exclaim_count':t.count('!'),'sarc_multi_exclaim':multi_exc,
            'sarc_question_exclaim':len(re.findall(r'[!?]{2,}',t)),'sarc_ellipsis_count':t.count('...'),
            'sarc_caps_words':len(caps),'sarc_caps_ratio':len(caps)/wc,'sarc_marker_count':mc,'sarc_marker_density':mc/wc,
            'sarc_pos_word_count':pc,'sarc_neg_word_count':nc,'sarc_pos_neg_mix':int(pc>0 and nc>0),
            'sarc_pos_neg_ratio':pc/max(nc,1),'sarc_contradiction_count':cc2,
            'sarc_quote_count':t.count('"')+t.count("'"),'sarc_has_quotes':int(t.count('"')+t.count("'")>=2),
            'sarc_emoji_contradiction':ec,'sarc_intensifier_density':id_,'sarc_heuristic_score':ss})
        all_rows.append(row)
    dense=pd.DataFrame(all_rows).values.astype(np.float32); dense=np.clip(dense,0,None)
    if fit:
        tfidf=TfidfVectorizer(max_features=15000,ngram_range=(1,2),min_df=3,sublinear_tf=True)
        tfidf_mat=tfidf.fit_transform(texts.astype(str))
    else:
        tfidf_mat=tfidf.transform(texts.astype(str))
    return hstack([csr_matrix(dense),tfidf_mat],format='csr'),tfidf


def evaluate(y_true, y_pred, elapsed=None):
    print("\n"+"="*55); print("  📊 Random Forest — Accuracy Report")
    if elapsed: print(f"  ⏱  Training time: {elapsed:.1f}s")
    print("="*55)
    print(f"  Exact Match Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Hamming Accuracy     : {1-hamming_loss(y_true, y_pred):.4f}  ✅")
    print(f"  Hamming Loss         : {hamming_loss(y_true, y_pred):.4f}")
    print(f"  F1 Micro   (main)    : {f1_score(y_true, y_pred, average='micro',    zero_division=0):.4f}  ✅")
    print(f"  F1 Macro             : {f1_score(y_true, y_pred, average='macro',    zero_division=0):.4f}")
    print(f"  F1 Weighted          : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"  F1 Samples           : {f1_score(y_true, y_pred, average='samples',  zero_division=0):.4f}")
    print(f"  Precision (micro)    : {precision_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
    print(f"  Recall    (micro)    : {recall_score(y_true,    y_pred, average='micro', zero_division=0):.4f}")
    print(f"\n  {'Label':<30} {'F1':>8} {'Support':>10}"); print(f"  {'-'*30} {'-'*8} {'-'*10}")
    f1s=[]
    for i,label in enumerate(LABEL_COLS):
        f=f1_score(y_true[:,i],y_pred[:,i],zero_division=0); s=int(y_true[:,i].sum())
        bar='█'*int(f*10)+'░'*(10-int(f*10)); print(f"  {label:<30} {f:>8.4f} {s:>10,}  {bar}"); f1s.append((label,f))
    best=max(f1s,key=lambda x:x[1]); worst=min(f1s,key=lambda x:x[1])
    print(f"\n  🏆 Best  : {best[0]:<30}  F1={best[1]:.4f}")
    print(f"  ⚠️  Worst : {worst[0]:<30}  F1={worst[1]:.4f}"); print("="*55)


def detect(text, model, tfidf, svd, threshold=THRESHOLD):
    X=extract_features(pd.Series([text]),tfidf=tfidf,fit=False)[0]
    X_rf=svd.transform(X)
    proba=model.predict_proba(X_rf)[0]; pred=(proba>=threshold).astype(int)
    detected=[LABEL_COLS[i] for i,p in enumerate(pred) if p==1 and LABEL_COLS[i]!='not_cyberbullying']
    print("\n"+"="*52); print(f"  📝 Text      : {text[:65]}{'...' if len(text)>65 else ''}")
    print(f"  🎚  Threshold : {threshold}"); print("="*52)
    if not detected:
        print("  ✅  NOT CYBERBULLYING")
    else:
        print("  🚨  CYBERBULLYING DETECTED"); print("  Categories found:")
        for label in detected:
            conf=proba[LABEL_COLS.index(label)]; bar='█'*int(conf*10)+'░'*(10-int(conf*10))
            print(f"    ⚠️  {label:<30} {conf:.0%}  {bar}")
    print("="*52); return detected


def run_detector(model, tfidf, svd):
    print("\n"+"="*52); print("  🤖 Cyberbullying Detector (Random Forest)")
    print(f"  Threshold: {THRESHOLD}  |  Type 'exit' to stop"); print("="*52+"\n")
    while True:
        text=input("Enter text: ").strip()
        if text.lower()=='exit': print("👋 Stopping..."); break
        if not text: print("⚠️  Please enter some text\n"); continue
        detect(text, model, tfidf, svd, threshold=THRESHOLD)


def main():
    print("="*55); print("  MODEL: Random Forest — Self Contained"); print("="*55)
    df=load_and_clean()
    df_train,df_test=train_test_split(df,test_size=TEST_SIZE,random_state=42)
    df_train=df_train.reset_index(drop=True); df_test=df_test.reset_index(drop=True)
    print(f"   Train: {len(df_train):,}  |  Test: {len(df_test):,}")
    print("\n🔧 Extracting ALL features (HateKeyword+POS+WordFreq+Sarcasm+TFIDF)...")
    X_train,tfidf=extract_features(df_train['text'],fit=True)
    X_test,_=extract_features(df_test['text'],tfidf=tfidf,fit=False)
    print(f"   ✅ Features: {X_train.shape[1]:,}")
    print(f"   Applying SVD ({SVD_COMPONENTS} components)...")
    svd=TruncatedSVD(n_components=SVD_COMPONENTS,random_state=42)
    X_train_rf=svd.fit_transform(X_train); X_test_rf=svd.transform(X_test)
    print(f"   Explained variance: {svd.explained_variance_ratio_.sum():.3f}")
    y_train=df_train[LABEL_COLS].values; y_test=df_test[LABEL_COLS].values
    print("\n🚀 Training Random Forest...")
    t0=time.time()
    model=OneVsRestClassifier(RandomForestClassifier(n_estimators=200,max_depth=30,min_samples_split=5,min_samples_leaf=2,max_features='sqrt',class_weight='balanced',n_jobs=-1,random_state=42),n_jobs=-1)
    model.fit(X_train_rf,y_train); elapsed=time.time()-t0; print(f"   Done in {elapsed:.1f}s")
    proba=model.predict_proba(X_test_rf); y_pred=(proba>=THRESHOLD).astype(int)
    evaluate(y_test,y_pred,elapsed)
    run_detector(model,tfidf,svd)
    return model,tfidf,svd

if __name__=='__main__':
    main()

  MODEL: Random Forest — Self Contained
📂 Loading data...
   684,383 → 684,172 rows after cleaning
   Sampled: 50,000 rows
   Train: 40,000  |  Test: 10,000

🔧 Extracting ALL features (HateKeyword+POS+WordFreq+Sarcasm+TFIDF)...
   ✅ Features: 15,072
   Applying SVD (100 components)...
   Explained variance: 1.000

🚀 Training Random Forest...
   Done in 1202.2s

  📊 Random Forest — Accuracy Report
  ⏱  Training time: 1202.2s
  Exact Match Accuracy : 0.2006
  Hamming Accuracy     : 0.9188  ✅
  Hamming Loss         : 0.0812
  F1 Micro   (main)    : 0.3126  ✅
  F1 Macro             : 0.2625
  F1 Weighted          : 0.2771
  F1 Samples           : 0.2261
  Precision (micro)    : 0.7395
  Recall    (micro)    : 0.1982

  Label                                F1    Support
  ------------------------------ -------- ----------
  religious_hate                   0.4328        580  ████░░░░░░
  ethnic_hate                      0.1862        826  █░░░░░░░░░
  age_discrimination               0.4965

In [ ]:
"""
model_decision_tree.py  —  Fully Self-Contained
=================================================
Decision Tree with ALL feature groups — NO external files needed.
  HateKeyword(19) + POS(14) + WordFreq(21) + Sarcasm(18) + TF-IDF(20k)

Just upload this ONE file + your dataset to Colab and run!
"""

import re
import time
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, hamming_loss, f1_score, precision_score, recall_score

# ── Config ────────────────────────────────────────────────────
DATA_PATH      = 'cyberbullying.csv'
SAMPLE_SIZE    = 100000  # Decision Tree is fast but memory heavy on full data
TEST_SIZE      = 0.2
THRESHOLD      = 0.55
SVD_COMPONENTS = 100     # reduce TF-IDF dims for Decision Tree
LABEL_COLS     = [
    'religious_hate','ethnic_hate','age_discrimination','gender_hate',
    'sexual_harassment','threats','body_shaming','political_hate',
    'trolling','mental_hate','discrimination','other_cyberbullying_types',
    'not_cyberbullying',
]

# ── Feature Definitions ───────────────────────────────────────
HATE_KEYWORDS = {
    'religious_hate':    ['infidel','kafir','blasphemy','blasphemer','heretic','cult','crusade','jihad','pagan','apostate','heathen','godless','anti-christian','anti-islam','anti-hindu','anti-semitic','zionist','islamophobe','christophobe'],
    'ethnic_hate':       ['nigger','nigga','chink','spic','wetback','gook','kike','raghead','cracker','redneck','monkey','ape','beaner','immigrant','illegal alien','foreigner','outsider','invader','terrorist'],
    'gender_hate':       ['bitch','whore','slut','cunt','feminazi','tranny','dyke','faggot','sissy','mangina','thot','hoe','skank','femoid','incel','simp','soyboy','misogynist','misandrist'],
    'sexual_harassment': ['rape','rapist','molest','grope','pervert','predator','creep','harass','assault'],
    'threats':           ['kill','murder','shoot','stab','bomb','attack','destroy','hurt','harm','threat','die','execute','eliminate','exterminate','torture','burn','acid','explode','lynch','hang','beat','bash','smash'],
    'body_shaming':      ['fat','ugly','obese','disgusting','gross','pig','whale','skinny','anorexic','hideous','repulsive','deformed','freak','monster','troll','goblin'],
    'mental_hate':       ['retard','retarded','psycho','crazy','insane','lunatic','mental','schizo','bipolar','autistic','mentally ill','nutcase','bonkers','deranged','demented'],
    'general_hate':      ['hate','stupid','idiot','moron','dumb','loser','trash','garbage','scum','filth','worthless','pathetic','useless','despise','abhor','vile','toxic','cancer','parasite','vermin','subhuman'],
}
POS_PATTERNS = {
    'NOUN': r'\b(?:man|woman|people|person|thing|way|day|time|year|world|life|child|children|hand|part|place|case|week|company|system|question|government|number|night|point|home|water|room|mother|area|money|story|fact|month|lot|right|study|book|eye|job|word|business|issue|side|kind|head|house|service|friend|father|power|hour|game|line|end|member|city|community|name|president|team|minute|idea|body|information|back|parent|face|level|office|door|health|art|war|history|party|result|change|morning|reason|research|girl|guy|moment|air|teacher|force|education)\b',
    'VERB': r'\b(?:is|are|was|were|be|been|being|have|has|had|do|does|did|will|would|could|should|may|might|shall|must|can|go|get|make|know|think|take|see|come|want|look|use|find|give|tell|work|call|try|ask|need|feel|become|leave|put|mean|keep|let|begin|show|hear|play|run|move|live|believe|hold|bring|happen|write|provide|sit|stand|lose|pay|meet|include|continue|set|learn|change|lead|understand|watch|follow|stop|create|speak|read|spend|grow|open|walk|win|offer|remember|love|consider|appear|buy|wait|serve|die|send|expect|build|stay|fall|cut|reach|kill|remain|suggest|raise|pass|sell|require|report|decide|pull)\b',
    'ADJ':  r'\b(?:good|new|first|last|long|great|little|own|other|old|right|big|high|different|small|large|next|early|young|important|public|private|real|best|free|few|same|able|political|social|economic|national|possible|local|white|black|strong|true|hot|happy|sad|bad|ugly|fat|stupid|evil|dangerous|terrible|horrible|awful|disgusting|pathetic|worthless|useless|dumb|crazy|insane|violent|racist|sexist|offensive|hateful|toxic|radical|extreme)\b',
    'ADV':  r'\b(?:up|so|out|just|now|how|then|more|also|here|well|only|very|even|back|there|down|still|in|as|too|really|most|never|much|often|always|actually|again|further|yet|already|soon|especially|finally|simply|probably|certainly|clearly|literally|absolutely|totally|completely|definitely|obviously|seriously|exactly|nearly|directly|quickly|easily)\b',
    'PRON': r'\b(?:i|me|my|myself|you|your|yourself|he|him|his|himself|she|her|hers|herself|it|its|itself|we|us|our|ourselves|they|them|their|theirs|themselves|this|that|these|those|who|whom|which|what)\b',
}
PROFANITY      = {'fuck','fucking','fucked','shit','ass','asshole','bitch','cunt','bastard','damn','piss','cock','dick','pussy','motherfucker','whore','slut','idiot','moron','loser','stupid','dumb','retard'}
SARC_MARKERS   = [r'\boh great\b',r'\byeah right\b',r'\bsure sure\b',r'\bthanks a lot\b',r'\bthanks so much\b',r'\bso helpful\b',r'\breally helpful\b',r'\bso smart\b',r'\bwow thanks\b',r'\boh wow\b',r'\boh really\b',r'\bno kidding\b',r'\bno way\b',r'\bright sure\b',r'\bas if\b',r'\bcongrats\b',r'\bcongratulations\b',r'\bwhat a surprise\b',r'\bshocking\b',r'\bunbelievable\b',r'\bi\'m sure\b',r'\bim sure\b',r'\bi bet\b',r'\bof course\b',r'\bobviously\b',r'\bclearly\b',r'\bjust great\b',r'\bjust perfect\b',r'\bso funny\b',r'\bhilarious\b']
SARC_POS_WORDS = {'great','wonderful','amazing','fantastic','brilliant','excellent','perfect','awesome','lovely','nice','good','fine','super','best','love','happy','glad','thrilled','helpful','smart','clever','impressive','incredible'}
SARC_NEG_WORDS = {'hate','terrible','awful','horrible','disgusting','pathetic','stupid','idiot','moron','useless','worthless','trash','garbage','fail','worst','bad','ugly','dumb','loser'}
SARC_CONTRA    = [(r'\b(?:great|wonderful|amazing|fantastic)\b',r'\b(?:not|never|no|hate|terrible)\b'),(r'\b(?:love|like|enjoy)\b',r'\b(?:not|never|no|hate|dislike)\b'),(r'\b(?:good|nice|fine)\b',r'\b(?:not|terrible|awful|horrible)\b')]
_INTENSIFIER_RE= re.compile(r'\b(?:so|very|extremely|incredibly|absolutely|totally|completely|literally|seriously|really|such|beyond|super)\b', re.IGNORECASE)
_POS_EMOJI_RE  = re.compile(r'[\U0001F600-\U0001F606\U0001F609\U0001F60A\U0001F60D\U0001F618\U0001F61C\U0001F61D\U0001F923]')


# ── Step 1: Load & Clean ──────────────────────────────────────
def load_and_clean():
    print("📂 Loading data...")
    df = pd.read_csv(DATA_PATH)
    before = len(df)
    df = df[df['text'].notnull() & (df['text'].str.strip() != '')]
    df = df[df['text'].astype(str).str.contains(r'[a-zA-Z]', regex=True)]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^@\w+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^\d+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^[^\w\s]+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^(.)\1+$|^(\w+)(\s+\2)+\s*$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^(#\w+\s*)+$')]
    df = df.reset_index(drop=True)
    print(f"   {before:,} → {len(df):,} rows after cleaning")
    if SAMPLE_SIZE:
        df = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
        print(f"   Sampled: {len(df):,} rows")
    return df


# ── Step 2: Feature Engineering ──────────────────────────────
def extract_features(texts, tfidf=None, fit=False):
    hate_pats   = {cat: re.compile(r'\b(?:' + '|'.join(re.escape(k) for k in kws) + r')\b', re.IGNORECASE) for cat, kws in HATE_KEYWORDS.items()}
    pos_pats    = {pos: re.compile(pat, re.IGNORECASE) for pos, pat in POS_PATTERNS.items()}
    marker_pats = [re.compile(p, re.IGNORECASE) for p in SARC_MARKERS]
    contra_pats = [(re.compile(a, re.IGNORECASE), re.compile(b, re.IGNORECASE)) for a, b in SARC_CONTRA]
    url_re      = re.compile(r'https?://\S+|www\.\S+')
    mention_re  = re.compile(r'@\w+')
    hashtag_re  = re.compile(r'#\w+')
    emoji_re    = re.compile(r'[\U0001F300-\U0001F9FF\U00002600-\U000027FF]', flags=re.UNICODE)
    repeat_re   = re.compile(r'(.)\1{2,}')
    caps_re     = re.compile(r'\b[A-Z]{2,}\b')

    all_rows = []
    for text in texts:
        t  = str(text); words = t.split()
        wc = max(len(words), 1); cc = max(len(t), 1)
        lw = [w.lower().strip('.,!?;:\'"') for w in words]
        sents = max(len(re.split(r'[.!?]+', t)), 1)
        wfreq = {}
        for w in lw: wfreq[w] = wfreq.get(w, 0) + 1
        caps = caps_re.findall(t)
        prof = sum(1 for w in lw if w in PROFANITY)
        row  = {}

        # HateKeyword (19)
        total_hate = 0
        for cat, pat in hate_pats.items():
            c = len(pat.findall(t))
            row[f'hate_count_{cat}'] = c; row[f'hate_flag_{cat}'] = int(c > 0); total_hate += c
        row['hate_total_count'] = total_hate; row['hate_density'] = total_hate / wc
        row['hate_category_count'] = sum(1 for cat in HATE_KEYWORDS if row[f'hate_flag_{cat}'] > 0)

        # POS (14)
        pcounts = {}
        for pos, pat in pos_pats.items():
            c = len(pat.findall(t)); pcounts[pos] = c
            row[f'pos_count_{pos.lower()}'] = c; row[f'pos_ratio_{pos.lower()}'] = c / wc
        row['pos_adj_noun_ratio']   = pcounts.get('ADJ', 0) / max(pcounts.get('NOUN', 0), 1)
        row['pos_verb_density']     = pcounts.get('VERB', 0) / wc
        row['pos_pron_density']     = pcounts.get('PRON', 0) / wc
        row['pos_modifier_density'] = (pcounts.get('ADJ', 0) + pcounts.get('ADV', 0)) / wc

        # WordFrequency (21)
        row.update({'wf_char_count':cc,'wf_word_count':wc,'wf_sentence_count':sents,
            'wf_avg_word_len':np.mean([len(w) for w in words]) if words else 0,
            'wf_avg_sent_len':wc/sents,'wf_unique_word_ratio':len(set(lw))/wc,
            'wf_caps_word_count':len(caps),'wf_caps_ratio':len(caps)/wc,
            'wf_upper_char_ratio':sum(1 for c in t if c.isupper())/cc,
            'wf_exclaim_count':t.count('!'),'wf_question_count':t.count('?'),
            'wf_ellipsis_count':t.count('...'),
            'wf_punct_density':sum(1 for c in t if c in '!?.,;:\'"')/cc,
            'wf_url_count':len(url_re.findall(t)),'wf_mention_count':len(mention_re.findall(t)),
            'wf_hashtag_count':len(hashtag_re.findall(t)),'wf_emoji_count':len(emoji_re.findall(t)),
            'wf_repeat_char_count':len(repeat_re.findall(t)),
            'wf_repeat_word_count':sum(1 for v in wfreq.values() if v > 1),
            'wf_profanity_count':prof,'wf_profanity_density':prof/wc})

        # Sarcasm (18)
        multi_exc = len(re.findall(r'!{2,}', t))
        mc  = sum(1 for p in marker_pats if p.search(t))
        pc  = sum(1 for w in lw if w in SARC_POS_WORDS)
        nc  = sum(1 for w in lw if w in SARC_NEG_WORDS)
        cc2 = sum(1 for a_p, b_p in contra_pats if a_p.search(t) and b_p.search(t))
        id_ = len(_INTENSIFIER_RE.findall(t)) / wc
        ec  = int(len(_POS_EMOJI_RE.findall(t)) > 0 and nc > 0)
        ss  = min((mc*2.0+multi_exc*1.5+cc2*2.0+int(pc>0 and nc>0)*1.5+(len(caps)/wc)*3.0+id_*2.0+ec*1.5)/10.0, 1.0)
        row.update({'sarc_exclaim_count':t.count('!'),'sarc_multi_exclaim':multi_exc,
            'sarc_question_exclaim':len(re.findall(r'[!?]{2,}',t)),'sarc_ellipsis_count':t.count('...'),
            'sarc_caps_words':len(caps),'sarc_caps_ratio':len(caps)/wc,
            'sarc_marker_count':mc,'sarc_marker_density':mc/wc,
            'sarc_pos_word_count':pc,'sarc_neg_word_count':nc,
            'sarc_pos_neg_mix':int(pc>0 and nc>0),'sarc_pos_neg_ratio':pc/max(nc,1),
            'sarc_contradiction_count':cc2,'sarc_quote_count':t.count('"')+t.count("'"),
            'sarc_has_quotes':int(t.count('"')+t.count("'")>=2),
            'sarc_emoji_contradiction':ec,'sarc_intensifier_density':id_,'sarc_heuristic_score':ss})
        all_rows.append(row)

    dense = pd.DataFrame(all_rows).values.astype(np.float32)
    dense = np.clip(dense, 0, None)
    if fit:
        tfidf     = TfidfVectorizer(max_features=15000, ngram_range=(1,2), min_df=3, sublinear_tf=True)
        tfidf_mat = tfidf.fit_transform(texts.astype(str))
    else:
        tfidf_mat = tfidf.transform(texts.astype(str))
    return hstack([csr_matrix(dense), tfidf_mat], format='csr'), tfidf


# ── Step 3: Evaluate ──────────────────────────────────────────
def evaluate(y_true, y_pred, elapsed=None):
    print("\n" + "="*55)
    print("  📊 Decision Tree — Accuracy Report")
    if elapsed: print(f"  ⏱  Training time: {elapsed:.1f}s")
    print("="*55)
    print(f"  Exact Match Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Hamming Accuracy     : {1-hamming_loss(y_true, y_pred):.4f}  ✅")
    print(f"  Hamming Loss         : {hamming_loss(y_true, y_pred):.4f}")
    print(f"  F1 Micro   (main)    : {f1_score(y_true, y_pred, average='micro',    zero_division=0):.4f}  ✅")
    print(f"  F1 Macro             : {f1_score(y_true, y_pred, average='macro',    zero_division=0):.4f}")
    print(f"  F1 Weighted          : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"  F1 Samples           : {f1_score(y_true, y_pred, average='samples',  zero_division=0):.4f}")
    print(f"  Precision (micro)    : {precision_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
    print(f"  Recall    (micro)    : {recall_score(y_true,    y_pred, average='micro', zero_division=0):.4f}")
    print(f"\n  {'Label':<30} {'F1':>8} {'Support':>10}")
    print(f"  {'-'*30} {'-'*8} {'-'*10}")
    f1s = []
    for i, label in enumerate(LABEL_COLS):
        f = f1_score(y_true[:,i], y_pred[:,i], zero_division=0)
        s = int(y_true[:,i].sum())
        bar = '█'*int(f*10) + '░'*(10-int(f*10))
        print(f"  {label:<30} {f:>8.4f} {s:>10,}  {bar}")
        f1s.append((label, f))
    best  = max(f1s, key=lambda x: x[1])
    worst = min(f1s, key=lambda x: x[1])
    print(f"\n  🏆 Best  : {best[0]:<30}  F1={best[1]:.4f}")
    print(f"  ⚠️  Worst : {worst[0]:<30}  F1={worst[1]:.4f}")
    print("="*55)


# ── Step 4: Real Time Detection ───────────────────────────────
def detect(text, model, tfidf, svd, threshold=THRESHOLD):
    X        = extract_features(pd.Series([text]), tfidf=tfidf, fit=False)[0]
    X_dt     = svd.transform(X)
    proba    = model.predict_proba(X_dt)[0]
    pred     = (proba >= threshold).astype(int)
    detected = [LABEL_COLS[i] for i, p in enumerate(pred)
                if p == 1 and LABEL_COLS[i] != 'not_cyberbullying']

    print("\n" + "="*52)
    print(f"  📝 Text      : {text[:65]}{'...' if len(text)>65 else ''}")
    print(f"  🎚  Threshold : {threshold}")
    print("="*52)
    if not detected:
        print("  ✅  NOT CYBERBULLYING")
    else:
        print("  🚨  CYBERBULLYING DETECTED")
        print("  Categories found:")
        for label in detected:
            conf = proba[LABEL_COLS.index(label)]
            bar  = '█'*int(conf*10) + '░'*(10-int(conf*10))
            print(f"    ⚠️  {label:<30} {conf:.0%}  {bar}")
    print("="*52)
    return detected


# ── Step 5: Real Time Loop ────────────────────────────────────
def run_detector(model, tfidf, svd):
    print("\n" + "="*52)
    print("  🤖 Cyberbullying Detector (Decision Tree)")
    print(f"  Threshold: {THRESHOLD}  |  Type 'exit' to stop")
    print("="*52 + "\n")
    while True:
        text = input("Enter text: ").strip()
        if text.lower() == 'exit':
            print("👋 Stopping..."); break
        if not text:
            print("⚠️  Please enter some text\n"); continue
        detect(text, model, tfidf, svd, threshold=THRESHOLD)


# ── Main ──────────────────────────────────────────────────────
def main():
    print("="*55)
    print("  MODEL: Decision Tree — Self Contained")
    print("="*55)

    df = load_and_clean()
    df_train, df_test = train_test_split(df, test_size=TEST_SIZE, random_state=42)
    df_train = df_train.reset_index(drop=True)
    df_test  = df_test.reset_index(drop=True)
    print(f"   Train: {len(df_train):,}  |  Test: {len(df_test):,}")

    print("\n🔧 Extracting ALL features (HateKeyword+POS+WordFreq+Sarcasm+TFIDF)...")
    X_train, tfidf = extract_features(df_train['text'], fit=True)
    X_test,  _     = extract_features(df_test['text'],  tfidf=tfidf, fit=False)
    print(f"   ✅ Features: {X_train.shape[1]:,}  (including 18 sarcasm features)")

    print(f"   Applying SVD ({SVD_COMPONENTS} components)...")
    svd        = TruncatedSVD(n_components=SVD_COMPONENTS, random_state=42)
    X_train_dt = svd.fit_transform(X_train)
    X_test_dt  = svd.transform(X_test)
    print(f"   Explained variance: {svd.explained_variance_ratio_.sum():.3f}")

    y_train = df_train[LABEL_COLS].values
    y_test  = df_test[LABEL_COLS].values

    print("\n🚀 Training Decision Tree...")
    t0 = time.time()
    model = OneVsRestClassifier(
        DecisionTreeClassifier(
            max_depth        = 20,      # prevent overfitting
            min_samples_split= 10,
            min_samples_leaf = 5,
            class_weight     = 'balanced',
            criterion        = 'gini',
            random_state     = 42,
        ),
        n_jobs=-1
    )
    model.fit(X_train_dt, y_train)
    elapsed = time.time() - t0
    print(f"   Done in {elapsed:.1f}s")

    proba  = model.predict_proba(X_test_dt)
    y_pred = (proba >= THRESHOLD).astype(int)
    evaluate(y_test, y_pred, elapsed)

    run_detector(model, tfidf, svd)
    return model, tfidf, svd

if __name__ == '__main__':
    main()

  MODEL: Decision Tree — Self Contained
📂 Loading data...
   684,383 → 684,172 rows after cleaning
   Sampled: 100,000 rows
   Train: 80,000  |  Test: 20,000

🔧 Extracting ALL features (HateKeyword+POS+WordFreq+Sarcasm+TFIDF)...
   ✅ Features: 15,072  (including 18 sarcasm features)
   Applying SVD (100 components)...
   Explained variance: 1.000

🚀 Training Decision Tree...
   Done in 213.6s

  📊 Decision Tree — Accuracy Report
  ⏱  Training time: 213.6s
  Exact Match Accuracy : 0.0877
  Hamming Accuracy     : 0.7968  ✅
  Hamming Loss         : 0.2032
  F1 Micro   (main)    : 0.3203  ✅
  F1 Macro             : 0.3219
  F1 Weighted          : 0.3352
  F1 Samples           : 0.3398
  Precision (micro)    : 0.2322
  Recall    (micro)    : 0.5161

  Label                                F1    Support
  ------------------------------ -------- ----------
  religious_hate                   0.3838      1,191  ███░░░░░░░
  ethnic_hate                      0.2710      1,685  ██░░░░░░░░
  age_dis

In [ ]:
"""
model_svm.py  —  Fully Self-Contained
=======================================
SVM with ALL feature groups — NO external files needed.
  HateKeyword(19) + POS(14) + WordFreq(21) + Sarcasm score(1) + TF-IDF(20k)

Just upload this ONE file + your dataset to Colab and run!
"""

import re
import time
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, hamming_loss, f1_score,
    precision_score, recall_score
)
# ── LIME Explainability ──────────────────────────────────────
!pip install lime

from lime.lime_text import LimeTextExplainer

# ── Config ────────────────────────────────────────────────────
DATA_PATH = '/content/drive/MyDrive/Cyberbullying_Project/cyberbullying.csv'
SAMPLE_SIZE = None
TEST_SIZE   = 0.2
THRESHOLD   = 0.55
LABEL_COLS  = [
    'religious_hate','ethnic_hate','age_discrimination','gender_hate',
    'sexual_harassment','threats','body_shaming','political_hate',
    'trolling','mental_hate','discrimination','other_cyberbullying_types',
    'not_cyberbullying',
]

# ── Feature Definitions ───────────────────────────────────────
HATE_KEYWORDS = {
    'religious_hate':    ['infidel','kafir','blasphemy','blasphemer','heretic','cult','crusade','jihad','pagan','apostate','heathen','godless','anti-christian','anti-islam','anti-hindu','anti-semitic','zionist','islamophobe','christophobe'],
    'ethnic_hate':       ['nigger','nigga','chink','spic','wetback','gook','kike','raghead','cracker','redneck','monkey','ape','beaner','immigrant','illegal alien','foreigner','outsider','invader','terrorist'],
    'gender_hate':       ['bitch','whore','slut','cunt','feminazi','tranny','dyke','faggot','sissy','mangina','thot','hoe','skank','femoid','incel','simp','soyboy','misogynist','misandrist'],
    'sexual_harassment': ['rape','rapist','molest','grope','pervert','predator','creep','harass','assault'],
    'threats':           ['kill','murder','shoot','stab','bomb','attack','destroy','hurt','harm','threat','die','execute','eliminate','exterminate','torture','burn','acid','explode','lynch','hang','beat','bash','smash'],
    'body_shaming':      ['fat','ugly','obese','disgusting','gross','pig','whale','skinny','anorexic','hideous','repulsive','deformed','freak','monster','troll','goblin'],
    'mental_hate':       ['retard','retarded','psycho','crazy','insane','lunatic','mental','schizo','bipolar','autistic','mentally ill','nutcase','bonkers','deranged','demented'],
    'general_hate':      ['hate','stupid','idiot','moron','dumb','loser','trash','garbage','scum','filth','worthless','pathetic','useless','despise','abhor','vile','toxic','cancer','parasite','vermin','subhuman'],
}
POS_PATTERNS = {
    'NOUN': r'\b(?:man|woman|people|person|thing|way|day|time|year|world|life|child|children|hand|part|place|case|week|company|system|question|government|number|night|point|home|water|room|mother|area|money|story|fact|month|lot|right|study|book|eye|job|word|business|issue|side|kind|head|house|service|friend|father|power|hour|game|line|end|member|city|community|name|president|team|minute|idea|body|information|back|parent|face|level|office|door|health|art|war|history|party|result|change|morning|reason|research|girl|guy|moment|air|teacher|force|education)\b',
    'VERB': r'\b(?:is|are|was|were|be|been|being|have|has|had|do|does|did|will|would|could|should|may|might|shall|must|can|go|get|make|know|think|take|see|come|want|look|use|find|give|tell|work|call|try|ask|need|feel|become|leave|put|mean|keep|let|begin|show|hear|play|run|move|live|believe|hold|bring|happen|write|provide|sit|stand|lose|pay|meet|include|continue|set|learn|change|lead|understand|watch|follow|stop|create|speak|read|spend|grow|open|walk|win|offer|remember|love|consider|appear|buy|wait|serve|die|send|expect|build|stay|fall|cut|reach|kill|remain|suggest|raise|pass|sell|require|report|decide|pull)\b',
    'ADJ':  r'\b(?:good|new|first|last|long|great|little|own|other|old|right|big|high|different|small|large|next|early|young|important|public|private|real|best|free|few|same|able|political|social|economic|national|possible|local|white|black|strong|true|hot|happy|sad|bad|ugly|fat|stupid|evil|dangerous|terrible|horrible|awful|disgusting|pathetic|worthless|useless|dumb|crazy|insane|violent|racist|sexist|offensive|hateful|toxic|radical|extreme)\b',
    'ADV':  r'\b(?:up|so|out|just|now|how|then|more|also|here|well|only|very|even|back|there|down|still|in|as|too|really|most|never|much|often|always|actually|again|further|yet|already|soon|especially|finally|simply|probably|certainly|clearly|literally|absolutely|totally|completely|definitely|obviously|seriously|exactly|nearly|directly|quickly|easily)\b',
    'PRON': r'\b(?:i|me|my|myself|you|your|yourself|he|him|his|himself|she|her|hers|herself|it|its|itself|we|us|our|ourselves|they|them|their|theirs|themselves|this|that|these|those|who|whom|which|what)\b',
}
PROFANITY      = {'fuck','fucking','fucked','shit','ass','asshole','bitch','cunt','bastard','damn','piss','cock','dick','pussy','motherfucker','whore','slut','idiot','moron','loser','stupid','dumb','retard'}
SARC_MARKERS   = [r'\boh great\b',r'\byeah right\b',r'\bsure sure\b',r'\bthanks a lot\b',r'\bthanks so much\b',r'\bso helpful\b',r'\breally helpful\b',r'\bso smart\b',r'\bwow thanks\b',r'\boh wow\b',r'\boh really\b',r'\bno kidding\b',r'\bno way\b',r'\bright sure\b',r'\bas if\b',r'\bcongrats\b',r'\bcongratulations\b',r'\bwhat a surprise\b',r'\bshocking\b',r'\bunbelievable\b',r'\bi\'m sure\b',r'\bim sure\b',r'\bi bet\b',r'\bof course\b',r'\bobviously\b',r'\bclearly\b',r'\bjust great\b',r'\bjust perfect\b',r'\bso funny\b',r'\bhilarious\b']
SARC_POS_WORDS = {'great','wonderful','amazing','fantastic','brilliant','excellent','perfect','awesome','lovely','nice','good','fine','super','best','love','happy','glad','thrilled','helpful','smart','clever','impressive','incredible'}
SARC_NEG_WORDS = {'hate','terrible','awful','horrible','disgusting','pathetic','stupid','idiot','moron','useless','worthless','trash','garbage','fail','worst','bad','ugly','dumb','loser'}
SARC_CONTRA    = [(r'\b(?:great|wonderful|amazing|fantastic)\b',r'\b(?:not|never|no|hate|terrible)\b'),(r'\b(?:love|like|enjoy)\b',r'\b(?:not|never|no|hate|dislike)\b'),(r'\b(?:good|nice|fine)\b',r'\b(?:not|terrible|awful|horrible)\b')]
_INTENSIFIER_RE= re.compile(r'\b(?:so|very|extremely|incredibly|absolutely|totally|completely|literally|seriously|really|such|beyond|super)\b', re.IGNORECASE)
_POS_EMOJI_RE  = re.compile(r'[\U0001F600-\U0001F606\U0001F609\U0001F60A\U0001F60D\U0001F618\U0001F61C\U0001F61D\U0001F923]')


# ── Step 1: Load & Clean ──────────────────────────────────────
def load_and_clean():
    print("📂 Loading data...")
    df = pd.read_csv(DATA_PATH)
    before = len(df)
    df = df[df['text'].notnull() & (df['text'].str.strip() != '')]
    df = df[df['text'].astype(str).str.contains(r'[a-zA-Z]', regex=True)]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^@\w+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^\d+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^[^\w\s]+$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^(.)\1+$|^(\w+)(\s+\2)+\s*$')]
    df = df[~df['text'].fillna('').str.strip().str.match(r'^(#\w+\s*)+$')]
    df = df.reset_index(drop=True)
    print(f"   {before:,} → {len(df):,} rows after cleaning")
    if SAMPLE_SIZE:
        df = df.sample(min(SAMPLE_SIZE, len(df)), random_state=42).reset_index(drop=True)
        print(f"   Sampled: {len(df):,} rows")
    return df


# ── Step 2: Feature Engineering ──────────────────────────────
def extract_features(texts, tfidf=None, fit=False):

    # ============================================================
    # COMPILE FEATURE PATTERNS
    # ============================================================

    hate_pats = {
        cat: re.compile(
            r'\b(?:' + '|'.join(re.escape(k) for k in kws) + r')\b',
            re.IGNORECASE
        )
        for cat, kws in HATE_KEYWORDS.items()
    }

    pos_pats = {
        pos: re.compile(pat, re.IGNORECASE)
        for pos, pat in POS_PATTERNS.items()
    }

    # Sarcasm marker patterns
    marker_pats = [
        re.compile(p, re.IGNORECASE)
        for p in SARC_MARKERS
    ]

    # Sarcasm contradiction patterns
    contra_pats = [
        (
            re.compile(a, re.IGNORECASE),
            re.compile(b, re.IGNORECASE)
        )
        for a, b in SARC_CONTRA
    ]

    # Other patterns
    url_re = re.compile(r'https?://\S+|www\.\S+')
    mention_re = re.compile(r'@\w+')
    hashtag_re = re.compile(r'#\w+')

    emoji_re = re.compile(
        r'[\U0001F300-\U0001F9FF\U00002600-\U000027FF]',
        flags=re.UNICODE
    )

    repeat_re = re.compile(r'(.)\1{2,}')
    caps_re = re.compile(r'\b[A-Z]{2,}\b')


    # ============================================================
    # EXTRACT FEATURES
    # ============================================================

    all_rows = []

    for text in texts:

        t = str(text)

        words = t.split()

        wc = max(len(words), 1)

        cc = max(len(t), 1)

        lw = [
            w.lower().strip('.,!?;:\'"')
            for w in words
        ]

        sents = max(
            len(re.split(r'[.!?]+', t)),
            1
        )

        # Word frequency dictionary
        wfreq = {}

        for w in lw:
            wfreq[w] = wfreq.get(w, 0) + 1

        caps = caps_re.findall(t)

        prof = sum(
            1 for w in lw
            if w in PROFANITY
        )

        row = {}


        # ========================================================
        # 1. HATE KEYWORD FEATURES
        # ========================================================

        total_hate = 0

        for cat, pat in hate_pats.items():

            c = len(pat.findall(t))

            row[f'hate_count_{cat}'] = c

            row[f'hate_flag_{cat}'] = int(c > 0)

            total_hate += c

        row['hate_total_count'] = total_hate

        row['hate_density'] = (
            total_hate / wc
        )

        row['hate_category_count'] = sum(
            1
            for cat in HATE_KEYWORDS
            if row[f'hate_flag_{cat}'] > 0
        )


        # ========================================================
        # 2. POS FEATURES
        # ========================================================

        pcounts = {}

        for pos, pat in pos_pats.items():

            c = len(pat.findall(t))

            pcounts[pos] = c

            row[f'pos_count_{pos.lower()}'] = c

            row[f'pos_ratio_{pos.lower()}'] = (
                c / wc
            )

        row['pos_adj_noun_ratio'] = (
            pcounts.get('ADJ', 0)
            /
            max(pcounts.get('NOUN', 0), 1)
        )

        row['pos_verb_density'] = (
            pcounts.get('VERB', 0) / wc
        )

        row['pos_pron_density'] = (
            pcounts.get('PRON', 0) / wc
        )

        row['pos_modifier_density'] = (
            pcounts.get('ADJ', 0)
            +
            pcounts.get('ADV', 0)
        ) / wc


        # ========================================================
        # 3. WORD FREQUENCY / LINGUISTIC FEATURES
        # ========================================================

        row.update({

            'wf_char_count':
                cc,

            'wf_word_count':
                wc,

            'wf_sentence_count':
                sents,

            'wf_avg_word_len':
                np.mean(
                    [len(w) for w in words]
                ) if words else 0,

            'wf_avg_sent_len':
                wc / sents,

            'wf_unique_word_ratio':
                len(set(lw)) / wc,

            'wf_caps_word_count':
                len(caps),

            'wf_caps_ratio':
                len(caps) / wc,

            'wf_upper_char_ratio':
                sum(
                    1 for c in t
                    if c.isupper()
                ) / cc,

            'wf_exclaim_count':
                t.count('!'),

            'wf_question_count':
                t.count('?'),

            'wf_ellipsis_count':
                t.count('...'),

            'wf_punct_density':
                sum(
                    1 for c in t
                    if c in '!?.,;:\'"'
                ) / cc,

            'wf_url_count':
                len(url_re.findall(t)),

            'wf_mention_count':
                len(mention_re.findall(t)),

            'wf_hashtag_count':
                len(hashtag_re.findall(t)),

            'wf_emoji_count':
                len(emoji_re.findall(t)),

            'wf_repeat_char_count':
                len(repeat_re.findall(t)),

            'wf_repeat_word_count':
                sum(
                    1 for v in wfreq.values()
                    if v > 1
                ),

            'wf_profanity_count':
                prof,

            'wf_profanity_density':
                prof / wc
        })


        # ========================================================
        # 4. SARCASM FEATURE ENGINEERING
        #
        # IMPORTANT:
        # The 17 raw sarcasm-related features are used ONLY
        # internally to calculate the final heuristic score s.
        #
        # ONLY the final normalized score is passed to the SVM.
        # ========================================================

        # Raw feature 1: Multiple exclamation marks
        multi_exc = len(
            re.findall(r'!{2,}', t)
        )

        # Raw feature 2: Sarcasm markers
        mc = sum(
            1
            for p in marker_pats
            if p.search(t)
        )

        # Raw features: positive and negative word counts
        pc = sum(
            1
            for w in lw
            if w in SARC_POS_WORDS
        )

        nc = sum(
            1
            for w in lw
            if w in SARC_NEG_WORDS
        )

        # Raw feature: contradiction patterns
        cc2 = sum(
            1
            for a_p, b_p in contra_pats
            if a_p.search(t)
            and b_p.search(t)
        )

        # Raw feature: intensifier density
        id_ = (
            len(
                _INTENSIFIER_RE.findall(t)
            )
            / wc
        )

        # Raw feature: positive emoji + negative text contradiction
        ec = int(
            len(
                _POS_EMOJI_RE.findall(t)
            ) > 0
            and nc > 0
        )

        # Raw feature: capitalization ratio
        caps_ratio = (
            len(caps) / wc
        )


        # ========================================================
        # 5. WEIGHTED SARCASM HEURISTIC SCORE
        #
        # Only this final normalized value is supplied to SVM.
        # ========================================================

        sarcasm_score = min(

            (
                mc * 2.0
                +
                multi_exc * 1.5
                +
                cc2 * 2.0
                +
                int(pc > 0 and nc > 0) * 1.5
                +
                caps_ratio * 3.0
                +
                id_ * 2.0
                +
                ec * 1.5
            )
            / 10.0,

            1.0
        )


        # ========================================================
        # ONLY FINAL SARCASM SCORE IS ADDED
        # ========================================================

        row['sarc_heuristic_score'] = sarcasm_score


        # ========================================================
        # SAVE ROW
        # ========================================================

        all_rows.append(row)


    # ============================================================
    # CONVERT NON-TEXT FEATURES TO NUMPY
    # ============================================================

    dense = pd.DataFrame(
        all_rows
    ).values.astype(
        np.float32
    )

    dense = np.clip(
        dense,
        0,
        None
    )


    # ============================================================
    # TF-IDF FEATURES
    # ============================================================

    if fit:

        tfidf = TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            min_df=3,
            sublinear_tf=True
        )

        tfidf_mat = tfidf.fit_transform(
            texts.astype(str)
        )

    else:

        tfidf_mat = tfidf.transform(
            texts.astype(str)
        )


    # ============================================================
    # COMBINE ALL FEATURES
    #
    # Dense features:
    #   Hate Keywords
    #   POS
    #   Word Frequency
    #   ONLY ONE SARCASM FEATURE (final score s)
    #
    # Sparse features:
    #   TF-IDF
    # ============================================================

    X = hstack(
        [
            csr_matrix(dense),
            tfidf_mat
        ],
        format='csr'
    )


    return X, tfidf


# ── Step 3: Evaluate ──────────────────────────────────────────
def evaluate(y_true, y_pred, elapsed=None):

    # ── Per-label accuracy ────────────────────────────────────
    per_label_acc = []
    for i in range(y_true.shape[1]):
        acc = accuracy_score(y_true[:, i], y_pred[:, i])
        per_label_acc.append(acc)
    avg_per_label_acc = np.mean(per_label_acc)

    # ── Overall subset accuracy ───────────────────────────────
    subset_acc   = accuracy_score(y_true, y_pred)

    # ── Hamming accuracy ──────────────────────────────────────
    hamming_acc  = 1 - hamming_loss(y_true, y_pred)

    print("\n" + "="*58)
    print("  📊 SVM — Accuracy Report")
    if elapsed: print(f"  ⏱  Training time: {elapsed:.1f}s")
    print("="*58)

    print(f"""
  ┌──────────────────────────────────────────────────┐
  │                  ACCURACY                        │
  ├──────────────────────────────────────────────────┤
  │  Per-Label Accuracy  : {avg_per_label_acc:.4f}  ✅ MAIN ACCURACY   │
  │    (avg accuracy across all 13 labels)           │
  │                                                  │
  │  Hamming Accuracy    : {hamming_acc:.4f}                    │
  │    (% of individual label predictions correct)   │
  │                                                  │
  │  Subset Accuracy     : {subset_acc:.4f}                    │
  │    (all 13 labels exactly correct — very strict) │
  └──────────────────────────────────────────────────┘
""")

    print(f"  F1 Micro   (main)    : {f1_score(y_true, y_pred, average='micro',    zero_division=0):.4f}  ✅")
    print(f"  F1 Macro             : {f1_score(y_true, y_pred, average='macro',    zero_division=0):.4f}")
    print(f"  F1 Weighted          : {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"  F1 Samples           : {f1_score(y_true, y_pred, average='samples',  zero_division=0):.4f}")
    print(f"  Precision (micro)    : {precision_score(y_true, y_pred, average='micro', zero_division=0):.4f}")
    print(f"  Recall    (micro)    : {recall_score(y_true,    y_pred, average='micro', zero_division=0):.4f}")

    print(f"\n  {'Label':<30} {'Accuracy':>10} {'F1':>8} {'Support':>10}")
    print(f"  {'-'*30} {'-'*10} {'-'*8} {'-'*10}")
    f1s = []
    for i, label in enumerate(LABEL_COLS):
        acc = accuracy_score(y_true[:,i], y_pred[:,i])
        f   = f1_score(y_true[:,i], y_pred[:,i], zero_division=0)
        s   = int(y_true[:,i].sum())
        bar = '█'*int(f*10)+'░'*(10-int(f*10))
        print(f"  {label:<30} {acc:>10.4f} {f:>8.4f} {s:>10,}  {bar}")
        f1s.append((label, f))

    best  = max(f1s, key=lambda x: x[1])
    worst = min(f1s, key=lambda x: x[1])
    print(f"\n  🏆 Best  : {best[0]:<30}  F1={best[1]:.4f}")
    print(f"  ⚠️  Worst : {worst[0]:<30}  F1={worst[1]:.4f}")
    print("="*58)
    print(f"\n  📌 Your model accuracy = {avg_per_label_acc*100:.2f}%")


# ── Step 4: Real Time Detection ───────────────────────────────
def detect(text, model, tfidf, threshold=THRESHOLD):
    X        = extract_features(pd.Series([text]), tfidf=tfidf, fit=False)[0]
    proba    = model.predict_proba(X)[0]
    pred     = (proba >= threshold).astype(int)
    detected = [LABEL_COLS[i] for i, p in enumerate(pred)
                if p == 1 and LABEL_COLS[i] != 'not_cyberbullying']
    print("\n" + "="*52)
    print(f"  📝 Text      : {text[:65]}{'...' if len(text)>65 else ''}")
    print(f"  🎚  Threshold : {threshold}")
    print("="*52)
    if not detected:
        print("  ✅  NOT CYBERBULLYING")
    else:
        print("  🚨  CYBERBULLYING DETECTED")
        print("  Categories found:")
        for label in detected:
            conf = proba[LABEL_COLS.index(label)]
            bar  = '█'*int(conf*10)+'░'*(10-int(conf*10))
            print(f"    ⚠️  {label:<30} {conf:.0%}  {bar}")
    print("="*52)
    return detected

    # ── Step 5: LIME Explainability ──────────────────────────────
def explain_prediction(text, model, tfidf, label_index=0):

    class_names = ['Not Present', 'Present']

    explainer = LimeTextExplainer(
        class_names=class_names
    )

    # --------------------------------------------------------
    # Prediction function for LIME
    # --------------------------------------------------------

    def predictor(texts):

        X, _ = extract_features(
            pd.Series(texts),
            tfidf=tfidf,
            fit=False
        )

        # Probability predictions
        probs = model.estimators_[label_index].predict_proba(X)

        return probs

    # --------------------------------------------------------
    # Generate explanation
    # --------------------------------------------------------

    explanation = explainer.explain_instance(
        text,
        predictor,
        num_features=10
    )

    # --------------------------------------------------------
    # Print explanation
    # --------------------------------------------------------

    print("\n" + "=" * 52)
    print("  🧠 LIME EXPLANATION")
    print(f"  📌 Label : {LABEL_COLS[label_index]}")
    print("=" * 52)

    for word, weight in explanation.as_list():

        effect = "⬆️ increases" if weight > 0 else "⬇️ decreases"

        print(
            f"  {word:<20} "
            f"{weight:+.4f}   "
            f"{effect}"
        )

    print("=" * 52)

    # --------------------------------------------------------
    # Notebook visualization
    # --------------------------------------------------------

    try:
        explanation.show_in_notebook(text=True)
    except:
        pass

    return explanation

# ── Step 6: Real Time Loop + LIME ────────────────────────────
def run_detector(model, tfidf):

    print("\n" + "=" * 52)
    print("  🤖 Cyberbullying Detector (SVM + LIME)")
    print(f"  Threshold: {THRESHOLD}  |  Type 'exit' to stop")
    print("=" * 52 + "\n")

    while True:

        text = input("Enter text: ").strip()

        if text.lower() == 'exit':
            print("👋 Stopping...")
            break

        if not text:
            print("⚠️  Please enter some text\n")
            continue

        # ----------------------------------------------------
        # Run prediction
        # ----------------------------------------------------

        detected = detect(
            text,
            model,
            tfidf,
            threshold=THRESHOLD
        )

        # ----------------------------------------------------
        # Generate LIME explanations
        # ----------------------------------------------------

        if detected:

            print("\n🧠 Generating LIME explanations...\n")

            for label in detected:

                label_index = LABEL_COLS.index(label)

                explain_prediction(
                    text,
                    model,
                    tfidf,
                    label_index=label_index
                )

        else:

            print("\n✅ No bullying labels detected.")
            print("ℹ️  Skipping LIME explanation.\n")

# ── Main ──────────────────────────────────────────────────────
def main():
    print("="*55)
    print("  MODEL: SVM — Self Contained")
    print("="*55)
    df = load_and_clean()
    df_train, df_test = train_test_split(df, test_size=TEST_SIZE, random_state=42)
    df_train = df_train.reset_index(drop=True); df_test = df_test.reset_index(drop=True)
    print(f"   Train: {len(df_train):,}  |  Test: {len(df_test):,}")
    print("\n🔧 Extracting ALL features (HateKeyword+POS+WordFreq+SarcasmScore+TFIDF)...")
    X_train, tfidf = extract_features(df_train['text'], fit=True)
    X_test,  _     = extract_features(df_test['text'],  tfidf=tfidf, fit=False)
    print(f"   ✅ Features: {X_train.shape[1]:,}")
    y_train = df_train[LABEL_COLS].values; y_test = df_test[LABEL_COLS].values
    print("\n🚀 Training SVM (with probability calibration)...")
    t0    = time.time()
    base  = LinearSVC(C=0.5, max_iter=2000, class_weight='balanced', dual=False)
    model = OneVsRestClassifier(CalibratedClassifierCV(base, cv=3), n_jobs=-1)
    model.fit(X_train, y_train)
    elapsed = time.time()-t0; print(f"   Done in {elapsed:.1f}s")
    proba  = model.predict_proba(X_test)
    y_pred = (proba >= THRESHOLD).astype(int)
    evaluate(y_test, y_pred, elapsed)
    run_detector(model, tfidf)
    return model, tfidf

if __name__ == '__main__':
    main()

  MODEL: SVM — Self Contained
📂 Loading data...
   684,383 → 684,172 rows after cleaning
   Train: 547,337  |  Test: 136,835

🔧 Extracting ALL features (HateKeyword+POS+WordFreq+Sarcasm+TFIDF)...
   ✅ Features: 20,072

🚀 Training SVM (with probability calibration)...


KeyboardInterrupt: 

In [ ]:
# ================================================================
# UNIFORM-WEIGHT SARCASM SVM EXPERIMENT
# ================================================================



import re
import time
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    f1_score,
    precision_score,
    recall_score
)


# ================================================================
# CONFIGURATION
# ================================================================

DATA_PATH = '/content/drive/MyDrive/Cyberbullying_Project/cyberbullying.csv'

TEST_SIZE = 0.2
RANDOM_STATE = 42

# Decision threshold
# 0.0 is the default LinearSVC decision boundary.
# You can tune this later if required.
THRESHOLD = 0.0


LABEL_COLS = [
    'religious_hate',
    'ethnic_hate',
    'age_discrimination',
    'gender_hate',
    'sexual_harassment',
    'threats',
    'body_shaming',
    'political_hate',
    'trolling',
    'mental_hate',
    'discrimination',
    'other_cyberbullying_types',
    'not_cyberbullying'
]


# ================================================================
# FEATURE DEFINITIONS
# ================================================================

HATE_KEYWORDS = {

    'religious_hate': [
        'infidel','kafir','blasphemy','blasphemer','heretic',
        'cult','crusade','jihad','pagan','apostate','heathen',
        'godless','anti-christian','anti-islam','anti-hindu',
        'anti-semitic','zionist','islamophobe','christophobe'
    ],

    'ethnic_hate': [
        'nigger','nigga','chink','spic','wetback','gook','kike',
        'raghead','cracker','redneck','monkey','ape','beaner',
        'immigrant','illegal alien','foreigner','outsider',
        'invader','terrorist'
    ],

    'gender_hate': [
        'bitch','whore','slut','cunt','feminazi','tranny','dyke',
        'faggot','sissy','mangina','thot','hoe','skank','femoid',
        'incel','simp','soyboy','misogynist','misandrist'
    ],

    'sexual_harassment': [
        'rape','rapist','molest','grope','pervert','predator',
        'creep','harass','assault'
    ],

    'threats': [
        'kill','murder','shoot','stab','bomb','attack','destroy',
        'hurt','harm','threat','die','execute','eliminate',
        'exterminate','torture','burn','acid','explode','lynch',
        'hang','beat','bash','smash'
    ],

    'body_shaming': [
        'fat','ugly','obese','disgusting','gross','pig','whale',
        'skinny','anorexic','hideous','repulsive','deformed',
        'freak','monster','troll','goblin'
    ],

    'mental_hate': [
        'retard','retarded','psycho','crazy','insane','lunatic',
        'mental','schizo','bipolar','autistic','mentally ill',
        'nutcase','bonkers','deranged','demented'
    ],

    'general_hate': [
        'hate','stupid','idiot','moron','dumb','loser','trash',
        'garbage','scum','filth','worthless','pathetic','useless',
        'despise','abhor','vile','toxic','cancer','parasite',
        'vermin','subhuman'
    ]
}


POS_PATTERNS = {

    'NOUN':
        r'\b(?:man|woman|people|person|thing|way|day|time|year|'
        r'world|life|child|children|hand|part|place|case|week|'
        r'company|system|question|government|number|night|point|'
        r'home|water|room|mother|area|money|story|fact|month|lot|'
        r'right|study|book|eye|job|word|business|issue|side|kind|'
        r'head|house|service|friend|father|power|hour|game|line|'
        r'end|member|city|community|name|president|team|minute|'
        r'idea|body|information|back|parent|face|level|office|'
        r'door|health|art|war|history|party|result|change|morning|'
        r'reason|research|girl|guy|moment|air|teacher|force|education)\b',

    'VERB':
        r'\b(?:is|are|was|were|be|been|being|have|has|had|do|'
        r'does|did|will|would|could|should|may|might|shall|must|'
        r'can|go|get|make|know|think|take|see|come|want|look|use|'
        r'find|give|tell|work|call|try|ask|need|feel|become|leave|'
        r'put|mean|keep|let|begin|show|hear|play|run|move|live|'
        r'believe|hold|bring|happen|write|provide|sit|stand|lose|'
        r'pay|meet|include|continue|set|learn|change|lead|understand|'
        r'watch|follow|stop|create|speak|read|spend|grow|open|walk|'
        r'win|offer|remember|love|consider|appear|buy|wait|serve|'
        r'die|send|expect|build|stay|fall|cut|reach|kill|remain|'
        r'suggest|raise|pass|sell|require|report|decide|pull)\b',

    'ADJ':
        r'\b(?:good|new|first|last|long|great|little|own|other|'
        r'old|right|big|high|different|small|large|next|early|young|'
        r'important|public|private|real|best|free|few|same|able|'
        r'political|social|economic|national|possible|local|white|'
        r'black|strong|true|hot|happy|sad|bad|ugly|fat|stupid|evil|'
        r'dangerous|terrible|horrible|awful|disgusting|pathetic|'
        r'worthless|useless|dumb|crazy|insane|violent|racist|sexist|'
        r'offensive|hateful|toxic|radical|extreme)\b',

    'ADV':
        r'\b(?:up|so|out|just|now|how|then|more|also|here|well|'
        r'only|very|even|back|there|down|still|in|as|too|really|'
        r'most|never|much|often|always|actually|again|further|yet|'
        r'already|soon|especially|finally|simply|probably|certainly|'
        r'clearly|literally|absolutely|totally|definitely|obviously|'
        r'seriously|exactly|nearly|directly|quickly|easily)\b',

    'PRON':
        r'\b(?:i|me|my|myself|you|your|yourself|he|him|his|himself|'
        r'she|her|hers|herself|it|its|itself|we|us|our|ourselves|'
        r'they|them|their|theirs|themselves|this|that|these|those|'
        r'who|whom|which|what)\b'
}


PROFANITY = {
    'fuck','fucking','fucked','shit','ass','asshole','bitch',
    'cunt','bastard','damn','piss','cock','dick','pussy',
    'motherfucker','whore','slut','idiot','moron','loser',
    'stupid','dumb','retard'
}


# ================================================================
# SARCASM DEFINITIONS
# UNIFORM-WEIGHT VERSION
# ================================================================

SARC_MARKERS = [

    r'\boh great\b',
    r'\byeah right\b',
    r'\bsure sure\b',
    r'\bthanks a lot\b',
    r'\bthanks so much\b',
    r'\bso helpful\b',
    r'\breally helpful\b',
    r'\bso smart\b',
    r'\bwow thanks\b',
    r'\boh wow\b',
    r'\boh really\b',
    r'\bno kidding\b',
    r'\bno way\b',
    r'\bright sure\b',
    r'\bas if\b',
    r'\bcongrats\b',
    r'\bcongratulations\b',
    r'\bwhat a surprise\b',
    r'\bshocking\b',
    r'\bunbelievable\b',
    r'\bi\'m sure\b',
    r'\bim sure\b',
    r'\bi bet\b',
    r'\bof course\b',
    r'\bobviously\b',
    r'\bclearly\b',
    r'\bjust great\b',
    r'\bjust perfect\b',
    r'\bso funny\b',
    r'\bhilarious\b'
]


SARC_POS_WORDS = {
    'great','wonderful','amazing','fantastic','brilliant',
    'excellent','perfect','awesome','lovely','nice','good',
    'fine','super','best','love','happy','glad','thrilled',
    'helpful','smart','clever','impressive','incredible'
}


SARC_NEG_WORDS = {
    'hate','terrible','awful','horrible','disgusting','pathetic',
    'stupid','idiot','moron','useless','worthless','trash',
    'garbage','fail','worst','bad','ugly','dumb','loser'
}


SARC_CONTRA = [

    (
        r'\b(?:great|wonderful|amazing|fantastic)\b',
        r'\b(?:not|never|no|hate|terrible)\b'
    ),

    (
        r'\b(?:love|like|enjoy)\b',
        r'\b(?:not|never|no|hate|dislike)\b'
    ),

    (
        r'\b(?:good|nice|fine)\b',
        r'\b(?:not|terrible|awful|horrible)\b'
    )
]


_INTENSIFIER_RE = re.compile(
    r'\b(?:so|very|extremely|incredibly|absolutely|totally|'
    r'completely|literally|seriously|really|such|beyond|super)\b',
    re.IGNORECASE
)


_POS_EMOJI_RE = re.compile(
    r'[\U0001F600-\U0001F606'
    r'\U0001F609\U0001F60A'
    r'\U0001F60D\U0001F618'
    r'\U0001F61C\U0001F61D'
    r'\U0001F923]'
)


# ================================================================
# STEP 1: LOAD AND CLEAN DATA
# ================================================================

def load_and_clean():
    print("📂 Loading data...")
    df = pd.read_csv(DATA_PATH)
    before = len(df)
    df = df[
        df['text'].notnull()
        & (df['text'].astype(str).str.strip() != '')
    ]
    df = df[
        df['text'].astype(str).str.contains(
            r'[a-zA-Z]',
            regex=True
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^@\w+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^\d+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^[^\w\s]+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^(.)\1+$|^(\w+)(\s+\2)+\s*$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^(#\w+\s*)+$'
        )
    ]

    df = df.reset_index(drop=True)

    print(
        f"   {before:,} → {len(df):,} rows after cleaning"
    )

    return df



# ============================================================
# HEURISTIC WEIGHTS (+10% Sensitivity Analysis)
# ============================================================

W_MC   = 2.2    # Sarcasm marker count
W_EX   = 1.65   # Repeated exclamation pattern count
W_CC   = 2.2    # Contextual contradiction count
W_PN   = 1.65   # Positive-negative sentiment incongruity
W_CAPS = 3.3    # ALL-CAPS ratio
W_ID   = 2.2    # Intensifier density
W_EC   = 1.65   # Emoji-text sentiment contradiction

# ================================================================
# STEP 2: FEATURE EXTRACTION
# ================================================================


NORMALIZATION_FACTOR = 10.0
def extract_features(
    texts,
    tfidf=None,
    fit=False
):

    print("   Extracting features...")

    # ------------------------------------------------------------
    # Compile patterns once
    # ------------------------------------------------------------

    hate_pats = {

        cat: re.compile(
            r'\b(?:'
            + '|'.join(
                re.escape(k)
                for k in kws
            )
            + r')\b',
            re.IGNORECASE
        )

        for cat, kws in HATE_KEYWORDS.items()
    }


    pos_pats = {

        pos: re.compile(
            pat,
            re.IGNORECASE
        )

        for pos, pat in POS_PATTERNS.items()
    }


    marker_pats = [

        re.compile(
            p,
            re.IGNORECASE
        )

        for p in SARC_MARKERS
    ]


    contra_pats = [

        (
            re.compile(
                a,
                re.IGNORECASE
            ),

            re.compile(
                b,
                re.IGNORECASE
            )
        )

        for a, b in SARC_CONTRA
    ]


    url_re = re.compile(
        r'https?://\S+|www\.\S+'
    )

    mention_re = re.compile(
        r'@\w+'
    )

    hashtag_re = re.compile(
        r'#\w+'
    )

    emoji_re = re.compile(
        r'[\U0001F300-\U0001F9FF'
        r'\U00002600-\U000027FF]',
        flags=re.UNICODE
    )

    repeat_re = re.compile(
        r'(.)\1{2,}'
    )

    caps_re = re.compile(
        r'\b[A-Z]{2,}\b'
    )


    all_rows = []


    # ============================================================
    # PROCESS EACH TEXT
    # ============================================================

    for text in texts:

        t = str(text)

        words = t.split()

        wc = max(
            len(words),
            1
        )

        cc = max(
            len(t),
            1
        )


        lw = [

            w.lower().strip(
                '.,!?;:\'"'
            )

            for w in words
        ]


        sents = max(

            len(
                re.split(
                    r'[.!?]+',
                    t
                )
            ),

            1
        )


        # --------------------------------------------------------
        # Word frequency
        # --------------------------------------------------------

        wfreq = {}

        for w in lw:

            wfreq[w] = (
                wfreq.get(
                    w,
                    0
                )
                + 1
            )


        caps = caps_re.findall(t)


        prof = sum(

            1

            for w in lw

            if w in PROFANITY
        )


        row = {}


        # ========================================================
        # HATE KEYWORD FEATURES
        # ========================================================

        total_hate = 0


        for cat, pat in hate_pats.items():

            c = len(
                pat.findall(t)
            )


            row[
                f'hate_count_{cat}'
            ] = c


            row[
                f'hate_flag_{cat}'
            ] = int(
                c > 0
            )


            total_hate += c


        row[
            'hate_total_count'
        ] = total_hate


        row[
            'hate_density'
        ] = total_hate / wc


        row[
            'hate_category_count'
        ] = sum(

            1

            for cat in HATE_KEYWORDS

            if row[
                f'hate_flag_{cat}'
            ] > 0
        )


        # ========================================================
        # POS FEATURES
        # ========================================================

        pcounts = {}


        for pos, pat in pos_pats.items():

            c = len(
                pat.findall(t)
            )


            pcounts[pos] = c


            row[
                f'pos_count_{pos.lower()}'
            ] = c


            row[
                f'pos_ratio_{pos.lower()}'
            ] = c / wc


        row[
            'pos_adj_noun_ratio'
        ] = (

            pcounts.get(
                'ADJ',
                0
            )

            /

            max(
                pcounts.get(
                    'NOUN',
                    0
                ),
                1
            )
        )


        row[
            'pos_verb_density'
        ] = (

            pcounts.get(
                'VERB',
                0
            )

            / wc
        )


        row[
            'pos_pron_density'
        ] = (

            pcounts.get(
                'PRON',
                0
            )

            / wc
        )


        row[
            'pos_modifier_density'
        ] = (

            pcounts.get(
                'ADJ',
                0
            )

            +

            pcounts.get(
                'ADV',
                0
            )

        ) / wc


        # ========================================================
        # WORD FREQUENCY FEATURES
        # ========================================================

        row.update({

            'wf_char_count':
                cc,

            'wf_word_count':
                wc,

            'wf_sentence_count':
                sents,

            'wf_avg_word_len':
                np.mean(
                    [
                        len(w)
                        for w in words
                    ]
                )
                if words
                else 0,

            'wf_avg_sent_len':
                wc / sents,

            'wf_unique_word_ratio':
                len(
                    set(lw)
                ) / wc,

            'wf_caps_word_count':
                len(caps),

            'wf_caps_ratio':
                len(caps) / wc,

            'wf_upper_char_ratio':
                sum(
                    1
                    for c in t
                    if c.isupper()
                ) / cc,

            'wf_exclaim_count':
                t.count('!'),

            'wf_question_count':
                t.count('?'),

            'wf_ellipsis_count':
                t.count('...'),

            'wf_punct_density':
                sum(
                    1
                    for c in t
                    if c in '!?.,;:\'"'
                ) / cc,

            'wf_url_count':
                len(
                    url_re.findall(t)
                ),

            'wf_mention_count':
                len(
                    mention_re.findall(t)
                ),

            'wf_hashtag_count':
                len(
                    hashtag_re.findall(t)
                ),

            'wf_emoji_count':
                len(
                    emoji_re.findall(t)
                ),

            'wf_repeat_char_count':
                len(
                    repeat_re.findall(t)
                ),

            'wf_repeat_word_count':
                sum(

                    1

                    for v in wfreq.values()

                    if v > 1
                ),

            'wf_profanity_count':
                prof,

            'wf_profanity_density':
                prof / wc
        })


        # ========================================================
        # UNIFORM-WEIGHT SARCASM SCORE
        # ========================================================

        multi_exc = len(
            re.findall(
                r'!{2,}',
                t
            )
        )


        mc = sum(

            1

            for p in marker_pats

            if p.search(t)
        )


        pc = sum(

            1

            for w in lw

            if w in SARC_POS_WORDS
        )


        nc = sum(

            1

            for w in lw

            if w in SARC_NEG_WORDS
        )


        cc2 = sum(

            1

            for a_p, b_p in contra_pats

            if (
                a_p.search(t)
                and
                b_p.search(t)
            )
        )


        id_ = (

            len(
                _INTENSIFIER_RE.findall(t)
            )

            / wc
        )


        ec = int(

            len(
                _POS_EMOJI_RE.findall(t)
            ) > 0

            and

            nc > 0
        )

        # --------------------------------------------------------
        # UNIFORM WEIGHTS
        # --------------------------------------------------------

        # --------------------------------------------------------
        # WEIGHTED SARCASM SCORE (Sensitivity Analysis)
        # --------------------------------------------------------

        sarcasm_score = min(
        (
          W_MC * mc
          + W_EX * multi_exc
          + W_CC * cc2
          + W_PN * int(
            pc > 0
            and
            nc > 0
          )
          + W_CAPS * (len(caps) / wc)
          + W_ID * id_
          + W_EC * ec
        ) / NORMALIZATION_FACTOR,
        1.0
        )





        # --------------------------------------------------------
        # ONLY FINAL SARCASM SCORE IS ADDED
        # --------------------------------------------------------

        row[
            'sarc_heuristic_score'
        ] = sarcasm_score


        all_rows.append(row)


    # ============================================================
    # CONVERT DENSE FEATURES
    # ============================================================

    dense = pd.DataFrame(
        all_rows
    ).values.astype(
        np.float32
    )


    dense = np.clip(
        dense,
        0,
        None
    )


    # ============================================================
    # TF-IDF
    # ============================================================

    if fit:

        print(
            "   Fitting TF-IDF..."
        )


        tfidf = TfidfVectorizer(

            max_features=20000,

            ngram_range=(1, 2),

            min_df=3,

            sublinear_tf=True
        )


        tfidf_mat = tfidf.fit_transform(

            texts.astype(str)
        )


    else:

        tfidf_mat = tfidf.transform(

            texts.astype(str)
        )


    # ============================================================
    # COMBINE FEATURES
    # ============================================================

    X = hstack(

        [

            csr_matrix(
                dense
            ),

            tfidf_mat

        ],

        format='csr'
    )


    return X, tfidf


# ================================================================
# EVALUATION
# ================================================================

def evaluate(
    y_true,
    y_pred,
    elapsed
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        "UNIFORM-WEIGHT SARCASM SVM RESULTS"
    )

    print(
        "=" * 70
    )


    # ------------------------------------------------------------
    # Accuracy
    # ------------------------------------------------------------

    subset_accuracy = accuracy_score(

        y_true,
        y_pred
    )


    hamming_accuracy = (

        1

        -

        hamming_loss(

            y_true,
            y_pred
        )
    )


    # ------------------------------------------------------------
    # F1
    # ------------------------------------------------------------

    micro_f1 = f1_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    macro_f1 = f1_score(

        y_true,
        y_pred,

        average='macro',

        zero_division=0
    )


    weighted_f1 = f1_score(

        y_true,
        y_pred,

        average='weighted',

        zero_division=0
    )


    # ------------------------------------------------------------
    # Precision / Recall
    # ------------------------------------------------------------

    micro_precision = precision_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    micro_recall = recall_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    print(
        f"\nTraining Time      : "
        f"{elapsed / 60:.2f} minutes"
    )

    print(
        f"Subset Accuracy    : "
        f"{subset_accuracy:.4f}"
    )

    print(
        f"Hamming Accuracy   : "
        f"{hamming_accuracy:.4f}"
    )

    print(
        f"Micro Precision    : "
        f"{micro_precision:.4f}"
    )

    print(
        f"Micro Recall       : "
        f"{micro_recall:.4f}"
    )

    print(
        f"Micro F1           : "
        f"{micro_f1:.4f}"
    )

    print(
        f"Macro F1           : "
        f"{macro_f1:.4f}"
    )

    print(
        f"Weighted F1        : "
        f"{weighted_f1:.4f}"
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        "FINAL MICRO F1 FOR UNIFORM SARCASM"
    )

    print(
        f"{micro_f1:.6f}"
    )

    print(
        "=" * 70
    )


    return {

        'subset_accuracy':
            subset_accuracy,

        'hamming_accuracy':
            hamming_accuracy,

        'micro_precision':
            micro_precision,

        'micro_recall':
            micro_recall,

        'micro_f1':
            micro_f1,

        'macro_f1':
            macro_f1,

        'weighted_f1':
            weighted_f1
    }


# ================================================================
# MAIN
# ================================================================

def main():

    print(
        "\n"
        + "=" * 70
    )

    print("SENSITIVITY ANALYSIS: +10% HEURISTIC WEIGHTS")


    print(
        "=" * 70
    )


    # ============================================================
    # LOAD DATA
    # ============================================================

    df = load_and_clean()


    # ============================================================
    # TRAIN-TEST SPLIT
    # ============================================================

    df_train, df_test = train_test_split(

        df,

        test_size=TEST_SIZE,

        random_state=RANDOM_STATE
    )


    df_train = df_train.reset_index(
        drop=True
    )


    df_test = df_test.reset_index(
        drop=True
    )


    print(
        f"\nTrain samples : "
        f"{len(df_train):,}"
    )


    print(
        f"Test samples  : "
        f"{len(df_test):,}"
    )


    # ============================================================
    # FEATURE EXTRACTION
    # ============================================================

    print(
        "\n🔧 Extracting features..."
    )


    print("Sarcasm mode: +10% heuristic weights")


    X_train, tfidf = extract_features(

        df_train['text'],

        fit=True
    )


    X_test, _ = extract_features(

        df_test['text'],

        tfidf=tfidf,

        fit=False
    )


    print(
        f"\nFeature matrix: "
        f"{X_train.shape[1]:,} features"
    )


    # ============================================================
    # LABEL MATRICES
    # ============================================================

    y_train = df_train[
        LABEL_COLS
    ].values.astype(
        np.int8
    )


    y_test = df_test[
        LABEL_COLS
    ].values.astype(
        np.int8
    )


    print(
        f"Label matrix: "
        f"{y_train.shape}"
    )


    # ============================================================
    # TRAIN LINEAR SVM
    # ============================================================

    print(
        "\n🚀 Training Linear SVM..."
    )


    print(
        "   NOTE: LinearSVC uses CPU, "
        "not the T4 GPU."
    )


    t0 = time.time()


    base_svm = LinearSVC(

        C=0.5,

        max_iter=2000,

        class_weight='balanced',

        dual=False
    )


    model = OneVsRestClassifier(

        base_svm,

        n_jobs=-1
    )


    model.fit(

        X_train,

        y_train
    )


    elapsed = (

        time.time()

        -

        t0
    )


    print(
        f"\n✅ SVM training completed "
        f"in {elapsed / 60:.2f} minutes"
    )


    # ============================================================
    # PREDICTION
    # ============================================================

    print(
        "\n🔮 Generating predictions..."
    )


    decision_scores = model.decision_function(

        X_test
    )


    y_pred = (

        decision_scores
        >= THRESHOLD
    ).astype(
        np.int8
    )


    # ============================================================
    # EVALUATION
    # ============================================================

    results = evaluate(

        y_test,

        y_pred,

        elapsed
    )


    return (

        model,

        tfidf,

        y_test,

        y_pred,

        results
    )


# ================================================================
# RUN
# ================================================================

if __name__ == '__main__':

    model, tfidf, y_test, y_pred, results = main()


SENSITIVITY ANALYSIS: +10% HEURISTIC WEIGHTS
📂 Loading data...
   684,383 → 684,172 rows after cleaning

Train samples : 547,337
Test samples  : 136,835

🔧 Extracting features...
Sarcasm mode: +10% heuristic weights
   Extracting features...
   Fitting TF-IDF...
   Extracting features...

Feature matrix: 20,055 features
Label matrix: (547337, 13)

🚀 Training Linear SVM...
   NOTE: LinearSVC uses CPU, not the T4 GPU.

✅ SVM training completed in 118.64 minutes

🔮 Generating predictions...

UNIFORM-WEIGHT SARCASM SVM RESULTS

Training Time      : 118.64 minutes
Subset Accuracy    : 0.3164
Hamming Accuracy   : 0.8506
Micro Precision    : 0.3615
Micro Recall       : 0.8034
Micro F1           : 0.4986
Macro F1           : 0.5210
Weighted F1        : 0.5335

FINAL MICRO F1 FOR UNIFORM SARCASM
0.498621


In [ ]:
# ============================================================
# SARCASM-BASED FEATURE EXTRACTION
# Final heuristic score: s
# ============================================================

import re
import numpy as np
import pandas as pd

# ============================================================
# HEURISTIC WEIGHTS (Sensitivity Analysis)
# ============================================================

# Original weights (Baseline)
W_MC   = 2.2
W_EX   = 1.65
W_CC   = 2.2
W_PN   = 1.65
W_CAPS = 3.3
W_ID   = 2.2
W_EC   = 1.65

# Normalization constant
NORMALIZATION_FACTOR = 10.0


class SarcasmFeatures:

    # --------------------------------------------------------
    # 1. Positive sentiment words
    # --------------------------------------------------------

    _POSITIVE_WORDS = {
        'great', 'wonderful', 'amazing', 'fantastic', 'brilliant',
        'excellent', 'perfect', 'awesome', 'lovely', 'beautiful',
        'nice', 'good', 'fine', 'super', 'best', 'love', 'like',
        'happy', 'glad', 'thrilled', 'exciting', 'interesting',
        'helpful', 'useful', 'smart', 'clever', 'genius',
        'impressive', 'outstanding', 'incredible', 'fabulous'
    }

    # --------------------------------------------------------
    # 2. Negative sentiment words
    # --------------------------------------------------------

    _NEGATIVE_WORDS = {
        'hate', 'terrible', 'awful', 'horrible', 'disgusting',
        'pathetic', 'stupid', 'idiot', 'moron', 'useless',
        'worthless', 'trash', 'garbage', 'fail', 'failure',
        'worst', 'bad', 'ugly', 'dumb', 'loser', 'joke',
        'seriously', 'really', 'actually'
    }

    # --------------------------------------------------------
    # 3. Sarcasm marker phrases
    # --------------------------------------------------------

    _SARCASM_MARKERS = [
        r'\boh great\b',
        r'\byeah right\b',
        r'\bsure sure\b',
        r'\bthanks a lot\b',
        r'\bthanks so much\b',
        r'\bso helpful\b',
        r'\breally helpful\b',
        r'\bso smart\b',
        r'\bwow thanks\b',
        r'\boh wow\b',
        r'\boh really\b',
        r'\bno kidding\b',
        r'\bno way\b',
        r'\bright sure\b',
        r'\bas if\b',
        r'\bcongrats\b',
        r'\bcongratulations\b',
        r'\bwhat a surprise\b',
        r'\bshocking\b',
        r'\bunbelievable\b',
        r'\bi\'m sure\b',
        r'\bim sure\b',
        r'\bi bet\b',
        r'\bof course\b',
        r'\bobviously\b',
        r'\bclearly\b',
        r'\bjust great\b',
        r'\bjust perfect\b',
        r'\bso funny\b',
        r'\bhilarious\b',
    ]

    # --------------------------------------------------------
    # 4. Contradiction patterns
    # --------------------------------------------------------

    _CONTRADICTION_PAIRS = [
        (
            r'\b(?:great|wonderful|amazing|fantastic)\b',
            r'\b(?:not|never|no|hate|terrible)\b'
        ),
        (
            r'\b(?:love|like|enjoy)\b',
            r'\b(?:not|never|no|hate|dislike)\b'
        ),
        (
            r'\b(?:good|nice|fine)\b',
            r'\b(?:not|terrible|awful|horrible)\b'
        ),
    ]

    # --------------------------------------------------------
    # 5. Initialize regular expressions
    # --------------------------------------------------------

    def __init__(self):

        self._marker_pats = [
            re.compile(pattern, re.IGNORECASE)
            for pattern in self._SARCASM_MARKERS
        ]

        self._contra_pats = [
            (
                re.compile(positive_pattern, re.IGNORECASE),
                re.compile(negative_pattern, re.IGNORECASE)
            )
            for positive_pattern, negative_pattern
            in self._CONTRADICTION_PAIRS
        ]

        self._intensifier_re = re.compile(
            r'\b(?:so|very|extremely|incredibly|absolutely|totally|'
            r'completely|literally|seriously|really|such|beyond|super)\b',
            re.IGNORECASE
        )

        self._pos_emoji_re = re.compile(
            r'[\U0001F600-\U0001F606'
            r'\U0001F609\U0001F60A'
            r'\U0001F60D\U0001F618'
            r'\U0001F61C\U0001F61D'
            r'\U0001F923]'
        )

    # --------------------------------------------------------
    # 6. Calculate final sarcasm score s
    # --------------------------------------------------------

    def transform(self, texts):

        sarcasm_scores = []

        for text in texts:

            t = str(text)
            lower = t.lower()

            # Word count (w)
            wc = max(len(lower.split()), 1)

            # Lowercase words
            lower_words = [
                word.strip('.,!?;:\'"')
                for word in lower.split()
            ]

            # ------------------------------------------------
            # Calculate variables used in the final equation
            # ------------------------------------------------

            # m_c : sarcasm marker count
            mc = sum(
                1
                for pattern in self._marker_pats
                if pattern.search(t)
            )

            # e_x : repeated exclamation pattern count
            ex = len(
                re.findall(r'!{2,}', t)
            )

            # c_c : contradiction count
            cc = sum(
                1
                for positive_pattern, negative_pattern
                in self._contra_pats
                if positive_pattern.search(t)
                and negative_pattern.search(t)
            )

            # p_c : positive word count
            pc = sum(
                1
                for word in lower_words
                if word in self._POSITIVE_WORDS
            )

            # n_c : negative word count
            nc = sum(
                1
                for word in lower_words
                if word in self._NEGATIVE_WORDS
            )

            # Indicator function:
            # I[p_c > 0 AND n_c > 0]
            pos_neg_flag = int(
                pc > 0 and nc > 0
            )

            # r_caps : ratio of ALL-CAPS words
            caps_words = len(
                re.findall(r'\b[A-Z]{2,}\b', t)
            )

            rcaps = caps_words / wc

            # i_d : intensifier word density
            intensifier_count = len(
                self._intensifier_re.findall(t)
            )

            intensity_density = intensifier_count / wc

            # e_c : emoji-text sentiment contradiction
            ec = int(
                len(self._pos_emoji_re.findall(t)) > 0
                and nc > 0
            )

            # ------------------------------------------------
            # Final normalized heuristic sarcasm score

            # ------------------------------------------------
            s = min(
    (
        W_MC   * mc
        + W_EX   * ex
        + W_CC   * cc
        + W_PN   * pos_neg_flag
        + W_CAPS * rcaps
        + W_ID   * intensity_density
        + W_EC   * ec
    ) / NORMALIZATION_FACTOR,
    1.0
)

            # Store only final normalized score s
            sarcasm_scores.append(s)

        # ----------------------------------------------------
        # Return ONLY final normalized sarcasm score s
        # ----------------------------------------------------

        return np.array(
            sarcasm_scores,
            dtype=np.float32
        ).reshape(-1, 1)


# ============================================================
# EXECUTE SARCASM FEATURE EXTRACTION
# ============================================================

# Assumes that 'df' already exists and contains a 'text' column.

sarcasm_extractor = SarcasmFeatures()

X_sarcasm = sarcasm_extractor.transform(
    df['text']
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 60)
print("SARCASM FEATURE EXTRACTION COMPLETED")
print("=" * 60)

print("Number of text samples:", len(df))

print("Sarcasm feature shape:", X_sarcasm.shape)

print("Feature used by the model:")
print("sarc_heuristic_score")

print("\nFirst 10 normalized sarcasm scores:")

for i, score in enumerate(
    X_sarcasm[:10].flatten(),
    start=1
):
    print(f"Sample {i}: {score:.6f}")

print("\nMinimum sarcasm score:",
      X_sarcasm.min())

print("Maximum sarcasm score:",
      X_sarcasm.max())

print("Mean sarcasm score:",
      X_sarcasm.mean())

print("=" * 60)

SARCASM FEATURE EXTRACTION COMPLETED
Number of text samples: 683935
Sarcasm feature shape: (683935, 1)
Feature used by the model:
sarc_heuristic_score

First 10 normalized sarcasm scores:
Sample 1: 0.000000
Sample 2: 0.220000
Sample 3: 0.000000
Sample 4: 0.000000
Sample 5: 0.169074
Sample 6: 0.000000
Sample 7: 0.000000
Sample 8: 0.004783
Sample 9: 0.000000
Sample 10: 0.000000

Minimum sarcasm score: 0.0
Maximum sarcasm score: 1.0
Mean sarcasm score: 0.035149917


In [ ]:
# ============================================================
# SARCASM-BASED FEATURE EXTRACTION
# Final heuristic score: s
# ============================================================

import re
import numpy as np
import pandas as pd

# ============================================================
# -10% Heuristic Weights (Sensitivity Analysis)
# ============================================================

# Original weights (Baseline)
W_MC   = 1.8
W_EX   = 1.35
W_CC   = 1.8
W_PN   = 1.35
W_CAPS = 2.7
W_ID   = 1.8
W_EC   = 1.35

# Normalization constant
NORMALIZATION_FACTOR = 10.0


class SarcasmFeatures:

    # --------------------------------------------------------
    # 1. Positive sentiment words
    # --------------------------------------------------------

    _POSITIVE_WORDS = {
        'great', 'wonderful', 'amazing', 'fantastic', 'brilliant',
        'excellent', 'perfect', 'awesome', 'lovely', 'beautiful',
        'nice', 'good', 'fine', 'super', 'best', 'love', 'like',
        'happy', 'glad', 'thrilled', 'exciting', 'interesting',
        'helpful', 'useful', 'smart', 'clever', 'genius',
        'impressive', 'outstanding', 'incredible', 'fabulous'
    }

    # --------------------------------------------------------
    # 2. Negative sentiment words
    # --------------------------------------------------------

    _NEGATIVE_WORDS = {
        'hate', 'terrible', 'awful', 'horrible', 'disgusting',
        'pathetic', 'stupid', 'idiot', 'moron', 'useless',
        'worthless', 'trash', 'garbage', 'fail', 'failure',
        'worst', 'bad', 'ugly', 'dumb', 'loser', 'joke',
        'seriously', 'really', 'actually'
    }

    # --------------------------------------------------------
    # 3. Sarcasm marker phrases
    # --------------------------------------------------------

    _SARCASM_MARKERS = [
        r'\boh great\b',
        r'\byeah right\b',
        r'\bsure sure\b',
        r'\bthanks a lot\b',
        r'\bthanks so much\b',
        r'\bso helpful\b',
        r'\breally helpful\b',
        r'\bso smart\b',
        r'\bwow thanks\b',
        r'\boh wow\b',
        r'\boh really\b',
        r'\bno kidding\b',
        r'\bno way\b',
        r'\bright sure\b',
        r'\bas if\b',
        r'\bcongrats\b',
        r'\bcongratulations\b',
        r'\bwhat a surprise\b',
        r'\bshocking\b',
        r'\bunbelievable\b',
        r'\bi\'m sure\b',
        r'\bim sure\b',
        r'\bi bet\b',
        r'\bof course\b',
        r'\bobviously\b',
        r'\bclearly\b',
        r'\bjust great\b',
        r'\bjust perfect\b',
        r'\bso funny\b',
        r'\bhilarious\b',
    ]

    # --------------------------------------------------------
    # 4. Contradiction patterns
    # --------------------------------------------------------

    _CONTRADICTION_PAIRS = [
        (
            r'\b(?:great|wonderful|amazing|fantastic)\b',
            r'\b(?:not|never|no|hate|terrible)\b'
        ),
        (
            r'\b(?:love|like|enjoy)\b',
            r'\b(?:not|never|no|hate|dislike)\b'
        ),
        (
            r'\b(?:good|nice|fine)\b',
            r'\b(?:not|terrible|awful|horrible)\b'
        ),
    ]

    # --------------------------------------------------------
    # 5. Initialize regular expressions
    # --------------------------------------------------------

    def __init__(self):

        self._marker_pats = [
            re.compile(pattern, re.IGNORECASE)
            for pattern in self._SARCASM_MARKERS
        ]

        self._contra_pats = [
            (
                re.compile(positive_pattern, re.IGNORECASE),
                re.compile(negative_pattern, re.IGNORECASE)
            )
            for positive_pattern, negative_pattern
            in self._CONTRADICTION_PAIRS
        ]

        self._intensifier_re = re.compile(
            r'\b(?:so|very|extremely|incredibly|absolutely|totally|'
            r'completely|literally|seriously|really|such|beyond|super)\b',
            re.IGNORECASE
        )

        self._pos_emoji_re = re.compile(
            r'[\U0001F600-\U0001F606'
            r'\U0001F609\U0001F60A'
            r'\U0001F60D\U0001F618'
            r'\U0001F61C\U0001F61D'
            r'\U0001F923]'
        )

    # --------------------------------------------------------
    # 6. Calculate final sarcasm score s
    # --------------------------------------------------------

    def transform(self, texts):

        sarcasm_scores = []

        for text in texts:

            t = str(text)
            lower = t.lower()

            # Word count (w)
            wc = max(len(lower.split()), 1)

            # Lowercase words
            lower_words = [
                word.strip('.,!?;:\'"')
                for word in lower.split()
            ]

            # ------------------------------------------------
            # Calculate variables used in the final equation
            # ------------------------------------------------

            # m_c : sarcasm marker count
            mc = sum(
                1
                for pattern in self._marker_pats
                if pattern.search(t)
            )

            # e_x : repeated exclamation pattern count
            ex = len(
                re.findall(r'!{2,}', t)
            )

            # c_c : contradiction count
            cc = sum(
                1
                for positive_pattern, negative_pattern
                in self._contra_pats
                if positive_pattern.search(t)
                and negative_pattern.search(t)
            )

            # p_c : positive word count
            pc = sum(
                1
                for word in lower_words
                if word in self._POSITIVE_WORDS
            )

            # n_c : negative word count
            nc = sum(
                1
                for word in lower_words
                if word in self._NEGATIVE_WORDS
            )

            # Indicator function:
            # I[p_c > 0 AND n_c > 0]
            pos_neg_flag = int(
                pc > 0 and nc > 0
            )

            # r_caps : ratio of ALL-CAPS words
            caps_words = len(
                re.findall(r'\b[A-Z]{2,}\b', t)
            )

            rcaps = caps_words / wc

            # i_d : intensifier word density
            intensifier_count = len(
                self._intensifier_re.findall(t)
            )

            intensity_density = intensifier_count / wc

            # e_c : emoji-text sentiment contradiction
            ec = int(
                len(self._pos_emoji_re.findall(t)) > 0
                and nc > 0
            )

            # ------------------------------------------------
            # Final normalized heuristic sarcasm score

            # ------------------------------------------------
            s = min(
    (
        W_MC   * mc
        + W_EX   * ex
        + W_CC   * cc
        + W_PN   * pos_neg_flag
        + W_CAPS * rcaps
        + W_ID   * intensity_density
        + W_EC   * ec
    ) / NORMALIZATION_FACTOR,
    1.0
)

            # Store only final normalized score s
            sarcasm_scores.append(s)

        # ----------------------------------------------------
        # Return ONLY final normalized sarcasm score s
        # ----------------------------------------------------

        return np.array(
            sarcasm_scores,
            dtype=np.float32
        ).reshape(-1, 1)


# ============================================================
# EXECUTE SARCASM FEATURE EXTRACTION
# ============================================================

# Assumes that 'df' already exists and contains a 'text' column.

sarcasm_extractor = SarcasmFeatures()

X_sarcasm = sarcasm_extractor.transform(
    df['text']
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 60)
print("SARCASM FEATURE EXTRACTION COMPLETED")
print("=" * 60)

print("Number of text samples:", len(df))

print("Sarcasm feature shape:", X_sarcasm.shape)

print("Feature used by the model:")
print("sarc_heuristic_score")

print("\nFirst 10 normalized sarcasm scores:")

for i, score in enumerate(
    X_sarcasm[:10].flatten(),
    start=1
):
    print(f"Sample {i}: {score:.6f}")

print("\nMinimum sarcasm score:",
      X_sarcasm.min())

print("Maximum sarcasm score:",
      X_sarcasm.max())

print("Mean sarcasm score:",
      X_sarcasm.mean())

print("=" * 60)

SARCASM FEATURE EXTRACTION COMPLETED
Number of text samples: 683935
Sarcasm feature shape: (683935, 1)
Feature used by the model:
sarc_heuristic_score

First 10 normalized sarcasm scores:
Sample 1: 0.000000
Sample 2: 0.180000
Sample 3: 0.000000
Sample 4: 0.000000
Sample 5: 0.138333
Sample 6: 0.000000
Sample 7: 0.000000
Sample 8: 0.003913
Sample 9: 0.000000
Sample 10: 0.000000

Minimum sarcasm score: 0.0
Maximum sarcasm score: 1.0
Mean sarcasm score: 0.028762262


In [ ]:
# ================================================================
# UNIFORM-WEIGHT SARCASM SVM EXPERIMENT
# ================================================================



import re
import time
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    f1_score,
    precision_score,
    recall_score
)


# ================================================================
# CONFIGURATION
# ================================================================

DATA_PATH = '/content/drive/MyDrive/Cyberbullying_Project/cyberbullying.csv'

TEST_SIZE = 0.2
RANDOM_STATE = 42

# Decision threshold
# 0.0 is the default LinearSVC decision boundary.
# You can tune this later if required.
THRESHOLD = 0.0


LABEL_COLS = [
    'religious_hate',
    'ethnic_hate',
    'age_discrimination',
    'gender_hate',
    'sexual_harassment',
    'threats',
    'body_shaming',
    'political_hate',
    'trolling',
    'mental_hate',
    'discrimination',
    'other_cyberbullying_types',
    'not_cyberbullying'
]


# ================================================================
# FEATURE DEFINITIONS
# ================================================================

HATE_KEYWORDS = {

    'religious_hate': [
        'infidel','kafir','blasphemy','blasphemer','heretic',
        'cult','crusade','jihad','pagan','apostate','heathen',
        'godless','anti-christian','anti-islam','anti-hindu',
        'anti-semitic','zionist','islamophobe','christophobe'
    ],

    'ethnic_hate': [
        'nigger','nigga','chink','spic','wetback','gook','kike',
        'raghead','cracker','redneck','monkey','ape','beaner',
        'immigrant','illegal alien','foreigner','outsider',
        'invader','terrorist'
    ],

    'gender_hate': [
        'bitch','whore','slut','cunt','feminazi','tranny','dyke',
        'faggot','sissy','mangina','thot','hoe','skank','femoid',
        'incel','simp','soyboy','misogynist','misandrist'
    ],

    'sexual_harassment': [
        'rape','rapist','molest','grope','pervert','predator',
        'creep','harass','assault'
    ],

    'threats': [
        'kill','murder','shoot','stab','bomb','attack','destroy',
        'hurt','harm','threat','die','execute','eliminate',
        'exterminate','torture','burn','acid','explode','lynch',
        'hang','beat','bash','smash'
    ],

    'body_shaming': [
        'fat','ugly','obese','disgusting','gross','pig','whale',
        'skinny','anorexic','hideous','repulsive','deformed',
        'freak','monster','troll','goblin'
    ],

    'mental_hate': [
        'retard','retarded','psycho','crazy','insane','lunatic',
        'mental','schizo','bipolar','autistic','mentally ill',
        'nutcase','bonkers','deranged','demented'
    ],

    'general_hate': [
        'hate','stupid','idiot','moron','dumb','loser','trash',
        'garbage','scum','filth','worthless','pathetic','useless',
        'despise','abhor','vile','toxic','cancer','parasite',
        'vermin','subhuman'
    ]
}


POS_PATTERNS = {

    'NOUN':
        r'\b(?:man|woman|people|person|thing|way|day|time|year|'
        r'world|life|child|children|hand|part|place|case|week|'
        r'company|system|question|government|number|night|point|'
        r'home|water|room|mother|area|money|story|fact|month|lot|'
        r'right|study|book|eye|job|word|business|issue|side|kind|'
        r'head|house|service|friend|father|power|hour|game|line|'
        r'end|member|city|community|name|president|team|minute|'
        r'idea|body|information|back|parent|face|level|office|'
        r'door|health|art|war|history|party|result|change|morning|'
        r'reason|research|girl|guy|moment|air|teacher|force|education)\b',

    'VERB':
        r'\b(?:is|are|was|were|be|been|being|have|has|had|do|'
        r'does|did|will|would|could|should|may|might|shall|must|'
        r'can|go|get|make|know|think|take|see|come|want|look|use|'
        r'find|give|tell|work|call|try|ask|need|feel|become|leave|'
        r'put|mean|keep|let|begin|show|hear|play|run|move|live|'
        r'believe|hold|bring|happen|write|provide|sit|stand|lose|'
        r'pay|meet|include|continue|set|learn|change|lead|understand|'
        r'watch|follow|stop|create|speak|read|spend|grow|open|walk|'
        r'win|offer|remember|love|consider|appear|buy|wait|serve|'
        r'die|send|expect|build|stay|fall|cut|reach|kill|remain|'
        r'suggest|raise|pass|sell|require|report|decide|pull)\b',

    'ADJ':
        r'\b(?:good|new|first|last|long|great|little|own|other|'
        r'old|right|big|high|different|small|large|next|early|young|'
        r'important|public|private|real|best|free|few|same|able|'
        r'political|social|economic|national|possible|local|white|'
        r'black|strong|true|hot|happy|sad|bad|ugly|fat|stupid|evil|'
        r'dangerous|terrible|horrible|awful|disgusting|pathetic|'
        r'worthless|useless|dumb|crazy|insane|violent|racist|sexist|'
        r'offensive|hateful|toxic|radical|extreme)\b',

    'ADV':
        r'\b(?:up|so|out|just|now|how|then|more|also|here|well|'
        r'only|very|even|back|there|down|still|in|as|too|really|'
        r'most|never|much|often|always|actually|again|further|yet|'
        r'already|soon|especially|finally|simply|probably|certainly|'
        r'clearly|literally|absolutely|totally|definitely|obviously|'
        r'seriously|exactly|nearly|directly|quickly|easily)\b',

    'PRON':
        r'\b(?:i|me|my|myself|you|your|yourself|he|him|his|himself|'
        r'she|her|hers|herself|it|its|itself|we|us|our|ourselves|'
        r'they|them|their|theirs|themselves|this|that|these|those|'
        r'who|whom|which|what)\b'
}


PROFANITY = {
    'fuck','fucking','fucked','shit','ass','asshole','bitch',
    'cunt','bastard','damn','piss','cock','dick','pussy',
    'motherfucker','whore','slut','idiot','moron','loser',
    'stupid','dumb','retard'
}


# ================================================================
# SARCASM DEFINITIONS
# UNIFORM-WEIGHT VERSION
# ================================================================

SARC_MARKERS = [

    r'\boh great\b',
    r'\byeah right\b',
    r'\bsure sure\b',
    r'\bthanks a lot\b',
    r'\bthanks so much\b',
    r'\bso helpful\b',
    r'\breally helpful\b',
    r'\bso smart\b',
    r'\bwow thanks\b',
    r'\boh wow\b',
    r'\boh really\b',
    r'\bno kidding\b',
    r'\bno way\b',
    r'\bright sure\b',
    r'\bas if\b',
    r'\bcongrats\b',
    r'\bcongratulations\b',
    r'\bwhat a surprise\b',
    r'\bshocking\b',
    r'\bunbelievable\b',
    r'\bi\'m sure\b',
    r'\bim sure\b',
    r'\bi bet\b',
    r'\bof course\b',
    r'\bobviously\b',
    r'\bclearly\b',
    r'\bjust great\b',
    r'\bjust perfect\b',
    r'\bso funny\b',
    r'\bhilarious\b'
]


SARC_POS_WORDS = {
    'great','wonderful','amazing','fantastic','brilliant',
    'excellent','perfect','awesome','lovely','nice','good',
    'fine','super','best','love','happy','glad','thrilled',
    'helpful','smart','clever','impressive','incredible'
}


SARC_NEG_WORDS = {
    'hate','terrible','awful','horrible','disgusting','pathetic',
    'stupid','idiot','moron','useless','worthless','trash',
    'garbage','fail','worst','bad','ugly','dumb','loser'
}


SARC_CONTRA = [

    (
        r'\b(?:great|wonderful|amazing|fantastic)\b',
        r'\b(?:not|never|no|hate|terrible)\b'
    ),

    (
        r'\b(?:love|like|enjoy)\b',
        r'\b(?:not|never|no|hate|dislike)\b'
    ),

    (
        r'\b(?:good|nice|fine)\b',
        r'\b(?:not|terrible|awful|horrible)\b'
    )
]


_INTENSIFIER_RE = re.compile(
    r'\b(?:so|very|extremely|incredibly|absolutely|totally|'
    r'completely|literally|seriously|really|such|beyond|super)\b',
    re.IGNORECASE
)


_POS_EMOJI_RE = re.compile(
    r'[\U0001F600-\U0001F606'
    r'\U0001F609\U0001F60A'
    r'\U0001F60D\U0001F618'
    r'\U0001F61C\U0001F61D'
    r'\U0001F923]'
)


# ================================================================
# STEP 1: LOAD AND CLEAN DATA
# ================================================================

def load_and_clean():
    print("📂 Loading data...")
    df = pd.read_csv(DATA_PATH)
    before = len(df)
    df = df[
        df['text'].notnull()
        & (df['text'].astype(str).str.strip() != '')
    ]
    df = df[
        df['text'].astype(str).str.contains(
            r'[a-zA-Z]',
            regex=True
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^@\w+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^\d+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^[^\w\s]+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^(.)\1+$|^(\w+)(\s+\2)+\s*$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^(#\w+\s*)+$'
        )
    ]

    df = df.reset_index(drop=True)

    print(
        f"   {before:,} → {len(df):,} rows after cleaning"
    )

    return df



# ============================================================
# HEURISTIC WEIGHTS (-10% Sensitivity Analysis)
# ============================================================

# ============================================================
# HEURISTIC WEIGHTS (-10% Sensitivity Analysis)
# ============================================================

W_MC   = 1.8    # Sarcasm marker count
W_EX   = 1.35   # Repeated exclamation pattern count
W_CC   = 1.8    # Contextual contradiction count
W_PN   = 1.35   # Positive-negative sentiment incongruity
W_CAPS = 2.7    # ALL-CAPS ratio
W_ID   = 1.8    # Intensifier density
W_EC   = 1.35   # Emoji-text sentiment contradiction

# ================================================================
# STEP 2: FEATURE EXTRACTION
# ================================================================


NORMALIZATION_FACTOR = 10.0
def extract_features(
    texts,
    tfidf=None,
    fit=False
):

    print("   Extracting features...")

    # ------------------------------------------------------------
    # Compile patterns once
    # ------------------------------------------------------------

    hate_pats = {

        cat: re.compile(
            r'\b(?:'
            + '|'.join(
                re.escape(k)
                for k in kws
            )
            + r')\b',
            re.IGNORECASE
        )

        for cat, kws in HATE_KEYWORDS.items()
    }


    pos_pats = {

        pos: re.compile(
            pat,
            re.IGNORECASE
        )

        for pos, pat in POS_PATTERNS.items()
    }


    marker_pats = [

        re.compile(
            p,
            re.IGNORECASE
        )

        for p in SARC_MARKERS
    ]


    contra_pats = [

        (
            re.compile(
                a,
                re.IGNORECASE
            ),

            re.compile(
                b,
                re.IGNORECASE
            )
        )

        for a, b in SARC_CONTRA
    ]


    url_re = re.compile(
        r'https?://\S+|www\.\S+'
    )

    mention_re = re.compile(
        r'@\w+'
    )

    hashtag_re = re.compile(
        r'#\w+'
    )

    emoji_re = re.compile(
        r'[\U0001F300-\U0001F9FF'
        r'\U00002600-\U000027FF]',
        flags=re.UNICODE
    )

    repeat_re = re.compile(
        r'(.)\1{2,}'
    )

    caps_re = re.compile(
        r'\b[A-Z]{2,}\b'
    )


    all_rows = []


    # ============================================================
    # PROCESS EACH TEXT
    # ============================================================

    for text in texts:

        t = str(text)

        words = t.split()

        wc = max(
            len(words),
            1
        )

        cc = max(
            len(t),
            1
        )


        lw = [

            w.lower().strip(
                '.,!?;:\'"'
            )

            for w in words
        ]


        sents = max(

            len(
                re.split(
                    r'[.!?]+',
                    t
                )
            ),

            1
        )


        # --------------------------------------------------------
        # Word frequency
        # --------------------------------------------------------

        wfreq = {}

        for w in lw:

            wfreq[w] = (
                wfreq.get(
                    w,
                    0
                )
                + 1
            )


        caps = caps_re.findall(t)


        prof = sum(

            1

            for w in lw

            if w in PROFANITY
        )


        row = {}


        # ========================================================
        # HATE KEYWORD FEATURES
        # ========================================================

        total_hate = 0


        for cat, pat in hate_pats.items():

            c = len(
                pat.findall(t)
            )


            row[
                f'hate_count_{cat}'
            ] = c


            row[
                f'hate_flag_{cat}'
            ] = int(
                c > 0
            )


            total_hate += c


        row[
            'hate_total_count'
        ] = total_hate


        row[
            'hate_density'
        ] = total_hate / wc


        row[
            'hate_category_count'
        ] = sum(

            1

            for cat in HATE_KEYWORDS

            if row[
                f'hate_flag_{cat}'
            ] > 0
        )


        # ========================================================
        # POS FEATURES
        # ========================================================

        pcounts = {}


        for pos, pat in pos_pats.items():

            c = len(
                pat.findall(t)
            )


            pcounts[pos] = c


            row[
                f'pos_count_{pos.lower()}'
            ] = c


            row[
                f'pos_ratio_{pos.lower()}'
            ] = c / wc


        row[
            'pos_adj_noun_ratio'
        ] = (

            pcounts.get(
                'ADJ',
                0
            )

            /

            max(
                pcounts.get(
                    'NOUN',
                    0
                ),
                1
            )
        )


        row[
            'pos_verb_density'
        ] = (

            pcounts.get(
                'VERB',
                0
            )

            / wc
        )


        row[
            'pos_pron_density'
        ] = (

            pcounts.get(
                'PRON',
                0
            )

            / wc
        )


        row[
            'pos_modifier_density'
        ] = (

            pcounts.get(
                'ADJ',
                0
            )

            +

            pcounts.get(
                'ADV',
                0
            )

        ) / wc


        # ========================================================
        # WORD FREQUENCY FEATURES
        # ========================================================

        row.update({

            'wf_char_count':
                cc,

            'wf_word_count':
                wc,

            'wf_sentence_count':
                sents,

            'wf_avg_word_len':
                np.mean(
                    [
                        len(w)
                        for w in words
                    ]
                )
                if words
                else 0,

            'wf_avg_sent_len':
                wc / sents,

            'wf_unique_word_ratio':
                len(
                    set(lw)
                ) / wc,

            'wf_caps_word_count':
                len(caps),

            'wf_caps_ratio':
                len(caps) / wc,

            'wf_upper_char_ratio':
                sum(
                    1
                    for c in t
                    if c.isupper()
                ) / cc,

            'wf_exclaim_count':
                t.count('!'),

            'wf_question_count':
                t.count('?'),

            'wf_ellipsis_count':
                t.count('...'),

            'wf_punct_density':
                sum(
                    1
                    for c in t
                    if c in '!?.,;:\'"'
                ) / cc,

            'wf_url_count':
                len(
                    url_re.findall(t)
                ),

            'wf_mention_count':
                len(
                    mention_re.findall(t)
                ),

            'wf_hashtag_count':
                len(
                    hashtag_re.findall(t)
                ),

            'wf_emoji_count':
                len(
                    emoji_re.findall(t)
                ),

            'wf_repeat_char_count':
                len(
                    repeat_re.findall(t)
                ),

            'wf_repeat_word_count':
                sum(

                    1

                    for v in wfreq.values()

                    if v > 1
                ),

            'wf_profanity_count':
                prof,

            'wf_profanity_density':
                prof / wc
        })


        # ========================================================
        # UNIFORM-WEIGHT SARCASM SCORE
        # ========================================================

        multi_exc = len(
            re.findall(
                r'!{2,}',
                t
            )
        )


        mc = sum(

            1

            for p in marker_pats

            if p.search(t)
        )


        pc = sum(

            1

            for w in lw

            if w in SARC_POS_WORDS
        )


        nc = sum(

            1

            for w in lw

            if w in SARC_NEG_WORDS
        )


        cc2 = sum(

            1

            for a_p, b_p in contra_pats

            if (
                a_p.search(t)
                and
                b_p.search(t)
            )
        )


        id_ = (

            len(
                _INTENSIFIER_RE.findall(t)
            )

            / wc
        )


        ec = int(

            len(
                _POS_EMOJI_RE.findall(t)
            ) > 0

            and

            nc > 0
        )

        # --------------------------------------------------------
        # UNIFORM WEIGHTS
        # --------------------------------------------------------

        # --------------------------------------------------------
        # WEIGHTED SARCASM SCORE (Sensitivity Analysis)
        # --------------------------------------------------------

        sarcasm_score = min(
        (
          W_MC * mc
          + W_EX * multi_exc
          + W_CC * cc2
          + W_PN * int(
            pc > 0
            and
            nc > 0
          )
          + W_CAPS * (len(caps) / wc)
          + W_ID * id_
          + W_EC * ec
        ) / NORMALIZATION_FACTOR,
        1.0
        )





        # --------------------------------------------------------
        # ONLY FINAL SARCASM SCORE IS ADDED
        # --------------------------------------------------------

        row[
            'sarc_heuristic_score'
        ] = sarcasm_score


        all_rows.append(row)


    # ============================================================
    # CONVERT DENSE FEATURES
    # ============================================================

    dense = pd.DataFrame(
        all_rows
    ).values.astype(
        np.float32
    )


    dense = np.clip(
        dense,
        0,
        None
    )


    # ============================================================
    # TF-IDF
    # ============================================================

    if fit:

        print(
            "   Fitting TF-IDF..."
        )


        tfidf = TfidfVectorizer(

            max_features=20000,

            ngram_range=(1, 2),

            min_df=3,

            sublinear_tf=True
        )


        tfidf_mat = tfidf.fit_transform(

            texts.astype(str)
        )


    else:

        tfidf_mat = tfidf.transform(

            texts.astype(str)
        )


    # ============================================================
    # COMBINE FEATURES
    # ============================================================

    X = hstack(

        [

            csr_matrix(
                dense
            ),

            tfidf_mat

        ],

        format='csr'
    )


    return X, tfidf


# ================================================================
# EVALUATION
# ================================================================

def evaluate(
    y_true,
    y_pred,
    elapsed
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        "UNIFORM-WEIGHT SARCASM SVM RESULTS"
    )

    print(
        "=" * 70
    )


    # ------------------------------------------------------------
    # Accuracy
    # ------------------------------------------------------------

    subset_accuracy = accuracy_score(

        y_true,
        y_pred
    )


    hamming_accuracy = (

        1

        -

        hamming_loss(

            y_true,
            y_pred
        )
    )


    # ------------------------------------------------------------
    # F1
    # ------------------------------------------------------------

    micro_f1 = f1_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    macro_f1 = f1_score(

        y_true,
        y_pred,

        average='macro',

        zero_division=0
    )


    weighted_f1 = f1_score(

        y_true,
        y_pred,

        average='weighted',

        zero_division=0
    )


    # ------------------------------------------------------------
    # Precision / Recall
    # ------------------------------------------------------------

    micro_precision = precision_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    micro_recall = recall_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    print(
        f"\nTraining Time      : "
        f"{elapsed / 60:.2f} minutes"
    )

    print(
        f"Subset Accuracy    : "
        f"{subset_accuracy:.4f}"
    )

    print(
        f"Hamming Accuracy   : "
        f"{hamming_accuracy:.4f}"
    )

    print(
        f"Micro Precision    : "
        f"{micro_precision:.4f}"
    )

    print(
        f"Micro Recall       : "
        f"{micro_recall:.4f}"
    )

    print(
        f"Micro F1           : "
        f"{micro_f1:.4f}"
    )

    print(
        f"Macro F1           : "
        f"{macro_f1:.4f}"
    )

    print(
        f"Weighted F1        : "
        f"{weighted_f1:.4f}"
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        "FINAL MICRO F1 FOR UNIFORM SARCASM"
    )

    print(
        f"{micro_f1:.6f}"
    )

    print(
        "=" * 70
    )


    return {

        'subset_accuracy':
            subset_accuracy,

        'hamming_accuracy':
            hamming_accuracy,

        'micro_precision':
            micro_precision,

        'micro_recall':
            micro_recall,

        'micro_f1':
            micro_f1,

        'macro_f1':
            macro_f1,

        'weighted_f1':
            weighted_f1
    }


# ================================================================
# MAIN
# ================================================================

def main():

    print(
        "\n"
        + "=" * 70
    )

    print("SENSITIVITY ANALYSIS: -10% HEURISTIC WEIGHTS")


    print(
        "=" * 70
    )


    # ============================================================
    # LOAD DATA
    # ============================================================

    df = load_and_clean()


    # ============================================================
    # TRAIN-TEST SPLIT
    # ============================================================

    df_train, df_test = train_test_split(

        df,

        test_size=TEST_SIZE,

        random_state=RANDOM_STATE
    )


    df_train = df_train.reset_index(
        drop=True
    )


    df_test = df_test.reset_index(
        drop=True
    )


    print(
        f"\nTrain samples : "
        f"{len(df_train):,}"
    )


    print(
        f"Test samples  : "
        f"{len(df_test):,}"
    )


    # ============================================================
    # FEATURE EXTRACTION
    # ============================================================

    print(
        "\n🔧 Extracting features..."
    )


    print("Sarcasm mode: +10% heuristic weights")


    X_train, tfidf = extract_features(

        df_train['text'],

        fit=True
    )


    X_test, _ = extract_features(

        df_test['text'],

        tfidf=tfidf,

        fit=False
    )


    print(
        f"\nFeature matrix: "
        f"{X_train.shape[1]:,} features"
    )


    # ============================================================
    # LABEL MATRICES
    # ============================================================

    y_train = df_train[
        LABEL_COLS
    ].values.astype(
        np.int8
    )


    y_test = df_test[
        LABEL_COLS
    ].values.astype(
        np.int8
    )


    print(
        f"Label matrix: "
        f"{y_train.shape}"
    )


    # ============================================================
    # TRAIN LINEAR SVM
    # ============================================================

    print(
        "\n🚀 Training Linear SVM..."
    )


    print(
        "   NOTE: LinearSVC uses CPU, "
        "not the T4 GPU."
    )


    t0 = time.time()


    base_svm = LinearSVC(

        C=0.5,

        max_iter=2000,

        class_weight='balanced',

        dual=False
    )


    model = OneVsRestClassifier(

        base_svm,

        n_jobs=-1
    )


    model.fit(

        X_train,

        y_train
    )


    elapsed = (

        time.time()

        -

        t0
    )


    print(
        f"\n✅ SVM training completed "
        f"in {elapsed / 60:.2f} minutes"
    )


    # ============================================================
    # PREDICTION
    # ============================================================

    print(
        "\n🔮 Generating predictions..."
    )


    decision_scores = model.decision_function(

        X_test
    )


    y_pred = (

        decision_scores
        >= THRESHOLD
    ).astype(
        np.int8
    )


    # ============================================================
    # EVALUATION
    # ============================================================

    results = evaluate(

        y_test,

        y_pred,

        elapsed
    )


    return (

        model,

        tfidf,

        y_test,

        y_pred,

        results
    )


# ================================================================
# RUN
# ================================================================

if __name__ == '__main__':

    model, tfidf, y_test, y_pred, results = main()


SENSITIVITY ANALYSIS: -10% HEURISTIC WEIGHTS
📂 Loading data...
   684,383 → 684,172 rows after cleaning

Train samples : 547,337
Test samples  : 136,835

🔧 Extracting features...
Sarcasm mode: +10% heuristic weights
   Extracting features...
   Fitting TF-IDF...
   Extracting features...

Feature matrix: 20,055 features
Label matrix: (547337, 13)

🚀 Training Linear SVM...
   NOTE: LinearSVC uses CPU, not the T4 GPU.

✅ SVM training completed in 133.49 minutes

🔮 Generating predictions...

UNIFORM-WEIGHT SARCASM SVM RESULTS

Training Time      : 133.49 minutes
Subset Accuracy    : 0.3163
Hamming Accuracy   : 0.8506
Micro Precision    : 0.3614
Micro Recall       : 0.8034
Micro F1           : 0.4985
Macro F1           : 0.5209
Weighted F1        : 0.5334

FINAL MICRO F1 FOR UNIFORM SARCASM
0.498541


In [ ]:
# ================================================================
# ORIGINAL-WEIGHT SARCASM SVM EXPERIMENT
# ================================================================



import re
import time
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    f1_score,
    precision_score,
    recall_score
)


# ================================================================
# CONFIGURATION
# ================================================================

DATA_PATH = '/content/drive/MyDrive/Cyberbullying_Project/cyberbullying.csv'

TEST_SIZE = 0.2
RANDOM_STATE = 42

# Decision threshold
# 0.0 is the default LinearSVC decision boundary.
# You can tune this later if required.
THRESHOLD = 0.0


LABEL_COLS = [
    'religious_hate',
    'ethnic_hate',
    'age_discrimination',
    'gender_hate',
    'sexual_harassment',
    'threats',
    'body_shaming',
    'political_hate',
    'trolling',
    'mental_hate',
    'discrimination',
    'other_cyberbullying_types',
    'not_cyberbullying'
]


# ================================================================
# FEATURE DEFINITIONS
# ================================================================

HATE_KEYWORDS = {

    'religious_hate': [
        'infidel','kafir','blasphemy','blasphemer','heretic',
        'cult','crusade','jihad','pagan','apostate','heathen',
        'godless','anti-christian','anti-islam','anti-hindu',
        'anti-semitic','zionist','islamophobe','christophobe'
    ],

    'ethnic_hate': [
        'nigger','nigga','chink','spic','wetback','gook','kike',
        'raghead','cracker','redneck','monkey','ape','beaner',
        'immigrant','illegal alien','foreigner','outsider',
        'invader','terrorist'
    ],

    'gender_hate': [
        'bitch','whore','slut','cunt','feminazi','tranny','dyke',
        'faggot','sissy','mangina','thot','hoe','skank','femoid',
        'incel','simp','soyboy','misogynist','misandrist'
    ],

    'sexual_harassment': [
        'rape','rapist','molest','grope','pervert','predator',
        'creep','harass','assault'
    ],

    'threats': [
        'kill','murder','shoot','stab','bomb','attack','destroy',
        'hurt','harm','threat','die','execute','eliminate',
        'exterminate','torture','burn','acid','explode','lynch',
        'hang','beat','bash','smash'
    ],

    'body_shaming': [
        'fat','ugly','obese','disgusting','gross','pig','whale',
        'skinny','anorexic','hideous','repulsive','deformed',
        'freak','monster','troll','goblin'
    ],

    'mental_hate': [
        'retard','retarded','psycho','crazy','insane','lunatic',
        'mental','schizo','bipolar','autistic','mentally ill',
        'nutcase','bonkers','deranged','demented'
    ],

    'general_hate': [
        'hate','stupid','idiot','moron','dumb','loser','trash',
        'garbage','scum','filth','worthless','pathetic','useless',
        'despise','abhor','vile','toxic','cancer','parasite',
        'vermin','subhuman'
    ]
}


POS_PATTERNS = {

    'NOUN':
        r'\b(?:man|woman|people|person|thing|way|day|time|year|'
        r'world|life|child|children|hand|part|place|case|week|'
        r'company|system|question|government|number|night|point|'
        r'home|water|room|mother|area|money|story|fact|month|lot|'
        r'right|study|book|eye|job|word|business|issue|side|kind|'
        r'head|house|service|friend|father|power|hour|game|line|'
        r'end|member|city|community|name|president|team|minute|'
        r'idea|body|information|back|parent|face|level|office|'
        r'door|health|art|war|history|party|result|change|morning|'
        r'reason|research|girl|guy|moment|air|teacher|force|education)\b',

    'VERB':
        r'\b(?:is|are|was|were|be|been|being|have|has|had|do|'
        r'does|did|will|would|could|should|may|might|shall|must|'
        r'can|go|get|make|know|think|take|see|come|want|look|use|'
        r'find|give|tell|work|call|try|ask|need|feel|become|leave|'
        r'put|mean|keep|let|begin|show|hear|play|run|move|live|'
        r'believe|hold|bring|happen|write|provide|sit|stand|lose|'
        r'pay|meet|include|continue|set|learn|change|lead|understand|'
        r'watch|follow|stop|create|speak|read|spend|grow|open|walk|'
        r'win|offer|remember|love|consider|appear|buy|wait|serve|'
        r'die|send|expect|build|stay|fall|cut|reach|kill|remain|'
        r'suggest|raise|pass|sell|require|report|decide|pull)\b',

    'ADJ':
        r'\b(?:good|new|first|last|long|great|little|own|other|'
        r'old|right|big|high|different|small|large|next|early|young|'
        r'important|public|private|real|best|free|few|same|able|'
        r'political|social|economic|national|possible|local|white|'
        r'black|strong|true|hot|happy|sad|bad|ugly|fat|stupid|evil|'
        r'dangerous|terrible|horrible|awful|disgusting|pathetic|'
        r'worthless|useless|dumb|crazy|insane|violent|racist|sexist|'
        r'offensive|hateful|toxic|radical|extreme)\b',

    'ADV':
        r'\b(?:up|so|out|just|now|how|then|more|also|here|well|'
        r'only|very|even|back|there|down|still|in|as|too|really|'
        r'most|never|much|often|always|actually|again|further|yet|'
        r'already|soon|especially|finally|simply|probably|certainly|'
        r'clearly|literally|absolutely|totally|definitely|obviously|'
        r'seriously|exactly|nearly|directly|quickly|easily)\b',

    'PRON':
        r'\b(?:i|me|my|myself|you|your|yourself|he|him|his|himself|'
        r'she|her|hers|herself|it|its|itself|we|us|our|ourselves|'
        r'they|them|their|theirs|themselves|this|that|these|those|'
        r'who|whom|which|what)\b'
}


PROFANITY = {
    'fuck','fucking','fucked','shit','ass','asshole','bitch',
    'cunt','bastard','damn','piss','cock','dick','pussy',
    'motherfucker','whore','slut','idiot','moron','loser',
    'stupid','dumb','retard'
}


# ================================================================
# SARCASM DEFINITIONS
# ORIGINAL-WEIGHT VERSION
# ================================================================

SARC_MARKERS = [

    r'\boh great\b',
    r'\byeah right\b',
    r'\bsure sure\b',
    r'\bthanks a lot\b',
    r'\bthanks so much\b',
    r'\bso helpful\b',
    r'\breally helpful\b',
    r'\bso smart\b',
    r'\bwow thanks\b',
    r'\boh wow\b',
    r'\boh really\b',
    r'\bno kidding\b',
    r'\bno way\b',
    r'\bright sure\b',
    r'\bas if\b',
    r'\bcongrats\b',
    r'\bcongratulations\b',
    r'\bwhat a surprise\b',
    r'\bshocking\b',
    r'\bunbelievable\b',
    r'\bi\'m sure\b',
    r'\bim sure\b',
    r'\bi bet\b',
    r'\bof course\b',
    r'\bobviously\b',
    r'\bclearly\b',
    r'\bjust great\b',
    r'\bjust perfect\b',
    r'\bso funny\b',
    r'\bhilarious\b'
]


SARC_POS_WORDS = {
    'great','wonderful','amazing','fantastic','brilliant',
    'excellent','perfect','awesome','lovely','nice','good',
    'fine','super','best','love','happy','glad','thrilled',
    'helpful','smart','clever','impressive','incredible'
}


SARC_NEG_WORDS = {
    'hate','terrible','awful','horrible','disgusting','pathetic',
    'stupid','idiot','moron','useless','worthless','trash',
    'garbage','fail','worst','bad','ugly','dumb','loser'
}


SARC_CONTRA = [

    (
        r'\b(?:great|wonderful|amazing|fantastic)\b',
        r'\b(?:not|never|no|hate|terrible)\b'
    ),

    (
        r'\b(?:love|like|enjoy)\b',
        r'\b(?:not|never|no|hate|dislike)\b'
    ),

    (
        r'\b(?:good|nice|fine)\b',
        r'\b(?:not|terrible|awful|horrible)\b'
    )
]


_INTENSIFIER_RE = re.compile(
    r'\b(?:so|very|extremely|incredibly|absolutely|totally|'
    r'completely|literally|seriously|really|such|beyond|super)\b',
    re.IGNORECASE
)


_POS_EMOJI_RE = re.compile(
    r'[\U0001F600-\U0001F606'
    r'\U0001F609\U0001F60A'
    r'\U0001F60D\U0001F618'
    r'\U0001F61C\U0001F61D'
    r'\U0001F923]'
)


# ================================================================
# STEP 1: LOAD AND CLEAN DATA
# ================================================================

def load_and_clean():
    print("📂 Loading data...")
    df = pd.read_csv(DATA_PATH)
    before = len(df)
    df = df[
        df['text'].notnull()
        & (df['text'].astype(str).str.strip() != '')
    ]
    df = df[
        df['text'].astype(str).str.contains(
            r'[a-zA-Z]',
            regex=True
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^@\w+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^\d+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^[^\w\s]+$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^(.)\1+$|^(\w+)(\s+\2)+\s*$'
        )
    ]

    df = df[
        ~df['text'].fillna('').str.strip().str.match(
            r'^(#\w+\s*)+$'
        )
    ]

    df = df.reset_index(drop=True)

    print(
        f"   {before:,} → {len(df):,} rows after cleaning"
    )

    return df
# ============================================================
# HEURISTIC WEIGHTS
# ============================================================

W_MC   = 2.0
W_EX   = 1.5
W_CC   = 2.0
W_PN   = 1.5
W_CAPS = 3.0
W_ID   = 2.0
W_EC   = 1.5

NORMALIZATION_FACTOR = 10.0

# ================================================================
# STEP 2: FEATURE EXTRACTION
# ================================================================
def extract_features(
    texts,
    tfidf=None,
    fit=False
):

    print("   Extracting features...")

    # ------------------------------------------------------------
    # Compile patterns once
    # ------------------------------------------------------------

    hate_pats = {

        cat: re.compile(
            r'\b(?:'
            + '|'.join(
                re.escape(k)
                for k in kws
            )
            + r')\b',
            re.IGNORECASE
        )

        for cat, kws in HATE_KEYWORDS.items()
    }


    pos_pats = {

        pos: re.compile(
            pat,
            re.IGNORECASE
        )

        for pos, pat in POS_PATTERNS.items()
    }


    marker_pats = [

        re.compile(
            p,
            re.IGNORECASE
        )

        for p in SARC_MARKERS
    ]


    contra_pats = [

        (
            re.compile(
                a,
                re.IGNORECASE
            ),

            re.compile(
                b,
                re.IGNORECASE
            )
        )

        for a, b in SARC_CONTRA
    ]


    url_re = re.compile(
        r'https?://\S+|www\.\S+'
    )

    mention_re = re.compile(
        r'@\w+'
    )

    hashtag_re = re.compile(
        r'#\w+'
    )

    emoji_re = re.compile(
        r'[\U0001F300-\U0001F9FF'
        r'\U00002600-\U000027FF]',
        flags=re.UNICODE
    )

    repeat_re = re.compile(
        r'(.)\1{2,}'
    )

    caps_re = re.compile(
        r'\b[A-Z]{2,}\b'
    )


    all_rows = []


    # ============================================================
    # PROCESS EACH TEXT
    # ============================================================

    for text in texts:

        t = str(text)

        words = t.split()

        wc = max(
            len(words),
            1
        )

        cc = max(
            len(t),
            1
        )


        lw = [

            w.lower().strip(
                '.,!?;:\'"'
            )

            for w in words
        ]


        sents = max(

            len(
                re.split(
                    r'[.!?]+',
                    t
                )
            ),

            1
        )


        # --------------------------------------------------------
        # Word frequency
        # --------------------------------------------------------

        wfreq = {}

        for w in lw:

            wfreq[w] = (
                wfreq.get(
                    w,
                    0
                )
                + 1
            )


        caps = caps_re.findall(t)


        prof = sum(

            1

            for w in lw

            if w in PROFANITY
        )


        row = {}


        # ========================================================
        # HATE KEYWORD FEATURES
        # ========================================================

        total_hate = 0


        for cat, pat in hate_pats.items():

            c = len(
                pat.findall(t)
            )


            row[
                f'hate_count_{cat}'
            ] = c


            row[
                f'hate_flag_{cat}'
            ] = int(
                c > 0
            )


            total_hate += c


        row[
            'hate_total_count'
        ] = total_hate


        row[
            'hate_density'
        ] = total_hate / wc


        row[
            'hate_category_count'
        ] = sum(

            1

            for cat in HATE_KEYWORDS

            if row[
                f'hate_flag_{cat}'
            ] > 0
        )


        # ========================================================
        # POS FEATURES
        # ========================================================

        pcounts = {}


        for pos, pat in pos_pats.items():

            c = len(
                pat.findall(t)
            )


            pcounts[pos] = c


            row[
                f'pos_count_{pos.lower()}'
            ] = c


            row[
                f'pos_ratio_{pos.lower()}'
            ] = c / wc


        row[
            'pos_adj_noun_ratio'
        ] = (

            pcounts.get(
                'ADJ',
                0
            )

            /

            max(
                pcounts.get(
                    'NOUN',
                    0
                ),
                1
            )
        )


        row[
            'pos_verb_density'
        ] = (

            pcounts.get(
                'VERB',
                0
            )

            / wc
        )


        row[
            'pos_pron_density'
        ] = (

            pcounts.get(
                'PRON',
                0
            )

            / wc
        )


        row[
            'pos_modifier_density'
        ] = (

            pcounts.get(
                'ADJ',
                0
            )

            +

            pcounts.get(
                'ADV',
                0
            )

        ) / wc


        # ========================================================
        # WORD FREQUENCY FEATURES
        # ========================================================

        row.update({

            'wf_char_count':
                cc,

            'wf_word_count':
                wc,

            'wf_sentence_count':
                sents,

            'wf_avg_word_len':
                np.mean(
                    [
                        len(w)
                        for w in words
                    ]
                )
                if words
                else 0,

            'wf_avg_sent_len':
                wc / sents,

            'wf_unique_word_ratio':
                len(
                    set(lw)
                ) / wc,

            'wf_caps_word_count':
                len(caps),

            'wf_caps_ratio':
                len(caps) / wc,

            'wf_upper_char_ratio':
                sum(
                    1
                    for c in t
                    if c.isupper()
                ) / cc,

            'wf_exclaim_count':
                t.count('!'),

            'wf_question_count':
                t.count('?'),

            'wf_ellipsis_count':
                t.count('...'),

            'wf_punct_density':
                sum(
                    1
                    for c in t
                    if c in '!?.,;:\'"'
                ) / cc,

            'wf_url_count':
                len(
                    url_re.findall(t)
                ),

            'wf_mention_count':
                len(
                    mention_re.findall(t)
                ),

            'wf_hashtag_count':
                len(
                    hashtag_re.findall(t)
                ),

            'wf_emoji_count':
                len(
                    emoji_re.findall(t)
                ),

            'wf_repeat_char_count':
                len(
                    repeat_re.findall(t)
                ),

            'wf_repeat_word_count':
                sum(

                    1

                    for v in wfreq.values()

                    if v > 1
                ),

            'wf_profanity_count':
                prof,

            'wf_profanity_density':
                prof / wc
        })


        # ========================================================
        # ORIGINAL-WEIGHT SARCASM SCORE
        # ========================================================

        multi_exc = len(
            re.findall(
                r'!{2,}',
                t
            )
        )


        mc = sum(

            1

            for p in marker_pats

            if p.search(t)
        )


        pc = sum(

            1

            for w in lw

            if w in SARC_POS_WORDS
        )


        nc = sum(

            1

            for w in lw

            if w in SARC_NEG_WORDS
        )


        cc2 = sum(

            1

            for a_p, b_p in contra_pats

            if (
                a_p.search(t)
                and
                b_p.search(t)
            )
        )


        id_ = (

            len(
                _INTENSIFIER_RE.findall(t)
            )

            / wc
        )


        ec = int(

            len(
                _POS_EMOJI_RE.findall(t)
            ) > 0

            and

            nc > 0
        )

        # --------------------------------------------------------
        # UNIFORM WEIGHTS
        # --------------------------------------------------------

        # --------------------------------------------------------
        # WEIGHTED SARCASM SCORE (Sensitivity Analysis)
        # --------------------------------------------------------

        sarcasm_score = min(
        (
          W_MC * mc
          + W_EX * multi_exc
          + W_CC * cc2
          + W_PN * int(
            pc > 0
            and
            nc > 0
          )
          + W_CAPS * (len(caps) / wc)
          + W_ID * id_
          + W_EC * ec
        ) / NORMALIZATION_FACTOR,
        1.0
        )





        # --------------------------------------------------------
        # ONLY FINAL SARCASM SCORE IS ADDED
        # --------------------------------------------------------

        row[
            'sarc_heuristic_score'
        ] = sarcasm_score


        all_rows.append(row)


    # ============================================================
    # CONVERT DENSE FEATURES
    # ============================================================

    dense = pd.DataFrame(
        all_rows
    ).values.astype(
        np.float32
    )


    dense = np.clip(
        dense,
        0,
        None
    )


    # ============================================================
    # TF-IDF
    # ============================================================

    if fit:

        print(
            "   Fitting TF-IDF..."
        )


        tfidf = TfidfVectorizer(

            max_features=20000,

            ngram_range=(1, 2),

            min_df=3,

            sublinear_tf=True
        )


        tfidf_mat = tfidf.fit_transform(

            texts.astype(str)
        )


    else:

        tfidf_mat = tfidf.transform(

            texts.astype(str)
        )


    # ============================================================
    # COMBINE FEATURES
    # ============================================================

    X = hstack(

        [

            csr_matrix(
                dense
            ),

            tfidf_mat

        ],

        format='csr'
    )


    return X, tfidf


# ================================================================
# EVALUATION
# ================================================================

def evaluate(
    y_true,
    y_pred,
    elapsed
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        "ORIGINAL-WEIGHT SARCASM SVM RESULTS"
    )

    print(
        "=" * 70
    )


    # ------------------------------------------------------------
    # Accuracy
    # ------------------------------------------------------------

    subset_accuracy = accuracy_score(

        y_true,
        y_pred
    )


    hamming_accuracy = (

        1

        -

        hamming_loss(

            y_true,
            y_pred
        )
    )


    # ------------------------------------------------------------
    # F1
    # ------------------------------------------------------------

    micro_f1 = f1_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    macro_f1 = f1_score(

        y_true,
        y_pred,

        average='macro',

        zero_division=0
    )


    weighted_f1 = f1_score(

        y_true,
        y_pred,

        average='weighted',

        zero_division=0
    )


    # ------------------------------------------------------------
    # Precision / Recall
    # ------------------------------------------------------------

    micro_precision = precision_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    micro_recall = recall_score(

        y_true,
        y_pred,

        average='micro',

        zero_division=0
    )


    print(
        f"\nTraining Time      : "
        f"{elapsed / 60:.2f} minutes"
    )

    print(
        f"Subset Accuracy    : "
        f"{subset_accuracy:.4f}"
    )

    print(
        f"Hamming Accuracy   : "
        f"{hamming_accuracy:.4f}"
    )

    print(
        f"Micro Precision    : "
        f"{micro_precision:.4f}"
    )

    print(
        f"Micro Recall       : "
        f"{micro_recall:.4f}"
    )

    print(
        f"Micro F1           : "
        f"{micro_f1:.4f}"
    )

    print(
        f"Macro F1           : "
        f"{macro_f1:.4f}"
    )

    print(
        f"Weighted F1        : "
        f"{weighted_f1:.4f}"
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        "FINAL MICRO F1 FOR ORIGINAL WEIGHT SARCASM"
    )

    print(
        f"{micro_f1:.6f}"
    )

    print(
        "=" * 70
    )


    return {

        'subset_accuracy':
            subset_accuracy,

        'hamming_accuracy':
            hamming_accuracy,

        'micro_precision':
            micro_precision,

        'micro_recall':
            micro_recall,

        'micro_f1':
            micro_f1,

        'macro_f1':
            macro_f1,

        'weighted_f1':
            weighted_f1
    }


# ================================================================
# MAIN
# ================================================================

def main():

    print(
        "\n"
        + "=" * 70
    )

    print("SENSITIVITY ANALYSIS: ORIGINAL HEURISTIC WEIGHTS")


    print(
        "=" * 70
    )


    # ============================================================
    # LOAD DATA
    # ============================================================

    df = load_and_clean()


    # ============================================================
    # TRAIN-TEST SPLIT
    # ============================================================

    df_train, df_test = train_test_split(

        df,

        test_size=TEST_SIZE,

        random_state=RANDOM_STATE
    )


    df_train = df_train.reset_index(
        drop=True
    )


    df_test = df_test.reset_index(
        drop=True
    )


    print(
        f"\nTrain samples : "
        f"{len(df_train):,}"
    )


    print(
        f"Test samples  : "
        f"{len(df_test):,}"
    )


    # ============================================================
    # FEATURE EXTRACTION
    # ============================================================

    print(
        "\n🔧 Extracting features..."
    )


    print("Sarcasm mode: Original heuristic weights")


    X_train, tfidf = extract_features(

        df_train['text'],

        fit=True
    )


    X_test, _ = extract_features(

        df_test['text'],

        tfidf=tfidf,

        fit=False
    )


    print(
        f"\nFeature matrix: "
        f"{X_train.shape[1]:,} features"
    )


    # ============================================================
    # LABEL MATRICES
    # ============================================================

    y_train = df_train[
        LABEL_COLS
    ].values.astype(
        np.int8
    )


    y_test = df_test[
        LABEL_COLS
    ].values.astype(
        np.int8
    )


    print(
        f"Label matrix: "
        f"{y_train.shape}"
    )


    # ============================================================
    # TRAIN LINEAR SVM
    # ============================================================

    print(
        "\n🚀 Training Linear SVM..."
    )


    print(
        "   NOTE: LinearSVC uses CPU, "
        "not the T4 GPU."
    )


    t0 = time.time()


    base_svm = LinearSVC(

        C=0.5,

        max_iter=2000,

        class_weight='balanced',

        dual=False
    )


    model = OneVsRestClassifier(

        base_svm,

        n_jobs=-1
    )


    model.fit(

        X_train,

        y_train
    )


    elapsed = (

        time.time()

        -

        t0
    )


    print(
        f"\n✅ SVM training completed "
        f"in {elapsed / 60:.2f} minutes"
    )


    # ============================================================
    # PREDICTION
    # ============================================================

    print(
        "\n🔮 Generating predictions..."
    )


    decision_scores = model.decision_function(

        X_test
    )


    y_pred = (

        decision_scores
        >= THRESHOLD
    ).astype(
        np.int8
    )


    # ============================================================
    # EVALUATION
    # ============================================================

    results = evaluate(

        y_test,

        y_pred,

        elapsed
    )


    return (

        model,

        tfidf,

        y_test,

        y_pred,

        results
    )


# ================================================================
# RUN
# ================================================================

if __name__ == '__main__':

    model, tfidf, y_test, y_pred, results = main()